# Street-View Semantic Segmentation — FAST V2 BALANCED

GitHub share-ready Google Colab runner.

Edit only the **USER CONFIGURATION** cell before production. The scientific taxonomy/fusion/refinement logic is otherwise preserved from v1.5.7.1.


In [ ]:
!pip -q install "transformers>=4.57,<5" "huggingface-hub<1.0" accelerate safetensors pandas matplotlib pillow scikit-learn tqdm opencv-python torchvision
print("✓ Dependencies installed")


In [ ]:
!pip uninstall -y gradio gradio-client -q

print("✓ Unused Gradio packages removed")


## 1. Frozen taxonomy / project setup

In [ ]:
from pathlib import Path
import json, gc, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path("/content/VLM_Street_Interface_Segmentation")
CONFIG_DIR = PROJECT_ROOT / "config"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for d in [PROJECT_ROOT, CONFIG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_BASELINE_VERSION = "hybrid_v0.9_semantic_backbone"
SHARE_VERSION = "v1.5.7.1-share-ready-audited"
TAXONOMY_VERSION = "street_interface_v1.5.7.1"
IGNORE_INDEX = 255

CLASS_NAMES = {0: 'other_unknown',
 1: 'roadway',
 2: 'sidewalk',
 3: 'bike_lane',
 4: 'curb_edge',
 5: 'upper_building_facade',
 6: 'ground_floor_solid_facade',
 7: 'ground_floor_glazing',
 8: 'door_entrance',
 9: 'signboard',
 10: 'awning_canopy',
 11: 'arcade_column',
 12: 'arcade_soffit',
 13: 'sidewalk_shed_scaffold',
 14: 'stoop_stair',
 15: 'wall_ledge',
 16: 'fence_railing',
 17: 'planter_container',
 18: 'tree',
 19: 'shrub_hedge',
 20: 'ground_vegetation',
 21: 'vertical_green_wall',
 22: 'bench_seating',
 23: 'pole_fixture',
 24: 'traffic_sign_signal',
 25: 'person',
 26: 'vehicle',
 27: 'sky',
 28: 'upper_building_glazing',
 29: 'traffic_cone_barrel'}
PALETTE = {0: (0, 0, 0),
 1: (128, 64, 128),
 2: (244, 35, 232),
 3: (255, 100, 100),
 4: (255, 180, 180),
 5: (70, 70, 70),
 6: (110, 80, 70),
 7: (0, 220, 255),
 8: (255, 140, 0),
 9: (255, 220, 0),
 10: (180, 100, 255),
 11: (120, 120, 160),
 12: (170, 170, 210),
 13: (0, 200, 200),
 14: (170, 90, 40),
 15: (150, 120, 80),
 16: (190, 153, 153),
 17: (180, 110, 40),
 18: (50, 130, 20),
 19: (100, 180, 40),
 20: (150, 220, 80),
 21: (0, 150, 80),
 22: (255, 80, 160),
 23: (153, 153, 153),
 24: (255, 255, 0),
 25: (220, 20, 60),
 26: (0, 0, 142),
 27: (70, 130, 180),
 28: (0, 170, 220),
 29: (255, 80, 0)}

LABEL2ID = {name: cid for cid, name in CLASS_NAMES.items()}
ID2LABEL = CLASS_NAMES.copy()
NUM_CLASSES = len(CLASS_NAMES)

# -----------------------------
# Taxonomy integrity invariants
# -----------------------------
assert NUM_CLASSES == 30
assert set(CLASS_NAMES) == set(PALETTE)
assert len(set(CLASS_NAMES.values())) == NUM_CLASSES
assert set(CLASS_NAMES.keys()) == set(range(NUM_CLASSES))
assert LABEL2ID["tree"] == 18
assert LABEL2ID["upper_building_glazing"] == 28
assert LABEL2ID["traffic_cone_barrel"] == 29
assert "tree_canopy" not in LABEL2ID
assert "tree_trunk" not in LABEL2ID
assert IGNORE_INDEX not in CLASS_NAMES

print("Source baseline:", SOURCE_BASELINE_VERSION)
print("Share package:", SHARE_VERSION)
print("Taxonomy:", TAXONOMY_VERSION)
print("Declared classes:", NUM_CLASSES)
print("Tree output policy: leaves + branches + trunk → tree")
print("Traffic-control policy: traffic_cone_barrel is a formal taxonomy class (ID 29)")
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU 未啟用。Colab: Runtime → Change runtime type → T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# AUTHORITATIVE TAXONOMY / ANNOTATION CONFIG — v1.5
#
# CLASS_NAMES + PALETTE defined above are the single source of truth.
# Config files are generated from those dictionaries so the runtime
# label map, exported taxonomy, CVAT config, and statistics cannot drift.
# ============================================================

TAXONOMY = {
    "taxonomy_version": TAXONOMY_VERSION,
    "num_classes": NUM_CLASSES,
    "ignore_index": IGNORE_INDEX,
    "classes": [
        {
            "id": cid,
            "name": CLASS_NAMES[cid],
            "rgb": list(PALETTE[cid]),
        }
        for cid in sorted(CLASS_NAMES)
    ],
}

ANNOTATION_RULES = {'other_unknown': {'include': ['Visible regions that cannot be reliably assigned to another taxonomy class after '
                               'class-specific recovery.'],
                   'exclude': ['Google Street View screenshot/UI artifacts; use IGNORE=255.']},
 'roadway': {'include': ['Motor-vehicle travel lanes', 'Visible roadway pavement', 'Bicycle-lane pavement in final output'],
             'exclude': ['Sidewalk']},
 'sidewalk': {'include': ['Pedestrian walking surface', 'Public sidewalk paving'],
              'exclude': ['Roadway', 'Building steps']},
 'bike_lane': {'include': ['Internal compatibility class only; remapped to roadway before final output'],
               'exclude': ['User-facing final statistics and legend']},
 'curb_edge': {'include': ['Visible curb face', 'Raised transition between roadway and pedestrian surface'],
               'exclude': ['Entire sidewalk']},
 'upper_building_facade': {'include': ['Opaque building facade above ground-floor interface zone'],
                           'exclude': ['Upper glazing', 'Ground-floor storefront']},
 'ground_floor_solid_facade': {'include': ['Opaque ground-floor wall or storefront facade'],
                               'exclude': ['Glazing', 'Door', 'Signboard']},
 'ground_floor_glazing': {'include': ['Ground-floor storefront glass',
                                      'Display windows',
                                      'Transparent/reflective ground-floor glazing'],
                          'exclude': ['Upper-floor glazing']},
 'door_entrance': {'include': ['Building entrance door', 'Store entrance', 'Doorway opening'], 'exclude': ['Windows']},
 'signboard': {'include': ['Storefront signage', 'Business fascia/projecting signs'], 'exclude': ['Traffic signs']},
 'awning_canopy': {'include': ['Storefront awning', 'Entrance canopy', 'Fabric entrance canopy', 'Residential entrance awning'],
                   'exclude': ['Construction sidewalk shed', 'Tree']},
 'arcade_column': {'include': ['Columns supporting a building-integrated covered pedestrian arcade'],
                   'exclude': ['Construction scaffold poles', 'Streetlight poles']},
 'arcade_soffit': {'include': ['Underside/ceiling of a building-integrated arcade'],
                   'exclude': ['Construction sidewalk shed']},
 'sidewalk_shed_scaffold': {'include': ['Temporary construction sidewalk shed',
                                        'Scaffolding structure/protective overhead shed'],
                            'exclude': ['Permanent arcade', 'Glass curtain wall', 'Glazed facade']},
 'stoop_stair': {'include': ['Exterior entrance steps', 'Stoop'], 'exclude': ['Ordinary sidewalk']},
 'wall_ledge': {'include': ['Low wall', 'Raised ledge', 'Residential perimeter wall', 'Community boundary wall', 'Masonry garden wall'], 'exclude': ['Openwork fence', 'Planter vegetation', 'Building facade']},
 'fence_railing': {'include': ['Park perimeter fence',
                               'Metal fence',
                               'Iron fence',
                               'Pedestrian railing',
                               'Openwork barrier'],
                   'exclude': ['Solid wall', 'Scaffolding', 'Building curtain wall']},
 'planter_container': {'include': ['Built planter box', 'Raised planter', 'Plant container'],
                       'exclude': ['Vegetation inside planter']},
 'tree': {'include': ['Tree leaves', 'Tree crown', 'Visible tree branches', 'Bare branches', 'Main visible tree trunk'],
          'exclude': ['Shrub/hedge', 'Ground vegetation', 'Vertical green wall']},
 'shrub_hedge': {'include': ['Shrub', 'Hedge', 'Dense eye-level woody vegetation'],
                 'exclude': ['Tree', 'Ground grass']},
 'ground_vegetation': {'include': ['Grass', 'Low groundcover vegetation'], 'exclude': ['Shrub', 'Tree']},
 'vertical_green_wall': {'include': ['Vegetation covering a vertical facade/support'],
                         'exclude': ['Tree overlapping wall in perspective']},
 'bench_seating': {'include': ['Bench', 'Fixed public seating'], 'exclude': ['Low wall']},
 'pole_fixture': {'include': ['Streetlight pole', 'Utility pole', 'Fixed vertical street pole'],
                  'exclude': ['Tree trunk', 'Scaffolding pole', 'Google Street View UI artifact']},
 'traffic_sign_signal': {'include': ['Traffic sign', 'Traffic signal'], 'exclude': ['Commercial signboard']},
 'person': {'include': ['Pedestrian', 'Visible person'], 'exclude': []},
 'vehicle': {'include': ['Car', 'Truck', 'Bus', 'Motorcycle', 'Bicycle as vehicle object'], 'exclude': []},
 'sky': {'include': ['Visible sky'], 'exclude': ['Sky reflection in glazing']},
 'upper_building_glazing': {'include': ['Upper-floor windows',
                                        'Curtain-wall glazing',
                                        'Glazed building surfaces above ground-floor interface'],
                            'exclude': ['Ground-floor storefront glazing', 'Vehicle glass']},
 'traffic_cone_barrel': {'include': ['Traffic cone', 'Construction cone', 'Traffic barrel', 'Construction barrel/drum'],
                         'exclude': ['Traffic sign/signal', 'Ordinary street furniture', 'Planter container']}}

BASELINE_SETTINGS = {'packaging_version': 'v1.5.7-roadway-precedence-cleanup',
 'algorithm_baseline': 'hybrid_v0.10_plus_targeted_v1.3_refinements',
 'baseline_status': 'SHARE_READY_SINGLE_IMAGE',
 'taxonomy_version': 'street_interface_v1.5.7',
 'models': {'full_scene_ade': 'nvidia/segformer-b5-finetuned-ade-640-640',
            'full_scene_mapillary': 'facebook/mask2former-swin-large-mapillary-vistas-semantic',
            'detector': 'IDEA-Research/grounding-dino-base',
            'mask_refiner': 'facebook/sam2.1-hiera-small'},
 'detail_detection_thresholds': {'ground_floor_glazing': 0.2,
                                 'door_entrance': 0.21,
                                 'signboard': 0.24,
                                 'awning_canopy': 0.25,
                                 'sidewalk_shed_scaffold': 0.28,
                                 'stoop_stair': 0.22,
                                 'fence_railing': 0.18,
                                 'planter_container': 0.22,
                                 'bench_seating': 0.24,
                                 'person': 0.27,
                                 'vehicle': 0.28,
                                 'traffic_cone_barrel': 0.18},
 'notes': ['Google Street View UI residuals remain IGNORE=255 and are excluded from measurement statistics.',
           'Fence recovery is class-specific and primarily targets other_unknown pixels.',
           'Glass-dominant vertical building components are vetoed from scaffold.',
           'Tree component prompts are detection-only; taxonomy output is one tree class.',
           'traffic_cone_barrel is a declared taxonomy class from initialization; it is not dynamically appended at '
           'runtime.',
           'other_unknown is retained as an epistemic fallback and is not generically filled.'],
 'declared_num_classes': 30}

CVAT_LABELS = [
    {
        "name": CLASS_NAMES[cid],
        "color": "#%02x%02x%02x" % PALETTE[cid],
        "attributes": [],
    }
    for cid in sorted(CLASS_NAMES)
]

# -----------------------------
# Cross-config audit
# -----------------------------
assert TAXONOMY["num_classes"] == NUM_CLASSES == 30
assert [x["id"] for x in TAXONOMY["classes"]] == list(range(NUM_CLASSES))
assert [x["name"] for x in TAXONOMY["classes"]] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert [x["name"] for x in CVAT_LABELS] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert set(ANNOTATION_RULES) == set(CLASS_NAMES.values())
assert "tree_canopy" not in ANNOTATION_RULES
assert "tree_trunk" not in ANNOTATION_RULES

for name, obj in [
    ("taxonomy_v1_5.json", TAXONOMY),
    ("annotation_rules_v1_5.json", ANNOTATION_RULES),
    ("baseline_settings_v1_5.json", BASELINE_SETTINGS),
    ("cvat_labels_v1_5.json", CVAT_LABELS),
]:
    with open(CONFIG_DIR / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

print("✓ v1.4 config exported to", CONFIG_DIR)
print("✓ Runtime taxonomy = exported taxonomy = CVAT taxonomy")
print("✓ 30 declared classes, including traffic_cone_barrel")
print("✓ Tree taxonomy is one class: tree")


## 2. Batch configuration

In [ ]:

# ============================================================
# USER CONFIGURATION — COLAB
# ============================================================
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Change this one folder if your team uses a different Drive location.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/StreetViewSegmentation')

INPUT_DIR = DRIVE_PROJECT_DIR / 'input_images'
BATCH_OUTPUT_ROOT = DRIVE_PROJECT_DIR / 'outputs'

# First test: 3 images. Production: set to None.
MAX_IMAGES = 3

RECURSIVE = True
RESUME = True
STOP_ON_ERROR = False

SAVE_SUMMARY_IMAGE = True
SAVE_LABEL_MAP = True
SAVE_EXTRA_AUDIT_OUTPUTS = False

SUMMARY_WIDTH = 1400
PNG_COMPRESS_LEVEL = 1

# Colab GPUs usually have >= 12 GB VRAM. The loader still falls back on OOM.
LOW_VRAM_MODE = False
VERBOSE_PER_IMAGE = False

FAST_V2 = True
FAST_USE_FP16_DINO = True
FAST_USE_FP16_SAM = True
CLEANUP_EVERY_N = 50

SUPPORTED_EXTENSIONS = {
    '.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'
}

if not INPUT_DIR.exists():
    raise RuntimeError(
        f'Input folder not found: {INPUT_DIR}\n'
        'Create the folder and add images, or edit DRIVE_PROJECT_DIR above.'
    )

_precheck_images = sorted(
    p for p in INPUT_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
)
if not _precheck_images:
    raise RuntimeError(f'No supported images found in: {INPUT_DIR}')

for d in [
    BATCH_OUTPUT_ROOT,
    BATCH_OUTPUT_ROOT / 'summary_images',
    BATCH_OUTPUT_ROOT / 'label_maps',
    BATCH_OUTPUT_ROOT / 'audit_outputs',
]:
    d.mkdir(parents=True, exist_ok=True)

MASTER_CSV = BATCH_OUTPUT_ROOT / 'segmentation_results.csv'
ERROR_LOG = BATCH_OUTPUT_ROOT / 'errors.csv'

print('========================================')
print('FAST V2 — COLAB SHARE-READY')
print('========================================')
print('Input folder :', INPUT_DIR)
print('Images visible:', len(_precheck_images))
print('First files  :', [p.name for p in _precheck_images[:5]])
print('Output folder:', BATCH_OUTPUT_ROOT)
print('MAX_IMAGES   :', MAX_IMAGES)
print('RESUME       :', RESUME)


## 3. Runtime helpers

In [ ]:

# ============================================================
# RUNTIME HELPERS
# ============================================================
import os, io, csv, time, gc, hashlib, contextlib, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('GPU not enabled. Colab: Runtime → Change runtime type → T4 GPU.')

UNKNOWN_ID = LABEL2ID['other_unknown']
IGNORE_ID = IGNORE_INDEX

def render_taxonomy(label_map):
    rgb = np.zeros((label_map.shape[0], label_map.shape[1], 3), dtype=np.uint8)
    for class_id, color in PALETTE.items():
        rgb[label_map == class_id] = color
    return rgb

def safe_test_id(path: Path, input_root: Path) -> str:
    try:
        rel = path.relative_to(input_root).with_suffix('')
        raw = '__'.join(rel.parts)
    except Exception:
        raw = path.stem
    raw = re.sub(r'[^A-Za-z0-9_-]+', '_', raw).strip('_').upper()
    return raw or hashlib.sha1(str(path).encode()).hexdigest()[:12].upper()

def get_image_paths():
    iterator = INPUT_DIR.rglob('*') if RECURSIVE else INPUT_DIR.glob('*')
    paths = sorted(p for p in iterator if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS)
    if MAX_IMAGES is not None:
        paths = paths[:int(MAX_IMAGES)]
    return paths

# Fast non-interactive summary image. Segmentation/statistics are unchanged;
# only the rendering implementation differs from the original Matplotlib figure.
def save_fast_summary(image, display_labels, final_labels, out_path):
    valid = final_labels != IGNORE_ID
    valid_total = max(int(valid.sum()), 1)
    stats = []
    for cid, cname in CLASS_NAMES.items():
        if cname == 'bike_lane':
            continue
        px = int(np.sum(valid & (final_labels == cid)))
        if px <= 0:
            continue
        stats.append((cid, cname, px, 100.0 * px / valid_total))
    stats.sort(key=lambda x: x[3], reverse=True)

    mask_img = Image.fromarray(render_taxonomy(display_labels).astype(np.uint8))
    orig = image.convert('RGB')
    panel_gap = 12
    margin = 16
    panel_w = (SUMMARY_WIDTH - margin * 2 - panel_gap) // 2
    aspect = orig.height / orig.width
    panel_h = max(1, int(panel_w * aspect))
    orig_r = orig.resize((panel_w, panel_h), Image.Resampling.LANCZOS)
    mask_r = mask_img.resize((panel_w, panel_h), Image.Resampling.NEAREST)

    try:
        font_path = font_manager.findfont('DejaVu Sans')
        title_font = ImageFont.truetype(font_path, 34)
        legend_font = ImageFont.truetype(font_path, 21)
    except Exception:
        title_font = ImageFont.load_default()
        legend_font = ImageFont.load_default()

    title_h = 52
    cols = 2 if len(stats) < 21 else 3
    rows = max(1, (len(stats) + cols - 1) // cols)
    row_h = 34
    legend_top = margin + title_h + panel_h + 35
    canvas_h = legend_top + rows * row_h + 25
    canvas = Image.new('RGB', (SUMMARY_WIDTH, canvas_h), 'white')
    draw = ImageDraw.Draw(canvas)

    x1 = margin
    x2 = margin + panel_w + panel_gap
    draw.text((x1 + panel_w//2, margin), 'Original SVI', fill='black', font=title_font, anchor='ma')
    draw.text((x2 + panel_w//2, margin), 'Combined Semantic Mask', fill='black', font=title_font, anchor='ma')
    y_img = margin + title_h
    canvas.paste(orig_r, (x1, y_img))
    canvas.paste(mask_r, (x2, y_img))

    col_w = (SUMMARY_WIDTH - margin*2) // cols
    swatch = 24
    for idx, (cid, cname, px, pct) in enumerate(stats):
        col = idx // rows
        row = idx % rows
        x = margin + col * col_w
        y = legend_top + row * row_h
        color = tuple(PALETTE[cid])
        draw.rectangle((x, y+2, x+swatch, y+swatch+2), fill=color)
        label = f"{cname.replace('_', ' ')} — {pct:.2f}%"
        draw.text((x + swatch + 12, y), label, fill='black', font=legend_font)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path, format='PNG', compress_level=PNG_COMPRESS_LEVEL)

print('Device:', device)
print('GPU:', torch.cuda.get_device_name(0))
print('✓ Batch runtime helpers ready')


## 4. Load models once

In [ ]:

# ============================================================
# LOAD ALL FOUR MODEL OBJECTS ONCE
# ============================================================
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation,
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    Sam2Processor,
    Sam2Model,
)

ADE_MODEL_ID = 'nvidia/segformer-b5-finetuned-ade-640-640'
MAPILLARY_MODEL_ID = 'facebook/mask2former-swin-large-mapillary-vistas-semantic'
GROUNDING_MODEL_ID = 'IDEA-Research/grounding-dino-base'
SAM_MODEL_ID = 'facebook/sam2.1-hiera-small'

print('Loading processors/models once...')
ade_processor = SegformerImageProcessor.from_pretrained(ADE_MODEL_ID)
ade_model = SegformerForSemanticSegmentation.from_pretrained(ADE_MODEL_ID).eval()

map_processor = AutoImageProcessor.from_pretrained(MAPILLARY_MODEL_ID)
map_model = Mask2FormerForUniversalSegmentation.from_pretrained(MAPILLARY_MODEL_ID).eval()

grounding_processor = AutoProcessor.from_pretrained(GROUNDING_MODEL_ID)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_MODEL_ID).eval()

sam_processor = Sam2Processor.from_pretrained(SAM_MODEL_ID)
sam_model = Sam2Model.from_pretrained(SAM_MODEL_ID).eval()

_MODEL_OBJECTS = {
    'ade': ade_model,
    'mapillary': map_model,
    'dino': grounding_model,
    'sam': sam_model,
}

# Try fastest mode first. If weights + inference workspace do not fit,
# set LOW_VRAM_MODE=True and models will be swapped from CPU RAM without re-downloading.
if not LOW_VRAM_MODE:
    try:
        for _name, _model in _MODEL_OBJECTS.items():
            _model.to(device)
        print('✓ All models resident on GPU')
    except torch.cuda.OutOfMemoryError:
        print('⚠ GPU cannot keep all models resident. Switching to LOW_VRAM_MODE.')
        LOW_VRAM_MODE = True
        for _model in _MODEL_OBJECTS.values():
            _model.to('cpu')
        gc.collect()
        torch.cuda.empty_cache()
else:
    for _model in _MODEL_OBJECTS.values():
        _model.to('cpu')
    print('✓ Models cached in CPU RAM; one model is moved to GPU at a time')

_ACTIVE_MODEL = None

def activate_model(name):
    global _ACTIVE_MODEL
    if not LOW_VRAM_MODE:
        return
    if _ACTIVE_MODEL == name:
        return
    # Move previous active model off GPU.
    if _ACTIVE_MODEL in _MODEL_OBJECTS:
        _MODEL_OBJECTS[_ACTIVE_MODEL].to('cpu')
    gc.collect()
    torch.cuda.empty_cache()
    _MODEL_OBJECTS[name].to(device)
    _ACTIVE_MODEL = name

print('LOW_VRAM_MODE =', LOW_VRAM_MODE)
print('✓ Model initialization complete')


## 5. Frozen per-image pipeline source
The long source dictionary below is mechanically derived from the uploaded notebook. Model load/release and intermediate plotting are removed; scientific inference/refinement code is retained.

In [ ]:
PIPELINE_SOURCE = {'ADE': '\n'
        '@torch.no_grad()\n'
        'def run_ade(image):\n'
        '\n'
        '    W, H = image.size\n'
        '\n'
        '    inputs = ade_processor(\n'
        '        images=image,\n'
        '        return_tensors="pt"\n'
        '    )\n'
        '\n'
        '    inputs = {\n'
        '        k: v.to(device)\n'
        '        for k, v in inputs.items()\n'
        '    }\n'
        '\n'
        '    outputs = ade_model(**inputs)\n'
        '\n'
        '    logits = F.interpolate(\n'
        '        outputs.logits,\n'
        '        size=(H, W),\n'
        '        mode="bilinear",\n'
        '        align_corners=False\n'
        '    )\n'
        '\n'
        '    probs = torch.softmax(\n'
        '        logits,\n'
        '        dim=1\n'
        '    )\n'
        '\n'
        '    confidence, prediction = probs.max(\n'
        '        dim=1\n'
        '    )\n'
        '\n'
        '    return (\n'
        '        prediction[0]\n'
        '        .cpu()\n'
        '        .numpy()\n'
        '        .astype(np.int32),\n'
        '\n'
        '        confidence[0]\n'
        '        .cpu()\n'
        '        .numpy()\n'
        '        .astype(np.float32)\n'
        '    )\n'
        '\n'
        '\n'
        'ade_pred, ade_confidence = run_ade(\n'
        '    image\n'
        ')\n'
        '\n'
        'ADE_ID2LABEL = {\n'
        '    int(k): str(v).lower()\n'
        '    for k, v\n'
        '    in ade_model.config.id2label.items()\n'
        '}\n'
        '\n'
        'print("✓ ADE inference complete")\n'
        '\n'
        'def ade_ids_fuzzy(\n'
        '    include_terms,\n'
        '    exclude_terms=None\n'
        '):\n'
        '\n'
        '    if exclude_terms is None:\n'
        '        exclude_terms = []\n'
        '\n'
        '    found = []\n'
        '\n'
        '    for class_id, name in ADE_ID2LABEL.items():\n'
        '\n'
        '        included = any(\n'
        '            term.lower() in name\n'
        '            for term in include_terms\n'
        '        )\n'
        '\n'
        '        excluded = any(\n'
        '            term.lower() in name\n'
        '            for term in exclude_terms\n'
        '        )\n'
        '\n'
        '        if included and not excluded:\n'
        '            found.append(class_id)\n'
        '\n'
        '    return found\n'
        '\n'
        '\n'
        'def ade_mask_fuzzy(\n'
        '    include_terms,\n'
        '    exclude_terms=None\n'
        '):\n'
        '\n'
        '    ids = ade_ids_fuzzy(\n'
        '        include_terms,\n'
        '        exclude_terms\n'
        '    )\n'
        '\n'
        '    if not ids:\n'
        '\n'
        '        return np.zeros(\n'
        '            (H, W),\n'
        '            dtype=bool\n'
        '        )\n'
        '\n'
        '    return np.isin(\n'
        '        ade_pred,\n'
        '        ids\n'
        '    )\n'
        '\n'
        '\n'
        'ADE_GLASS_V2 = ade_mask_fuzzy([\n'
        '    "window",\n'
        '    "glass"\n'
        '])\n'
        '\n'
        'ADE_ROAD_V2 = ade_mask_fuzzy([\n'
        '    "road"\n'
        '])\n'
        '\n'
        'ADE_SIDEWALK_V2 = ade_mask_fuzzy([\n'
        '    "sidewalk",\n'
        '    "path"\n'
        '])\n'
        '\n'
        'ADE_BUILDING_V2 = ade_mask_fuzzy([\n'
        '    "building",\n'
        '    "house",\n'
        '    "skyscraper"\n'
        '])\n'
        '\n'
        'ADE_SKY_V2 = ade_mask_fuzzy([\n'
        '    "sky"\n'
        '])\n'
        '\n'
        'ADE_TREE_V2 = ade_mask_fuzzy([\n'
        '    "tree"\n'
        '])\n'
        '\n'
        'ADE_PLANT_V2 = ade_mask_fuzzy([\n'
        '    "plant",\n'
        '    "grass",\n'
        '    "flower"\n'
        '])\n'
        '\n'
        'ADE_FENCE_V3 = ade_mask_fuzzy([\n'
        '    "fence",\n'
        '    "railing"\n'
        '])\n'
        '\n'
        '\n'
        'print("ADE glass:", int(ADE_GLASS_V2.sum()))\n'
        'print("ADE building:", int(ADE_BUILDING_V2.sum()))\n'
        'print("ADE trees:", int(ADE_TREE_V2.sum()))\n'
        'print("ADE fence/railing:", int(ADE_FENCE_V3.sum()))\n'
        '\n',
 'MAPILLARY': '\n'
              'inputs = map_processor(\n'
              '    images=image,\n'
              '    return_tensors="pt"\n'
              ')\n'
              '\n'
              'inputs = {\n'
              '    k: v.to(device)\n'
              '    for k, v in inputs.items()\n'
              '}\n'
              '\n'
              'with torch.no_grad():\n'
              '\n'
              '    outputs = map_model(\n'
              '        **inputs\n'
              '    )\n'
              '\n'
              '\n'
              'map_pred = (\n'
              '    map_processor\n'
              '    .post_process_semantic_segmentation(\n'
              '        outputs,\n'
              '        target_sizes=[\n'
              '            (H, W)\n'
              '        ]\n'
              '    )[0]\n'
              '    .cpu()\n'
              '    .numpy()\n'
              '    .astype(np.int32)\n'
              ')\n'
              '\n'
              '\n'
              'MAP_ID2LABEL = {\n'
              '    int(k): str(v).lower()\n'
              '    for k, v\n'
              '    in map_model.config.id2label.items()\n'
              '}\n'
              '\n'
              'print("✓ Mapillary inference complete")\n'
              '\n'
              'def mapillary_ids(\n'
              '    include_terms,\n'
              '    exclude_terms=None\n'
              '):\n'
              '\n'
              '    if exclude_terms is None:\n'
              '        exclude_terms = []\n'
              '\n'
              '    found = []\n'
              '\n'
              '    for class_id, name in MAP_ID2LABEL.items():\n'
              '\n'
              '        included = any(\n'
              '            term.lower() in name\n'
              '            for term in include_terms\n'
              '        )\n'
              '\n'
              '        excluded = any(\n'
              '            term.lower() in name\n'
              '            for term in exclude_terms\n'
              '        )\n'
              '\n'
              '        if included and not excluded:\n'
              '            found.append(class_id)\n'
              '\n'
              '    return found\n'
              '\n'
              '\n'
              'def mapillary_mask(\n'
              '    include_terms,\n'
              '    exclude_terms=None\n'
              '):\n'
              '\n'
              '    ids = mapillary_ids(\n'
              '        include_terms,\n'
              '        exclude_terms\n'
              '    )\n'
              '\n'
              '    if not ids:\n'
              '\n'
              '        return np.zeros(\n'
              '            (H, W),\n'
              '            dtype=bool\n'
              '        )\n'
              '\n'
              '    return np.isin(\n'
              '        map_pred,\n'
              '        ids\n'
              '    )\n'
              '\n'
              '\n'
              'MAP_ROAD = mapillary_mask(\n'
              '    ["road"],\n'
              '    [\n'
              '        "marking",\n'
              '        "bike lane",\n'
              '        "service lane"\n'
              '    ]\n'
              ')\n'
              '\n'
              'MAP_ROAD_MARKINGS = mapillary_mask(\n'
              '    ["marking"]\n'
              ')\n'
              '\n'
              'MAP_SIDEWALK = mapillary_mask(\n'
              '    ["sidewalk"]\n'
              ')\n'
              '\n'
              'MAP_BIKE_LANE = mapillary_mask(\n'
              '    ["bike lane"]\n'
              ')\n'
              '\n'
              'MAP_CURB = mapillary_mask(\n'
              '    ["curb"]\n'
              ')\n'
              '\n'
              'MAP_BUILDING = mapillary_mask(\n'
              '    ["building"]\n'
              ')\n'
              '\n'
              'MAP_SKY = mapillary_mask(\n'
              '    ["sky"]\n'
              ')\n'
              '\n'
              'MAP_FENCE = mapillary_mask(\n'
              '    ["fence"]\n'
              ')\n'
              '\n'
              'MAP_WINDOW = mapillary_mask(\n'
              '    ["window", "glass"]\n'
              ')\n'
              '\n'
              '\n'
              'print("Road:", int(MAP_ROAD.sum()))\n'
              'print("Sidewalk:", int(MAP_SIDEWALK.sum()))\n'
              'print("Bike lane:", int(MAP_BIKE_LANE.sum()))\n'
              'print("Building:", int(MAP_BUILDING.sum()))\n'
              'print("Sky:", int(MAP_SKY.sum()))\n'
              'print("Fence:", int(MAP_FENCE.sum()))\n'
              'print("Window/glass:", int(MAP_WINDOW.sum()))\n'
              '\n'
              'final_test = np.full(\n'
              '    (H, W),\n'
              '    UNKNOWN_ID,\n'
              '    dtype=np.uint8\n'
              ')\n'
              '\n'
              '\n'
              '# Road\n'
              'final_test[\n'
              '    MAP_ROAD |\n'
              '    MAP_ROAD_MARKINGS\n'
              '] = LABEL2ID["roadway"]\n'
              '\n'
              '\n'
              '# Bike lane overrides road\n'
              'final_test[\n'
              '    MAP_BIKE_LANE\n'
              '] = LABEL2ID["bike_lane"]\n'
              '\n'
              '\n'
              '# Sidewalk\n'
              'final_test[\n'
              '    MAP_SIDEWALK\n'
              '] = LABEL2ID["sidewalk"]\n'
              '\n'
              '\n'
              '# Curb\n'
              'final_test[\n'
              '    MAP_CURB\n'
              '] = LABEL2ID["curb_edge"]\n'
              '\n'
              '\n'
              '# Building\n'
              'final_test[\n'
              '    MAP_BUILDING\n'
              '] = LABEL2ID[\n'
              '    "upper_building_facade"\n'
              ']\n'
              '\n'
              '\n'
              '# Sky\n'
              'final_test[\n'
              '    MAP_SKY\n'
              '] = LABEL2ID["sky"]\n'
              '\n'
              '\n'
              '# ADE fallback\n'
              'unknown = (\n'
              '    final_test == UNKNOWN_ID\n'
              ')\n'
              '\n'
              'final_test[\n'
              '    unknown & ADE_ROAD_V2\n'
              '] = LABEL2ID["roadway"]\n'
              '\n'
              '\n'
              'unknown = (\n'
              '    final_test == UNKNOWN_ID\n'
              ')\n'
              '\n'
              'final_test[\n'
              '    unknown & ADE_SIDEWALK_V2\n'
              '] = LABEL2ID["sidewalk"]\n'
              '\n'
              '\n'
              'unknown = (\n'
              '    final_test == UNKNOWN_ID\n'
              ')\n'
              '\n'
              'final_test[\n'
              '    unknown & ADE_BUILDING_V2\n'
              '] = LABEL2ID[\n'
              '    "upper_building_facade"\n'
              ']\n'
              '\n'
              '\n'
              'unknown = (\n'
              '    final_test == UNKNOWN_ID\n'
              ')\n'
              '\n'
              'final_test[\n'
              '    unknown & ADE_SKY_V2\n'
              '] = LABEL2ID["sky"]\n'
              '\n'
              '\n'
              '# preliminary vegetation\n'
              'final_test[\n'
              '    ADE_TREE_V2\n'
              '] = LABEL2ID[\n'
              '    "tree"\n'
              ']\n'
              '\n'
              '\n'
              'building_context = (\n'
              '    MAP_BUILDING\n'
              '    |\n'
              '    ADE_BUILDING_V2\n'
              ')\n'
              '\n'
              '\n'
              '# ADE preliminary glazing\n'
              'basic_glass = (\n'
              '    ADE_GLASS_V2\n'
              '    &\n'
              '    building_context\n'
              ')\n'
              '\n'
              'final_test[\n'
              '    basic_glass\n'
              '] = LABEL2ID[\n'
              '    "upper_building_glazing"\n'
              ']\n'
              '\n'
              '\n'
              'print("✓ Full scene base created")\n'
              '\n'
              'rgb_base = render_taxonomy(\n'
              '    final_test\n'
              ')\n'
              '\n'
              '# [batch] intermediate matplotlib QA figure skipped\n'
              '\n'
              '\n',
 'DINO': '\n'
         'AUTO_LABEL_QUERIES = {\n'
         '    "ground_floor_glazing": [\n'
         '        "storefront glass window",\n'
         '        "shopfront window",\n'
         '        "ground floor glass window",\n'
         '        "glass storefront"\n'
         '    ],\n'
         '    "door_entrance": [\n'
         '        "building entrance door",\n'
         '        "store entrance door",\n'
         '        "entrance doorway"\n'
         '    ],\n'
         '    "signboard": [\n'
         '        "storefront business sign",\n'
         '        "shop signboard",\n'
         '        "commercial sign"\n'
         '    ],\n'
         '    "awning_canopy": [\n'
         '        "storefront awning",\n'
         '        "entrance awning",\n'
         '        "building entrance canopy",\n'
         '        "fabric entrance canopy",\n'
         '        "residential entrance awning",\n'
         '        "projecting entrance canopy"\n'
         '    ],\n'
         '    "sidewalk_shed_scaffold": [\n'
         '        "construction sidewalk shed",\n'
         '        "sidewalk scaffolding",\n'
         '        "construction scaffolding",\n'
         '        "scaffold structure"\n'
         '    ],\n'
         '    "stoop_stair": [\n'
         '        "building entrance steps",\n'
         '        "residential stoop",\n'
         '        "entrance stairs"\n'
         '    ],\n'
         '    "wall_ledge": [\n'
         '        "residential perimeter wall",\n'
         '        "community boundary wall",\n'
         '        "masonry boundary wall",\n'
         '        "low garden wall",\n'
         '        "property boundary wall",\n'
         '        "stone retaining wall"\n'
         '    ],\n'
         '    "fence_railing": [\n'
         '        "park perimeter fence",\n'
         '        "black iron park fence",\n'
         '        "metal park fence",\n'
         '        "pedestrian railing",\n'
         '        "metal railing",\n'
         '        "ornamental iron fence",\n'
         '        "openwork metal fence"\n'
         '    ],\n'
         '    "planter_container": [\n'
         '        "street planter",\n'
         '        "planter box",\n'
         '        "plant container"\n'
         '    ],\n'
         '    "bench_seating": [\n'
         '        "street bench",\n'
         '        "outdoor bench"\n'
         '    ],\n'
         '    "person": [\n'
         '        "pedestrian",\n'
         '        "person"\n'
         '    ],\n'
         '    "vehicle": [\n'
         '        "car",\n'
         '        "truck",\n'
         '        "bus",\n'
         '        "bicycle"\n'
         '    ]\n'
         '}\n'
         '\n'
         'CLASS_THRESHOLDS = {\n'
         '    "ground_floor_glazing": 0.20,\n'
         '    "door_entrance": 0.21,\n'
         '    "signboard": 0.24,\n'
         '    "awning_canopy": 0.18,\n'
         '    "sidewalk_shed_scaffold": 0.28,\n'
         '    "stoop_stair": 0.22,\n'
         '    "wall_ledge": 0.18,\n'
         '    "fence_railing": 0.18,\n'
         '    "planter_container": 0.22,\n'
         '    "bench_seating": 0.24,\n'
         '    "person": 0.27,\n'
         '    "vehicle": 0.28\n'
         '}\n'
         '\n'
         'DETAIL_CROPS = {\n'
         '    "left_interface": (\n'
         '        0, int(H * 0.22), int(W * 0.58), int(H * 0.88)\n'
         '    ),\n'
         '    "center_interface": (\n'
         '        int(W * 0.18), int(H * 0.20), int(W * 0.82), int(H * 0.90)\n'
         '    ),\n'
         '    "right_interface": (\n'
         '        int(W * 0.40), int(H * 0.18), W, int(H * 0.90)\n'
         '    )\n'
         '}\n'
         '\n'
         'DETAIL_CLASSES = [\n'
         '    "ground_floor_glazing",\n'
         '    "door_entrance",\n'
         '    "signboard",\n'
         '    "awning_canopy",\n'
         '    "sidewalk_shed_scaffold",\n'
         '    "stoop_stair",\n'
         '    "wall_ledge",\n'
         '    "fence_railing",\n'
         '    "planter_container",\n'
         '    "bench_seating",\n'
         '    "person",\n'
         '    "vehicle"\n'
         ']\n'
         '\n'
         'def _dino_autocast():\n'
         '    enabled = bool(\n'
         '        globals().get("FAST_USE_FP16_DINO", False)\n'
         '        and device.type == "cuda"\n'
         '    )\n'
         '    return torch.autocast(\n'
         '        device_type="cuda",\n'
         '        dtype=torch.float16,\n'
         '        enabled=enabled\n'
         '    )\n'
         '\n'
         '@torch.no_grad()\n'
         'def detect_same_class_batch(\n'
         '    crop_items,\n'
         '    class_name,\n'
         '    threshold=None,\n'
         '    text_threshold=0.18,\n'
         '    output_class_name=None,\n'
         '    output_extra=None,\n'
         '):\n'
         '    """\n'
         '    Batch the 3 same-purpose crops for ONE semantic query together.\n'
         '    This preserves class-specific prompts/thresholds while reducing\n'
         '    Grounding DINO launch overhead.\n'
         '    """\n'
         '    if not crop_items:\n'
         '        return []\n'
         '\n'
         '    images = [x["image"] for x in crop_items]\n'
         '    prompts = AUTO_LABEL_QUERIES[class_name]\n'
         '    texts = [prompts for _ in images]\n'
         '\n'
         '    inputs = grounding_processor(\n'
         '        images=images,\n'
         '        text=texts,\n'
         '        return_tensors="pt",\n'
         '        padding=True,\n'
         '    ).to(device)\n'
         '\n'
         '    with _dino_autocast():\n'
         '        outputs = grounding_model(**inputs)\n'
         '\n'
         '    thr = CLASS_THRESHOLDS[class_name] if threshold is None else threshold\n'
         '\n'
         '    results = grounding_processor.post_process_grounded_object_detection(\n'
         '        outputs,\n'
         '        inputs.input_ids,\n'
         '        threshold=thr,\n'
         '        text_threshold=text_threshold,\n'
         '        target_sizes=[(im.height, im.width) for im in images],\n'
         '    )\n'
         '\n'
         '    detections = []\n'
         '    for item, result in zip(crop_items, results):\n'
         '        ox, oy = item["offset"]\n'
         '        for box, score in zip(result["boxes"], result["scores"]):\n'
         '            x1, y1, x2, y2 = box.detach().cpu().tolist()\n'
         '            det = {\n'
         '                "class_name": output_class_name or class_name,\n'
         '                "score": float(score.detach().cpu().item()),\n'
         '                "box": [x1 + ox, y1 + oy, x2 + ox, y2 + oy],\n'
         '            }\n'
         '            if output_extra:\n'
         '                det.update(output_extra)\n'
         '            elif (output_class_name or class_name) in LABEL2ID:\n'
         '                det["class_id"] = LABEL2ID[output_class_name or class_name]\n'
         '            detections.append(det)\n'
         '\n'
         '    return detections\n'
         '\n'
         'detail_crop_items = []\n'
         'for crop_name, crop_box in DETAIL_CROPS.items():\n'
         '    x1, y1, x2, y2 = crop_box\n'
         '    detail_crop_items.append({\n'
         '        "name": crop_name,\n'
         '        "image": image.crop(crop_box),\n'
         '        "offset": (x1, y1),\n'
         '    })\n'
         '\n'
         'detail_detections_raw = []\n'
         'for class_name in DETAIL_CLASSES:\n'
         '    detail_detections_raw.extend(\n'
         '        detect_same_class_batch(\n'
         '            detail_crop_items,\n'
         '            class_name,\n'
         '            threshold=CLASS_THRESHOLDS[class_name],\n'
         '            text_threshold=0.18,\n'
         '        )\n'
         '    )\n'
         '\n'
         'WINDOW_PROMPTS = [\n'
         '    "apartment building window",\n'
         '    "building window",\n'
         '    "residential window",\n'
         '    "glass window",\n'
         '    "storefront window",\n'
         '    "glass curtain wall"\n'
         ']\n'
         '\n'
         'FACADE_TILES = {\n'
         '    "left_upper": (\n'
         '        0, 0, int(W * 0.60), int(H * 0.68)\n'
         '    ),\n'
         '    "center_upper": (\n'
         '        int(W * 0.15), 0, int(W * 0.85), int(H * 0.72)\n'
         '    ),\n'
         '    "right_upper": (\n'
         '        int(W * 0.40), 0, W, int(H * 0.70)\n'
         '    )\n'
         '}\n'
         '\n'
         '@torch.no_grad()\n'
         'def detect_windows_batch(tile_items):\n'
         '    if not tile_items:\n'
         '        return []\n'
         '\n'
         '    images = [x["image"] for x in tile_items]\n'
         '    texts = [WINDOW_PROMPTS for _ in images]\n'
         '\n'
         '    inputs = grounding_processor(\n'
         '        images=images,\n'
         '        text=texts,\n'
         '        return_tensors="pt",\n'
         '        padding=True,\n'
         '    ).to(device)\n'
         '\n'
         '    with _dino_autocast():\n'
         '        outputs = grounding_model(**inputs)\n'
         '\n'
         '    results = grounding_processor.post_process_grounded_object_detection(\n'
         '        outputs,\n'
         '        inputs.input_ids,\n'
         '        threshold=0.15,\n'
         '        text_threshold=0.17,\n'
         '        target_sizes=[(im.height, im.width) for im in images],\n'
         '    )\n'
         '\n'
         '    detections = []\n'
         '    for item, result in zip(tile_items, results):\n'
         '        ox, oy = item["offset"]\n'
         '        for box, score in zip(result["boxes"], result["scores"]):\n'
         '            x1, y1, x2, y2 = box.detach().cpu().tolist()\n'
         '            detections.append({\n'
         '                "class_name": "building_glazing",\n'
         '                "score": float(score.detach().cpu().item()),\n'
         '                "box": [x1 + ox, y1 + oy, x2 + ox, y2 + oy],\n'
         '            })\n'
         '    return detections\n'
         '\n'
         'facade_tile_items = []\n'
         'for tile_name, crop_box in FACADE_TILES.items():\n'
         '    x1, y1, x2, y2 = crop_box\n'
         '    facade_tile_items.append({\n'
         '        "name": tile_name,\n'
         '        "image": image.crop(crop_box),\n'
         '        "offset": (x1, y1),\n'
         '    })\n'
         '\n'
         'window_detections_raw = detect_windows_batch(facade_tile_items)\n'
         '\n'
         'TREE_QUERY_GROUPS = {\n'
         '    "foliage": [\n'
         '        "green tree foliage",\n'
         '        "leafy tree crown",\n'
         '        "tree leaves",\n'
         '        "green leaves on a tree"\n'
         '    ],\n'
         '    "woody": [\n'
         '        "leafless street tree",\n'
         '        "bare deciduous tree branches",\n'
         '        "woody tree branches",\n'
         '        "street tree trunk",\n'
         '        "tree trunk",\n'
         '        "tree limb"\n'
         '    ]\n'
         '}\n'
         '\n'
         'TREE_ROIS = {\n'
         '    "left_tree": (\n'
         '        0, 0, int(W * 0.52), int(H * 0.78)\n'
         '    ),\n'
         '    "center_tree": (\n'
         '        int(W * 0.18), 0, int(W * 0.75), int(H * 0.75)\n'
         '    ),\n'
         '    "right_tree": (\n'
         '        int(W * 0.45), 0, W, int(H * 0.75)\n'
         '    )\n'
         '}\n'
         '\n'
         '@torch.no_grad()\n'
         'def detect_tree_component_batch(roi_items, prompts, component_name, threshold):\n'
         '    if not roi_items:\n'
         '        return []\n'
         '\n'
         '    images = [x["image"] for x in roi_items]\n'
         '    texts = [prompts for _ in images]\n'
         '\n'
         '    inputs = grounding_processor(\n'
         '        images=images,\n'
         '        text=texts,\n'
         '        return_tensors="pt",\n'
         '        padding=True,\n'
         '    ).to(device)\n'
         '\n'
         '    with _dino_autocast():\n'
         '        outputs = grounding_model(**inputs)\n'
         '\n'
         '    results = grounding_processor.post_process_grounded_object_detection(\n'
         '        outputs,\n'
         '        inputs.input_ids,\n'
         '        threshold=threshold,\n'
         '        text_threshold=0.17,\n'
         '        target_sizes=[(im.height, im.width) for im in images],\n'
         '    )\n'
         '\n'
         '    detections = []\n'
         '    for item, result in zip(roi_items, results):\n'
         '        ox, oy = item["offset"]\n'
         '        for box, score in zip(result["boxes"], result["scores"]):\n'
         '            x1, y1, x2, y2 = box.detach().cpu().tolist()\n'
         '            detections.append({\n'
         '                "component": component_name,\n'
         '                "score": float(score.detach().cpu().item()),\n'
         '                "box": [x1 + ox, y1 + oy, x2 + ox, y2 + oy],\n'
         '            })\n'
         '    return detections\n'
         '\n'
         'tree_roi_items = []\n'
         'for roi_name, roi_box in TREE_ROIS.items():\n'
         '    x1, y1, x2, y2 = roi_box\n'
         '    tree_roi_items.append({\n'
         '        "name": roi_name,\n'
         '        "image": image.crop(roi_box),\n'
         '        "offset": (x1, y1),\n'
         '    })\n'
         '\n'
         'tree_detections_raw = []\n'
         'tree_detections_raw.extend(\n'
         '    detect_tree_component_batch(\n'
         '        tree_roi_items,\n'
         '        TREE_QUERY_GROUPS["foliage"],\n'
         '        "foliage",\n'
         '        threshold=0.27,\n'
         '    )\n'
         ')\n'
         'tree_detections_raw.extend(\n'
         '    detect_tree_component_batch(\n'
         '        tree_roi_items,\n'
         '        TREE_QUERY_GROUPS["woody"],\n'
         '        "woody",\n'
         '        threshold=0.16,\n'
         '    )\n'
         ')\n'
         '\n'
         'from torchvision.ops import nms\n'
         '\n'
         'def nms_group(detections, key_name, iou=0.42):\n'
         '    if not detections:\n'
         '        return []\n'
         '\n'
         '    result = []\n'
         '    groups = sorted(set(d[key_name] for d in detections))\n'
         '\n'
         '    for group in groups:\n'
         '        subset = [d for d in detections if d[key_name] == group]\n'
         '        boxes = torch.tensor(\n'
         '            [d["box"] for d in subset],\n'
         '            dtype=torch.float32\n'
         '        )\n'
         '        scores = torch.tensor(\n'
         '            [d["score"] for d in subset],\n'
         '            dtype=torch.float32\n'
         '        )\n'
         '        keep = nms(boxes, scores, iou)\n'
         '        result.extend(subset[i] for i in keep.tolist())\n'
         '\n'
         '    return result\n'
         '\n'
         'detail_detections = nms_group(\n'
         '    detail_detections_raw,\n'
         '    "class_name"\n'
         ')\n'
         '\n'
         'window_detections = nms_group(\n'
         '    window_detections_raw,\n'
         '    "class_name"\n'
         ')\n'
         '\n'
         'tree_detections = nms_group(\n'
         '    tree_detections_raw,\n'
         '    "component"\n'
         ')\n'
         '\n'
         'print("FAST DINO detail:", len(detail_detections))\n'
         'print("FAST DINO windows:", len(window_detections))\n'
         'print("FAST DINO trees:", len(tree_detections))\n',
 'SAM': '\n'
        '@torch.no_grad()\n'
        'def boxes_to_masks(\n'
        '    image,\n'
        '    detections\n'
        '):\n'
        '\n'
        '    if not detections:\n'
        '        return []\n'
        '\n'
        '    boxes = [\n'
        '        d["box"]\n'
        '        for d in detections\n'
        '    ]\n'
        '\n'
        '    inputs = (\n'
        '        sam_processor(\n'
        '            images=image,\n'
        '            input_boxes=[boxes],\n'
        '            return_tensors="pt"\n'
        '        )\n'
        '        .to(device)\n'
        '    )\n'
        '\n'
        '    _sam_fp16 = bool(\n'
        '        globals().get("FAST_USE_FP16_SAM", False)\n'
        '        and device.type == "cuda"\n'
        '    )\n'
        '    with torch.autocast(\n'
        '        device_type="cuda",\n'
        '        dtype=torch.float16,\n'
        '        enabled=_sam_fp16\n'
        '    ):\n'
        '        outputs = sam_model(\n'
        '            **inputs,\n'
        '            multimask_output=False\n'
        '        )\n'
        '\n'
        '    masks = (\n'
        '        sam_processor\n'
        '        .post_process_masks(\n'
        '            outputs.pred_masks.cpu(),\n'
        '            inputs["original_sizes"]\n'
        '        )[0]\n'
        '    )\n'
        '\n'
        '    output = []\n'
        '\n'
        '    for i, det in enumerate(\n'
        '        detections\n'
        '    ):\n'
        '\n'
        '        mask = masks[i]\n'
        '\n'
        '        while mask.ndim > 2:\n'
        '            mask = mask[0]\n'
        '\n'
        '        output.append({\n'
        '            **det,\n'
        '            "mask":\n'
        '                mask.numpy() > 0\n'
        '        })\n'
        '\n'
        '    return output\n'
        '\n'
        'detail_objects = boxes_to_masks(\n'
        '    image,\n'
        '    detail_detections\n'
        ')\n'
        '\n'
        'window_objects = boxes_to_masks(\n'
        '    image,\n'
        '    window_detections\n'
        ')\n'
        '\n'
        'tree_objects = boxes_to_masks(\n'
        '    image,\n'
        '    tree_detections\n'
        ')\n'
        '\n'
        '\n'
        'print(\n'
        '    "Detail masks:",\n'
        '    len(detail_objects)\n'
        ')\n'
        '\n'
        'print(\n'
        '    "Window masks:",\n'
        '    len(window_objects)\n'
        ')\n'
        '\n'
        'print(\n'
        '    "Tree masks:",\n'
        '    len(tree_objects)\n'
        ')\n',
 'TRAFFIC_DETECT': '\n'
                   'TRAFFIC_CONTROL_PROMPTS = [\n'
                   '    "orange traffic cone",\n'
                   '    "road traffic cone",\n'
                   '    "construction traffic cone",\n'
                   '    "construction cone",\n'
                   '    "orange construction barrel",\n'
                   '    "traffic barrel",\n'
                   '    "construction barrel",\n'
                   '    "construction drum"\n'
                   ']\n'
                   '\n'
                   'TRAFFIC_ROIS = {\n'
                   '    "left_street": (\n'
                   '        0, int(H * 0.28), int(W * 0.58), int(H * 0.90)\n'
                   '    ),\n'
                   '    "center_street": (\n'
                   '        int(W * 0.18), int(H * 0.26), int(W * 0.84), int(H * 0.90)\n'
                   '    ),\n'
                   '    "right_street": (\n'
                   '        int(W * 0.42), int(H * 0.28), W, int(H * 0.90)\n'
                   '    )\n'
                   '}\n'
                   '\n'
                   '@torch.no_grad()\n'
                   'def detect_traffic_control_batch(roi_items):\n'
                   '    images = [x["image"] for x in roi_items]\n'
                   '    texts = [TRAFFIC_CONTROL_PROMPTS for _ in images]\n'
                   '\n'
                   '    inputs = grounding_processor(\n'
                   '        images=images,\n'
                   '        text=texts,\n'
                   '        return_tensors="pt",\n'
                   '        padding=True,\n'
                   '    ).to(device)\n'
                   '\n'
                   '    with _dino_autocast():\n'
                   '        outputs = grounding_model(**inputs)\n'
                   '\n'
                   '    results = grounding_processor.post_process_grounded_object_detection(\n'
                   '        outputs,\n'
                   '        inputs.input_ids,\n'
                   '        threshold=0.18,\n'
                   '        text_threshold=0.16,\n'
                   '        target_sizes=[(im.height, im.width) for im in images],\n'
                   '    )\n'
                   '\n'
                   '    detections = []\n'
                   '    for item, result in zip(roi_items, results):\n'
                   '        ox, oy = item["offset"]\n'
                   '\n'
                   '        for box, score in zip(result["boxes"], result["scores"]):\n'
                   '            x1, y1, x2, y2 = box.detach().cpu().tolist()\n'
                   '            x1 += ox\n'
                   '            x2 += ox\n'
                   '            y1 += oy\n'
                   '            y2 += oy\n'
                   '\n'
                   '            box_w = max(x2 - x1, 1)\n'
                   '            box_h = max(y2 - y1, 1)\n'
                   '            box_area_ratio = box_w * box_h / (W * H)\n'
                   '            center_y = ((y1 + y2) / 2) / H\n'
                   '\n'
                   '            if box_area_ratio > 0.025:\n'
                   '                continue\n'
                   '            if box_area_ratio < 0.00001:\n'
                   '                continue\n'
                   '            if not (0.28 <= center_y <= 0.92):\n'
                   '                continue\n'
                   '\n'
                   '            detections.append({\n'
                   '                "class_name": TRAFFIC_CONTROL_NAME,\n'
                   '                "class_id": TRAFFIC_CONTROL_ID,\n'
                   '                "score": float(score.detach().cpu().item()),\n'
                   '                "box": [float(x1), float(y1), float(x2), float(y2)],\n'
                   '            })\n'
                   '\n'
                   '    return detections\n'
                   '\n'
                   'traffic_roi_items = []\n'
                   'for roi_name, roi_box in TRAFFIC_ROIS.items():\n'
                   '    x1, y1, x2, y2 = roi_box\n'
                   '    traffic_roi_items.append({\n'
                   '        "name": roi_name,\n'
                   '        "image": image.crop(roi_box),\n'
                   '        "offset": (x1, y1),\n'
                   '    })\n'
                   '\n'
                   'traffic_detections_raw = detect_traffic_control_batch(traffic_roi_items)\n'
                   '\n'
                   'if len(traffic_detections_raw) > 0:\n'
                   '    boxes = torch.tensor(\n'
                   '        [d["box"] for d in traffic_detections_raw],\n'
                   '        dtype=torch.float32\n'
                   '    )\n'
                   '    scores = torch.tensor(\n'
                   '        [d["score"] for d in traffic_detections_raw],\n'
                   '        dtype=torch.float32\n'
                   '    )\n'
                   '    keep = nms(boxes, scores, 0.35)\n'
                   '    traffic_detections = [\n'
                   '        traffic_detections_raw[i]\n'
                   '        for i in keep.tolist()\n'
                   '    ]\n'
                   'else:\n'
                   '    traffic_detections = []\n'
                   '\n'
                   'print("FAST traffic-control:", len(traffic_detections))\n',
 'TRAFFIC_SAM': 'traffic_objects = (\n'
                '    boxes_to_masks(\n'
                '        image,\n'
                '        traffic_detections\n'
                '    )\n'
                ')\n'
                '\n'
                '\n'
                'print(\n'
                '    "Traffic-control SAM masks:",\n'
                '    len(\n'
                '        traffic_objects\n'
                '    )\n'
                ')\n',
 'cell_18': 'final_v06 = final_test.copy()\n'
            '\n'
            '\n'
            'INTERFACE_ANCHOR_CLASSES = {\n'
            '    "ground_floor_glazing",\n'
            '    "door_entrance",\n'
            '    "signboard",\n'
            '    "awning_canopy",\n'
            '    "sidewalk_shed_scaffold",\n'
            '    "stoop_stair"\n'
            '}\n'
            '\n'
            '\n'
            'interface_anchor = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'for obj in detail_objects:\n'
            '\n'
            '    if (\n'
            '        obj["class_name"]\n'
            '        in INTERFACE_ANCHOR_CLASSES\n'
            '        and\n'
            '        obj["score"] >= 0.23\n'
            '    ):\n'
            '\n'
            '        interface_anchor |= obj[\n'
            '            "mask"\n'
            '        ]\n'
            '\n'
            '\n'
            'kernel = cv2.getStructuringElement(\n'
            '    cv2.MORPH_RECT,\n'
            '    (51, 31)\n'
            ')\n'
            '\n'
            '\n'
            'interface_neighborhood = (\n'
            '    cv2.dilate(\n'
            '        interface_anchor.astype(\n'
            '            np.uint8\n'
            '        ),\n'
            '        kernel,\n'
            '        iterations=1\n'
            '    ) > 0\n'
            ')\n'
            '\n'
            '\n'
            'ground_floor_solid = (\n'
            '    building_context\n'
            '    &\n'
            '    interface_neighborhood\n'
            '    &\n'
            '    ~ADE_GLASS_V2\n'
            ')\n'
            '\n'
            '\n'
            'final_v06[\n'
            '    ground_floor_solid\n'
            '] = LABEL2ID[\n'
            '    "ground_floor_solid_facade"\n'
            ']\n'
            '\n'
            '\n'
            'print("✓ v0.6 interface neighborhood built")\n'
            '\n'
            'facade_glass_aggressive = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'for obj in window_objects:\n'
            '\n'
            '    gated = (\n'
            '        obj["mask"]\n'
            '        &\n'
            '        building_context\n'
            '    )\n'
            '\n'
            '    original_area = max(\n'
            '        int(\n'
            '            obj["mask"].sum()\n'
            '        ),\n'
            '        1\n'
            '    )\n'
            '\n'
            '    agreement = (\n'
            '        gated.sum()\n'
            '        /\n'
            '        original_area\n'
            '    )\n'
            '\n'
            '    if agreement < 0.25:\n'
            '        continue\n'
            '\n'
            '    facade_glass_aggressive |= gated\n'
            '\n'
            '\n'
            'ground_glass = (\n'
            '    facade_glass_aggressive\n'
            '    &\n'
            '    interface_neighborhood\n'
            ')\n'
            '\n'
            'upper_glass = (\n'
            '    facade_glass_aggressive\n'
            '    &\n'
            '    ~interface_neighborhood\n'
            ')\n'
            '\n'
            '\n'
            'final_v06[\n'
            '    ground_glass\n'
            '] = LABEL2ID[\n'
            '    "ground_floor_glazing"\n'
            ']\n'
            '\n'
            'final_v06[\n'
            '    upper_glass\n'
            '] = LABEL2ID[\n'
            '    "upper_building_glazing"\n'
            ']\n'
            '\n'
            '\n'
            'print(\n'
            '    "Aggressive upper glass:",\n'
            '    upper_glass.mean() * 100\n'
            ')\n'
            '\n'
            'FUSION_MIN_SCORE = {\n'
            '\n'
            '    "ground_floor_glazing":\n'
            '        0.22,\n'
            '\n'
            '    "door_entrance":\n'
            '        0.23,\n'
            '\n'
            '    "signboard":\n'
            '        0.26,\n'
            '\n'
            '    "awning_canopy":\n'
            '        0.27,\n'
            '\n'
            '    "sidewalk_shed_scaffold":\n'
            '        0.28,\n'
            '\n'
            '    "stoop_stair":\n'
            '        0.24,\n'
            '\n'
            '    "wall_ledge":\n'
            '        0.28,\n'
            '\n'
            '    "fence_railing":\n'
            '        0.20,\n'
            '\n'
            '    "planter_container":\n'
            '        0.24,\n'
            '\n'
            '    "bench_seating":\n'
            '        0.25,\n'
            '\n'
            '    "person":\n'
            '        0.27,\n'
            '\n'
            '    "vehicle":\n'
            '        0.28\n'
            '}\n'
            '\n'
            '\n'
            'for obj in detail_objects:\n'
            '\n'
            '    class_name = obj[\n'
            '        "class_name"\n'
            '    ]\n'
            '\n'
            '    score = obj[\n'
            '        "score"\n'
            '    ]\n'
            '\n'
            '    if (\n'
            '        score\n'
            '        <\n'
            '        FUSION_MIN_SCORE[\n'
            '            class_name\n'
            '        ]\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '    mask = obj[\n'
            '        "mask"\n'
            '    ]\n'
            '\n'
            '\n'
            '    if (\n'
            '        class_name\n'
            '        ==\n'
            '        "ground_floor_glazing"\n'
            '    ):\n'
            '\n'
            '        allowed = (\n'
            '            building_context\n'
            '            &\n'
            '            interface_neighborhood\n'
            '        )\n'
            '\n'
            '\n'
            '    elif class_name in {\n'
            '        "door_entrance",\n'
            '        "signboard",\n'
            '        "awning_canopy"\n'
            '    }:\n'
            '\n'
            '        allowed = (\n'
            '            building_context\n'
            '            |\n'
            '            interface_neighborhood\n'
            '        )\n'
            '\n'
            '\n'
            '    elif (\n'
            '        class_name\n'
            '        ==\n'
            '        "sidewalk_shed_scaffold"\n'
            '    ):\n'
            '\n'
            '        allowed = (\n'
            '            MAP_BUILDING\n'
            '            |\n'
            '            MAP_SIDEWALK\n'
            '        )\n'
            '\n'
            '\n'
            '    elif class_name == "wall_ledge":\n'
            '\n'
            '        wall_sidewalk_near = cv2.dilate(\n'
            '            MAP_SIDEWALK.astype(np.uint8),\n'
            '            cv2.getStructuringElement(cv2.MORPH_RECT, (91, 61)),\n'
            '            iterations=1\n'
            '        ) > 0\n'
            '\n'
            '        allowed = (\n'
            '            building_context\n'
            '            |\n'
            '            wall_sidewalk_near\n'
            '        )\n'
            '\n'
            '\n'
            '    elif class_name == "fence_railing":\n'
            '\n'
            '        # v1.3: park perimeter fences are often next to open space,\n'
            '        # not directly on a building footprint.\n'
            '        fence_sidewalk_near = cv2.dilate(\n'
            '            MAP_SIDEWALK.astype(np.uint8),\n'
            '            cv2.getStructuringElement(cv2.MORPH_RECT, (81, 51)),\n'
            '            iterations=1\n'
            '        ) > 0\n'
            '\n'
            '        allowed = (\n'
            '            MAP_FENCE\n'
            '            | ADE_FENCE_V3\n'
            '            | fence_sidewalk_near\n'
            '            | building_context\n'
            '        )\n'
            '\n'
            '    elif class_name in {\n'
            '        "stoop_stair",\n'
            '        "planter_container",\n'
            '        "bench_seating"\n'
            '    }:\n'
            '\n'
            '        allowed = (\n'
            '            MAP_SIDEWALK\n'
            '            |\n'
            '            building_context\n'
            '        )\n'
            '\n'
            '\n'
            '    else:\n'
            '\n'
            '        allowed = np.ones(\n'
            '            (H, W),\n'
            '            dtype=bool\n'
            '        )\n'
            '\n'
            '\n'
            '    gated = (\n'
            '        mask\n'
            '        &\n'
            '        allowed\n'
            '    )\n'
            '\n'
            '\n'
            '    original_area = max(\n'
            '        int(mask.sum()),\n'
            '        1\n'
            '    )\n'
            '\n'
            '\n'
            '    if (\n'
            '        gated.sum()\n'
            '        /\n'
            '        original_area\n'
            '        <\n'
            '        0.12\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    final_v06[\n'
            '        gated\n'
            '    ] = LABEL2ID[\n'
            '        class_name\n'
            '    ]\n'
            '\n'
            '\n'
            'print("✓ v0.6-style detail fusion complete")\n'
            '\n'
            'dynamic_mask = np.isin(\n'
            '    final_v06,\n'
            '    [\n'
            '        LABEL2ID["person"],\n'
            '        LABEL2ID["vehicle"]\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'final_v06[\n'
            '    MAP_ROAD\n'
            '    &\n'
            '    ~dynamic_mask\n'
            '] = LABEL2ID[\n'
            '    "roadway"\n'
            ']\n'
            '\n'
            '\n'
            'final_v06[\n'
            '    MAP_BIKE_LANE\n'
            '    &\n'
            '    ~dynamic_mask\n'
            '] = LABEL2ID[\n'
            '    "bike_lane"\n'
            ']\n'
            '\n'
            '\n'
            'final_v06[\n'
            '    MAP_CURB\n'
            '    &\n'
            '    ~dynamic_mask\n'
            '] = LABEL2ID[\n'
            '    "curb_edge"\n'
            ']\n'
            '\n'
            '\n'
            'print("✓ v0.6 street surfaces protected")\n'
            '\n'
            'def mask_geometry(mask):\n'
            '\n'
            '    ys, xs = np.where(mask)\n'
            '\n'
            '    if len(xs) == 0:\n'
            '\n'
            '        return {\n'
            '            "area_ratio": 0,\n'
            '            "cx": 0,\n'
            '            "cy": 0\n'
            '        }\n'
            '\n'
            '    return {\n'
            '\n'
            '        "area_ratio":\n'
            '            float(mask.mean()),\n'
            '\n'
            '        "cx":\n'
            '            float(xs.mean() / W),\n'
            '\n'
            '        "cy":\n'
            '            float(ys.mean() / H)\n'
            '    }\n'
            '\n'
            '\n'
            'accepted_windows = []\n'
            '\n'
            '\n'
            'for obj in window_objects:\n'
            '\n'
            '    geometry = mask_geometry(\n'
            '        obj["mask"]\n'
            '    )\n'
            '\n'
            '    # Reject huge window hallucinations\n'
            '    if (\n'
            '        geometry["area_ratio"]\n'
            '        >\n'
            '        0.015\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '    if (\n'
            '        geometry["cy"]\n'
            '        >\n'
            '        0.80\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '    gated = (\n'
            '        obj["mask"]\n'
            '        &\n'
            '        building_context\n'
            '    )\n'
            '\n'
            '    original_area = max(\n'
            '        int(\n'
            '            obj["mask"].sum()\n'
            '        ),\n'
            '        1\n'
            '    )\n'
            '\n'
            '    agreement = (\n'
            '        gated.sum()\n'
            '        /\n'
            '        original_area\n'
            '    )\n'
            '\n'
            '    if agreement < 0.50:\n'
            '        continue\n'
            '\n'
            '    accepted_windows.append({\n'
            '        **obj,\n'
            '        "mask":\n'
            '            gated\n'
            '    })\n'
            '\n'
            '\n'
            'print(\n'
            '    "Windows:",\n'
            '    len(window_objects),\n'
            '    "→ accepted:",\n'
            '    len(accepted_windows)\n'
            ')\n'
            '\n'
            'final_v09 = final_v06.copy()\n'
            '\n'
            '\n'
            'UPPER_FACADE_ID = LABEL2ID[\n'
            '    "upper_building_facade"\n'
            ']\n'
            '\n'
            'UPPER_GLASS_ID = LABEL2ID[\n'
            '    "upper_building_glazing"\n'
            ']\n'
            '\n'
            '\n'
            'precise_window_mask = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'for obj in accepted_windows:\n'
            '\n'
            '    precise_window_mask |= (\n'
            '        obj["mask"]\n'
            '    )\n'
            '\n'
            'SCAFFOLD_ID = LABEL2ID[\n'
            '    "sidewalk_shed_scaffold"\n'
            ']\n'
            '\n'
            '\n'
            'scaffold_core = (\n'
            '    final_v06\n'
            '    ==\n'
            '    SCAFFOLD_ID\n'
            ')\n'
            '\n'
            '\n'
            'def shift_mask(mask, dy=0, dx=0):\n'
            '\n'
            '    out = np.zeros_like(\n'
            '        mask,\n'
            '        dtype=bool\n'
            '    )\n'
            '\n'
            '    y0 = max(0, -dy)\n'
            '    y1 = mask.shape[0] - max(0, dy)\n'
            '\n'
            '    x0 = max(0, -dx)\n'
            '    x1 = mask.shape[1] - max(0, dx)\n'
            '\n'
            '    yd0 = max(0, dy)\n'
            '    xd0 = max(0, dx)\n'
            '\n'
            '    if y1 > y0 and x1 > x0:\n'
            '\n'
            '        out[\n'
            '            yd0:yd0 + (y1-y0),\n'
            '            xd0:xd0 + (x1-x0)\n'
            '        ] = mask[\n'
            '            y0:y1,\n'
            '            x0:x1\n'
            '        ]\n'
            '\n'
            '    return out\n'
            '\n'
            '\n'
            'under_scaffold_keep = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'for dy in [\n'
            '    0,\n'
            '    20,\n'
            '    40,\n'
            '    60,\n'
            '    80,\n'
            '    100\n'
            ']:\n'
            '\n'
            '    under_scaffold_keep |= (\n'
            '        shift_mask(\n'
            '            scaffold_core,\n'
            '            dy=dy\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            'under_scaffold_keep = (\n'
            '    cv2.dilate(\n'
            '        under_scaffold_keep.astype(\n'
            '            np.uint8\n'
            '        ),\n'
            '        cv2.getStructuringElement(\n'
            '            cv2.MORPH_RECT,\n'
            '            (71, 29)\n'
            '        ),\n'
            '        iterations=1\n'
            '    ) > 0\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "Protected under-scaffold:",\n'
            '    int(\n'
            '        under_scaffold_keep.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'upper_precision_zone = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'upper_precision_zone[\n'
            '    :int(H * 0.55),\n'
            '    :\n'
            '] = True\n'
            '\n'
            '\n'
            'upper_precision_zone &= (\n'
            '    building_context\n'
            ')\n'
            '\n'
            'upper_precision_zone &= (\n'
            '    ~under_scaffold_keep\n'
            ')\n'
            '\n'
            '\n'
            '# Reset building pixels in upper precision zone\n'
            '# to solid façade first\n'
            'final_v09[\n'
            '    upper_precision_zone\n'
            '] = UPPER_FACADE_ID\n'
            '\n'
            '\n'
            '# Then install only accepted windows\n'
            'final_v09[\n'
            '    precise_window_mask\n'
            '    &\n'
            '    upper_precision_zone\n'
            '] = UPPER_GLASS_ID\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ upper building precision transplanted"\n'
            ')\n'
            '\n'
            'STOOP_ID = LABEL2ID[\n'
            '    "stoop_stair"\n'
            ']\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# v1.4 STOOP PRECISION\n'
            '#\n'
            '# A sidewalk pixel does NOT become a stoop merely because it is\n'
            '# on / near the sidewalk.\n'
            '#\n'
            '# Keep stoop only when the component is:\n'
            '# - close to a reliable entrance / doorway anchor\n'
            '# - compact enough to behave like entrance steps\n'
            '# - located in the lower / interface portion of the scene\n'
            '#\n'
            '# Rejected false stoop is restored to its underlying semantic\n'
            '# surface, with sidewalk taking priority when Mapillary says\n'
            '# sidewalk.\n'
            '# ============================================================\n'
            '\n'
            'door_anchor = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'for obj in detail_objects:\n'
            '\n'
            '    if (\n'
            '        obj["class_name"] == "door_entrance"\n'
            '        and\n'
            '        obj["score"] >= 0.23\n'
            '    ):\n'
            '\n'
            '        geom = mask_geometry(\n'
            '            obj["mask"]\n'
            '        )\n'
            '\n'
            '        if geom["area_ratio"] <= 0.035:\n'
            '            door_anchor |= obj["mask"]\n'
            '\n'
            '\n'
            'entrance_neighborhood = (\n'
            '    cv2.dilate(\n'
            '        door_anchor.astype(np.uint8),\n'
            '        cv2.getStructuringElement(\n'
            '            cv2.MORPH_RECT,\n'
            '            (61, 41)\n'
            '        ),\n'
            '        iterations=1\n'
            '    ) > 0\n'
            ')\n'
            '\n'
            '\n'
            'current_stoop = (\n'
            '    final_v09 == STOOP_ID\n'
            ')\n'
            '\n'
            '\n'
            'num_stoop, stoop_cc, stoop_stats, _ = (\n'
            '    cv2.connectedComponentsWithStats(\n'
            '        current_stoop.astype(np.uint8),\n'
            '        connectivity=8\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'valid_stoop_v14 = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'rejected_stoop_v14 = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'stoop_audit_rows = []\n'
            '\n'
            '\n'
            'for cid in range(\n'
            '    1,\n'
            '    num_stoop\n'
            '):\n'
            '\n'
            '    component = (\n'
            '        stoop_cc == cid\n'
            '    )\n'
            '\n'
            '    area = int(\n'
            '        component.sum()\n'
            '    )\n'
            '\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(\n'
            '        stoop_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_LEFT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    y = int(\n'
            '        stoop_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_TOP\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    w = int(\n'
            '        stoop_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_WIDTH\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    h = int(\n'
            '        stoop_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_HEIGHT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    area_ratio = (\n'
            '        area\n'
            '        /\n'
            '        (H * W)\n'
            '    )\n'
            '\n'
            '    width_ratio = (\n'
            '        w\n'
            '        /\n'
            '        W\n'
            '    )\n'
            '\n'
            '    center_y = (\n'
            '        y + h / 2\n'
            '    ) / H\n'
            '\n'
            '    entrance_overlap = float(\n'
            '        entrance_neighborhood[\n'
            '            component\n'
            '        ].mean()\n'
            '    )\n'
            '\n'
            '    sidewalk_overlap = float(\n'
            '        MAP_SIDEWALK[\n'
            '            component\n'
            '        ].mean()\n'
            '    )\n'
            '\n'
            '    compact = (\n'
            '        area_ratio <= 0.022\n'
            '        and\n'
            '        width_ratio <= 0.28\n'
            '    )\n'
            '\n'
            '    lower_interface = (\n'
            '        center_y >= 0.38\n'
            '    )\n'
            '\n'
            '    entrance_supported = (\n'
            '        entrance_overlap >= 0.08\n'
            '    )\n'
            '\n'
            '    keep = (\n'
            '        compact\n'
            '        and\n'
            '        lower_interface\n'
            '        and\n'
            '        entrance_supported\n'
            '    )\n'
            '\n'
            '    if keep:\n'
            '\n'
            '        valid_stoop_v14[\n'
            '            component\n'
            '        ] = True\n'
            '\n'
            '    else:\n'
            '\n'
            '        rejected_stoop_v14[\n'
            '            component\n'
            '        ] = True\n'
            '\n'
            '    stoop_audit_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "area_ratio": area_ratio,\n'
            '        "width_ratio": width_ratio,\n'
            '        "center_y": center_y,\n'
            '        "entrance_overlap": entrance_overlap,\n'
            '        "sidewalk_overlap": sidewalk_overlap,\n'
            '        "kept_as_stoop": keep\n'
            '    })\n'
            '\n'
            '\n'
            '# Restore false stoop to underlying semantics.\n'
            'restore_stoop_sidewalk = (\n'
            '    rejected_stoop_v14\n'
            '    &\n'
            '    MAP_SIDEWALK\n'
            ')\n'
            '\n'
            'restore_stoop_building = (\n'
            '    rejected_stoop_v14\n'
            '    &\n'
            '    ~MAP_SIDEWALK\n'
            '    &\n'
            '    building_context\n'
            ')\n'
            '\n'
            'restore_stoop_unknown = (\n'
            '    rejected_stoop_v14\n'
            '    &\n'
            '    ~MAP_SIDEWALK\n'
            '    &\n'
            '    ~building_context\n'
            ')\n'
            '\n'
            '\n'
            'final_v09[\n'
            '    restore_stoop_sidewalk\n'
            '] = LABEL2ID[\n'
            '    "sidewalk"\n'
            ']\n'
            '\n'
            '\n'
            'final_v09[\n'
            '    restore_stoop_building\n'
            '] = LABEL2ID[\n'
            '    "ground_floor_solid_facade"\n'
            ']\n'
            '\n'
            '\n'
            'final_v09[\n'
            '    restore_stoop_unknown\n'
            '] = UNKNOWN_ID\n'
            '\n'
            '\n'
            'print(\n'
            '    "v1.4 stoop kept:",\n'
            '    int(\n'
            '        valid_stoop_v14.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "v1.4 false stoop rejected:",\n'
            '    int(\n'
            '        rejected_stoop_v14.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'if stoop_audit_rows:\n'
            '\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            stoop_audit_rows\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# v1.4 SEMANTIC BACKBONE COMMIT\n'
            '#\n'
            '# STOP HERE.\n'
            '#\n'
            '# Legacy v0.10 used DINO/SAM tree proposals to write broad\n'
            '# foliage / woody masks directly into the label map.\n'
            '#\n'
            '# v1.4 intentionally does NOT do that.\n'
            '#\n'
            '# final_v09 is preserved as the authoritative semantic backbone.\n'
            '# ============================================================\n'
            '\n'
            'semantic_backbone = (\n'
            '    final_v09.copy()\n'
            ')\n'
            '\n'
            'final_v010 = (\n'
            '    semantic_backbone.copy()\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Backbone protection snapshots\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'BACKBONE_PROTECTED_CLASS_NAMES = [\n'
            '    "roadway",\n'
            '    "sidewalk",\n'
            '    "curb_edge",\n'
            '    "upper_building_facade",\n'
            '    "upper_building_glazing",\n'
            '    "ground_floor_solid_facade",\n'
            '    "ground_floor_glazing",\n'
            '    "sky",\n'
            ']\n'
            '\n'
            '\n'
            'BACKBONE_PROTECTED_IDS = [\n'
            '    LABEL2ID[name]\n'
            '    for name in\n'
            '    BACKBONE_PROTECTED_CLASS_NAMES\n'
            ']\n'
            '\n'
            '\n'
            'backbone_protected_mask = np.isin(\n'
            '    semantic_backbone,\n'
            '    BACKBONE_PROTECTED_IDS\n'
            ')\n'
            '\n'
            '\n'
            'backbone_glazing_mask = np.isin(\n'
            '    semantic_backbone,\n'
            '    [\n'
            '        LABEL2ID[\n'
            '            "upper_building_glazing"\n'
            '        ],\n'
            '        LABEL2ID[\n'
            '            "ground_floor_glazing"\n'
            '        ]\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'backbone_sidewalk_mask = (\n'
            '    semantic_backbone\n'
            '    ==\n'
            '    LABEL2ID[\n'
            '        "sidewalk"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'backbone_tree_mask = (\n'
            '    semantic_backbone\n'
            '    ==\n'
            '    LABEL2ID[\n'
            '        "tree"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ v0.9 semantic backbone committed"\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Backbone glazing pixels:",\n'
            '    int(\n'
            '        backbone_glazing_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Backbone sidewalk pixels:",\n'
            '    int(\n'
            '        backbone_sidewalk_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Backbone tree pixels:",\n'
            '    int(\n'
            '        backbone_tree_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Visual checkpoint\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'rgb_backbone = render_taxonomy(\n'
            '    semantic_backbone\n'
            ')\n'
            '\n'
            'original_np = np.array(\n'
            '    image\n'
            ')\n'
            '\n'
            'overlay_backbone = (\n'
            '    original_np.copy()\n'
            ')\n'
            '\n'
            'classified_backbone = (\n'
            '    semantic_backbone\n'
            '    !=\n'
            '    UNKNOWN_ID\n'
            ')\n'
            '\n'
            'overlay_backbone[\n'
            '    classified_backbone\n'
            '] = (\n'
            '    original_np[\n'
            '        classified_backbone\n'
            '    ] * 0.42\n'
            '    +\n'
            '    rgb_backbone[\n'
            '        classified_backbone\n'
            '    ] * 0.58\n'
            ').astype(\n'
            '    np.uint8\n'
            ')\n'
            '\n'
            '\n'
            '# [batch] intermediate matplotlib QA figure skipped\n'
            '\n',
 'cell_20': '# ============================================================\n'
            '# v1.4 WORKING STATE\n'
            '#\n'
            '# All later refinements start from the v0.9 semantic backbone.\n'
            '# There is no v0.10 broad-tree rewrite in this version.\n'
            '# ============================================================\n'
            '\n'
            'import numpy as np\n'
            'import pandas as pd\n'
            'import matplotlib.pyplot as plt\n'
            'import cv2\n'
            'import torch\n'
            'import gc\n'
            '\n'
            '\n'
            'final_working = (\n'
            '    semantic_backbone.copy()\n'
            ')\n'
            '\n'
            '\n'
            'TRAFFIC_CONTROL_NAME = (\n'
            '    "traffic_cone_barrel"\n'
            ')\n'
            '\n'
            'TRAFFIC_CONTROL_ID = LABEL2ID[\n'
            '    TRAFFIC_CONTROL_NAME\n'
            ']\n'
            '\n'
            '\n'
            'assert TRAFFIC_CONTROL_ID == 29\n'
            'assert NUM_CLASSES == 30\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ v1.4 starts from semantic_backbone"\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Traffic-control class ID:",\n'
            '    TRAFFIC_CONTROL_ID\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Declared taxonomy classes:",\n'
            '    NUM_CLASSES\n'
            ')\n',
 'cell_21': '# ============================================================\n'
            '# TREE POLICY — v1.4 BACKBONE PROTECTED\n'
            '#\n'
            '# IMPORTANT:\n'
            '#\n'
            '# The old v0.10 behavior:\n'
            '#\n'
            '#     final[woody_mask]   = tree\n'
            '#     final[foliage_mask] = tree\n'
            '#\n'
            '# allowed a broad DINO/SAM proposal to overwrite an already\n'
            '# correct building façade / glazing result.\n'
            '#\n'
            '# v1.4 removes that overwrite path completely.\n'
            '#\n'
            '# Tree is taken from the v0.9 semantic backbone.\n'
            '#\n'
            '# DINO/SAM tree proposals are retained only as diagnostics for\n'
            '# future validation. They have ZERO write permission here.\n'
            '# ============================================================\n'
            '\n'
            'TREE_ID = LABEL2ID[\n'
            '    "tree"\n'
            ']\n'
            '\n'
            'ROAD_ID = LABEL2ID[\n'
            '    "roadway"\n'
            ']\n'
            '\n'
            'SIDEWALK_ID = LABEL2ID[\n'
            '    "sidewalk"\n'
            ']\n'
            '\n'
            'CURB_ID = LABEL2ID[\n'
            '    "curb_edge"\n'
            ']\n'
            '\n'
            'UPPER_FACADE_ID = LABEL2ID[\n'
            '    "upper_building_facade"\n'
            ']\n'
            '\n'
            'GROUND_SOLID_ID = LABEL2ID[\n'
            '    "ground_floor_solid_facade"\n'
            ']\n'
            '\n'
            'UPPER_GLASS_ID = LABEL2ID[\n'
            '    "upper_building_glazing"\n'
            ']\n'
            '\n'
            'GROUND_GLASS_ID = LABEL2ID[\n'
            '    "ground_floor_glazing"\n'
            ']\n'
            '\n'
            '\n'
            '# Diagnostic proposal masks only.\n'
            'raw_foliage_diagnostic = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'raw_woody_diagnostic = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'for obj in tree_objects:\n'
            '\n'
            '    if (\n'
            '        obj["component"] == "foliage"\n'
            '        and\n'
            '        obj["score"] >= 0.27\n'
            '    ):\n'
            '\n'
            '        raw_foliage_diagnostic |= (\n'
            '            obj["mask"]\n'
            '        )\n'
            '\n'
            '    elif (\n'
            '        obj["component"] == "woody"\n'
            '        and\n'
            '        obj["score"] >= 0.18\n'
            '    ):\n'
            '\n'
            '        raw_woody_diagnostic |= (\n'
            '            obj["mask"]\n'
            '        )\n'
            '\n'
            '\n'
            'tree_proposal_on_glazing = (\n'
            '    (\n'
            '        raw_foliage_diagnostic\n'
            '        |\n'
            '        raw_woody_diagnostic\n'
            '    )\n'
            '    &\n'
            '    backbone_glazing_mask\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "Backbone tree %:",\n'
            '    backbone_tree_mask.mean()\n'
            '    *\n'
            '    100\n'
            ')\n'
            '\n'
            'print(\n'
            '    "DINO/SAM foliage proposal % (diagnostic only):",\n'
            '    raw_foliage_diagnostic.mean()\n'
            '    *\n'
            '    100\n'
            ')\n'
            '\n'
            'print(\n'
            '    "DINO/SAM woody proposal % (diagnostic only):",\n'
            '    raw_woody_diagnostic.mean()\n'
            '    *\n'
            '    100\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Tree proposal pixels that WOULD have hit glazing:",\n'
            '    int(\n'
            '        tree_proposal_on_glazing.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "✓ Tree proposals have no label-write permission in v1.4"\n'
            ')\n',
 'cell_23': '# ============================================================\n'
            '# TRAFFIC CONTROL QUALITY GATE\n'
            '# ============================================================\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Check whether Mapillary itself has a cone-like class\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'traffic_mapillary_ids = (\n'
            '    mapillary_ids(\n'
            '        [\n'
            '            "traffic cone",\n'
            '            "traffic-cone",\n'
            '            "cone"\n'
            '        ],\n'
            '        [\n'
            '            "traffic sign"\n'
            '        ]\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "Mapillary cone-like classes:"\n'
            ')\n'
            '\n'
            '\n'
            'if traffic_mapillary_ids:\n'
            '\n'
            '    for cid in traffic_mapillary_ids:\n'
            '\n'
            '        print(\n'
            '            cid,\n'
            '            MAP_ID2LABEL[\n'
            '                cid\n'
            '            ]\n'
            '        )\n'
            '\n'
            'else:\n'
            '\n'
            '    print(\n'
            '        "None — DINO/SAM + scene logic will be used."\n'
            '    )\n'
            '\n'
            '\n'
            'if traffic_mapillary_ids:\n'
            '\n'
            '    MAP_TRAFFIC_CONTROL = (\n'
            '        np.isin(\n'
            '            map_pred,\n'
            '            traffic_mapillary_ids\n'
            '        )\n'
            '    )\n'
            '\n'
            'else:\n'
            '\n'
            '    MAP_TRAFFIC_CONTROL = np.zeros(\n'
            '        (\n'
            '            H,\n'
            '            W\n'
            '        ),\n'
            '        dtype=bool\n'
            '    )\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Street-surface context\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'street_surface = (\n'
            '    MAP_ROAD\n'
            '    |\n'
            '    MAP_SIDEWALK\n'
            '    |\n'
            '    MAP_CURB\n'
            ')\n'
            '\n'
            '\n'
            'surface_kernel = (\n'
            '    cv2.getStructuringElement(\n'
            '        cv2.MORPH_ELLIPSE,\n'
            '        (\n'
            '            31,\n'
            '            31\n'
            '        )\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'street_surface_support = (\n'
            '    cv2.dilate(\n'
            '        street_surface.astype(\n'
            '            np.uint8\n'
            '        ),\n'
            '        surface_kernel,\n'
            '        iterations=1\n'
            '    )\n'
            '    >\n'
            '    0\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Orange color evidence\n'
            '#\n'
            '# Used as supporting evidence,\n'
            '# not as the segmentation itself.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'image_np = np.array(\n'
            '    image\n'
            ')\n'
            '\n'
            '\n'
            'hsv = (\n'
            '    cv2.cvtColor(\n'
            '        image_np,\n'
            '        cv2.COLOR_RGB2HSV\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'h_channel = hsv[\n'
            '    :,\n'
            '    :,\n'
            '    0\n'
            ']\n'
            '\n'
            's_channel = hsv[\n'
            '    :,\n'
            '    :,\n'
            '    1\n'
            ']\n'
            '\n'
            'v_channel = hsv[\n'
            '    :,\n'
            '    :,\n'
            '    2\n'
            ']\n'
            '\n'
            '\n'
            'orange_support = (\n'
            '    (\n'
            '        h_channel\n'
            '        >=\n'
            '        3\n'
            '    )\n'
            '    &\n'
            '    (\n'
            '        h_channel\n'
            '        <=\n'
            '        27\n'
            '    )\n'
            '    &\n'
            '    (\n'
            '        s_channel\n'
            '        >=\n'
            '        90\n'
            '    )\n'
            '    &\n'
            '    (\n'
            '        v_channel\n'
            '        >=\n'
            '        65\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Evaluate each SAM proposal\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'traffic_control_mask = np.zeros(\n'
            '    (\n'
            '        H,\n'
            '        W\n'
            '    ),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'accepted_traffic_objects = []\n'
            '\n'
            '\n'
            'for obj in traffic_objects:\n'
            '\n'
            '    mask = obj[\n'
            '        "mask"\n'
            '    ]\n'
            '\n'
            '\n'
            '    area = max(\n'
            '        int(\n'
            '            mask.sum()\n'
            '        ),\n'
            '        1\n'
            '    )\n'
            '\n'
            '\n'
            '    area_ratio = (\n'
            '        area\n'
            '        /\n'
            '        (\n'
            '            H\n'
            '            *\n'
            '            W\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '    ys, xs = np.where(\n'
            '        mask\n'
            '    )\n'
            '\n'
            '\n'
            '    if (\n'
            '        len(\n'
            '            ys\n'
            '        )\n'
            '        ==\n'
            '        0\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    cy = (\n'
            '        ys.mean()\n'
            '        /\n'
            '        H\n'
            '    )\n'
            '\n'
            '\n'
            '    # ----------------------------------------\n'
            '    # Geometry filter\n'
            '    # ----------------------------------------\n'
            '\n'
            '    if (\n'
            '        area_ratio\n'
            '        >\n'
            '        0.02\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    if not (\n'
            '        0.30\n'
            '        <=\n'
            '        cy\n'
            '        <=\n'
            '        0.92\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    # ----------------------------------------\n'
            '    # Street semantic support\n'
            '    # ----------------------------------------\n'
            '\n'
            '    surface_ratio = (\n'
            '        street_surface_support[\n'
            '            mask\n'
            '        ]\n'
            '        .mean()\n'
            '    )\n'
            '\n'
            '\n'
            '    if (\n'
            '        surface_ratio\n'
            '        <\n'
            '        0.25\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    # ----------------------------------------\n'
            '    # Orange evidence\n'
            '    # ----------------------------------------\n'
            '\n'
            '    orange_ratio = (\n'
            '        orange_support[\n'
            '            mask\n'
            '        ]\n'
            '        .mean()\n'
            '    )\n'
            '\n'
            '\n'
            '    # ----------------------------------------\n'
            '    # Mapillary evidence, when available\n'
            '    # ----------------------------------------\n'
            '\n'
            '    mapillary_ratio = (\n'
            '        MAP_TRAFFIC_CONTROL[\n'
            '            mask\n'
            '        ]\n'
            '        .mean()\n'
            '    )\n'
            '\n'
            '\n'
            '    # Require either orange evidence\n'
            '    # OR direct Mapillary support.\n'
            '\n'
            '    if (\n'
            '        orange_ratio\n'
            '        <\n'
            '        0.025\n'
            '        and\n'
            '        mapillary_ratio\n'
            '        <\n'
            '        0.05\n'
            '    ):\n'
            '        continue\n'
            '\n'
            '\n'
            '    gated_mask = (\n'
            '        mask\n'
            '        &\n'
            '        street_surface_support\n'
            '        &\n'
            '        ~MAP_SKY\n'
            '    )\n'
            '\n'
            '\n'
            '    traffic_control_mask |= (\n'
            '        gated_mask\n'
            '    )\n'
            '\n'
            '\n'
            '    accepted_traffic_objects.append({\n'
            '\n'
            '        "score":\n'
            '            obj["score"],\n'
            '\n'
            '        "area_ratio":\n'
            '            area_ratio,\n'
            '\n'
            '        "surface_ratio":\n'
            '            float(\n'
            '                surface_ratio\n'
            '            ),\n'
            '\n'
            '        "orange_ratio":\n'
            '            float(\n'
            '                orange_ratio\n'
            '            ),\n'
            '\n'
            '        "mapillary_ratio":\n'
            '            float(\n'
            '                mapillary_ratio\n'
            '            )\n'
            '    })\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Also accept direct Mapillary traffic-control pixels\n'
            '# if such a class exists\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'traffic_control_mask |= (\n'
            '\n'
            '    MAP_TRAFFIC_CONTROL\n'
            '\n'
            '    &\n'
            '    street_surface_support\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            "# Don't overwrite person / vehicle\n"
            '# ------------------------------------------------------------\n'
            '\n'
            'protected_dynamic = np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        LABEL2ID[\n'
            '            "person"\n'
            '        ],\n'
            '        LABEL2ID[\n'
            '            "vehicle"\n'
            '        ]\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'traffic_control_mask &= (\n'
            '    ~protected_dynamic\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Install class\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'final_working[\n'
            '    traffic_control_mask\n'
            '] = TRAFFIC_CONTROL_ID\n'
            '\n'
            '\n'
            'print(\n'
            '    "Accepted traffic-control objects:",\n'
            '    len(\n'
            '        accepted_traffic_objects\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Traffic cone / barrel pixels:",\n'
            '    int(\n'
            '        traffic_control_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Traffic cone / barrel image %:",\n'
            '    traffic_control_mask.mean()\n'
            '    *\n'
            '    100\n'
            ')\n'
            '\n'
            '\n'
            'if (\n'
            '    len(\n'
            '        accepted_traffic_objects\n'
            '    )\n'
            '    >\n'
            '    0\n'
            '):\n'
            '\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            accepted_traffic_objects\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ Traffic-control class installed"\n'
            ')\n',
 'cell_24': '# ============================================================\n'
            '# GOOGLE STREET VIEW ARTIFACT DETECTION — v1.5 STRICT IGNORE\n'
            '#\n'
            '# SCIENTIFIC RULE:\n'
            '#   Google Street View screenshot UI / watermark residuals are\n'
            '#   INVALID EVIDENCE, not scene objects.\n'
            '#\n'
            '# Therefore:\n'
            '# - detect the residual footprint\n'
            '# - set it to IGNORE=255\n'
            '# - protect it permanently\n'
            '# - NEVER assign a taxonomy class back into these pixels\n'
            '# ============================================================\n'
            '\n'
            'IGNORE_ID = IGNORE_INDEX if "IGNORE_INDEX" in globals() else 255\n'
            '\n'
            'image_np = np.array(image)\n'
            'gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 1. Search lower portion of Street View screenshot.\n'
            '#    Dataset artifacts are dark, vertically persistent UI blocks\n'
            '#    that touch the lower image boundary.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'bottom_search_region = np.zeros((H, W), dtype=bool)\n'
            'bottom_search_region[int(H * 0.58):, :] = True\n'
            '\n'
            'very_dark = gray < 58\n'
            'artifact_dark = gray < 70\n'
            'candidate_dark = artifact_dark & bottom_search_region\n'
            '\n'
            '# Close Google lettering / striped internal gaps.\n'
            'artifact_close_kernel = cv2.getStructuringElement(\n'
            '    cv2.MORPH_RECT,\n'
            '    (9, 21)\n'
            ')\n'
            '\n'
            'candidate_closed = cv2.morphologyEx(\n'
            '    candidate_dark.astype(np.uint8),\n'
            '    cv2.MORPH_CLOSE,\n'
            '    artifact_close_kernel,\n'
            '    iterations=2\n'
            ') > 0\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 2. Connected-component geometry gate\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'num_components, artifact_labels, artifact_stats, artifact_centroids = (\n'
            '    cv2.connectedComponentsWithStats(\n'
            '        candidate_closed.astype(np.uint8),\n'
            '        connectivity=8\n'
            '    )\n'
            ')\n'
            '\n'
            'google_artifact_mask = np.zeros((H, W), dtype=bool)\n'
            'accepted_artifact_components = []\n'
            '\n'
            'for component_id in range(1, num_components):\n'
            '\n'
            '    x = int(artifact_stats[component_id, cv2.CC_STAT_LEFT])\n'
            '    y = int(artifact_stats[component_id, cv2.CC_STAT_TOP])\n'
            '    w = int(artifact_stats[component_id, cv2.CC_STAT_WIDTH])\n'
            '    h = int(artifact_stats[component_id, cv2.CC_STAT_HEIGHT])\n'
            '    area = int(artifact_stats[component_id, cv2.CC_STAT_AREA])\n'
            '\n'
            '    component_mask = artifact_labels == component_id\n'
            '\n'
            '    bottom_touch = (y + h) >= (H - 3)\n'
            '    width_ratio = w / W\n'
            '    height_ratio = h / H\n'
            '    area_ratio = area / (H * W)\n'
            '    fill_ratio = area / max(w * h, 1)\n'
            '    very_dark_ratio = float(very_dark[component_mask].mean())\n'
            '\n'
            '    accept = (\n'
            '        bottom_touch\n'
            '        and y >= H * 0.55\n'
            '        and 0.006 <= width_ratio <= 0.18\n'
            '        and height_ratio >= 0.06\n'
            '        and area_ratio <= 0.040\n'
            '        and fill_ratio >= 0.24\n'
            '        and very_dark_ratio >= 0.35\n'
            '    )\n'
            '\n'
            '    if not accept:\n'
            '        continue\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Protect the WHOLE UI footprint, not only dark pixels.\n'
            '    #\n'
            '    # v1.2 intentionally uses a slightly more conservative\n'
            '    # margin than v1.1 to catch antialiased edges and the pale\n'
            '    # / gray remnants that could otherwise be called "pole".\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    pad_x = max(10, int(round(W * 0.0070)))\n'
            '    pad_top = max(18, int(round(H * 0.0200)))\n'
            '\n'
            '    x0 = max(0, x - pad_x)\n'
            '    x1 = min(W, x + w + pad_x)\n'
            '    y0 = max(0, y - pad_top)\n'
            '    y1 = H\n'
            '\n'
            '    component_bbox_mask = np.zeros((H, W), dtype=bool)\n'
            '    component_bbox_mask[y0:y1, x0:x1] = True\n'
            '\n'
            '    google_artifact_mask |= component_bbox_mask\n'
            '\n'
            '    accepted_artifact_components.append({\n'
            '        "component": int(component_id),\n'
            '        "x": x,\n'
            '        "y": y,\n'
            '        "width": w,\n'
            '        "height": h,\n'
            '        "protected_x0": x0,\n'
            '        "protected_y0": y0,\n'
            '        "protected_x1": x1,\n'
            '        "protected_y1": y1,\n'
            '        "dark_component_pixels": area,\n'
            '        "protected_pixels": int(component_bbox_mask.sum()),\n'
            '        "image_pct": float(component_bbox_mask.mean() * 100.0),\n'
            '        "very_dark_ratio": very_dark_ratio\n'
            '    })\n'
            '\n'
            '# Small final safety expansion for antialiased borders.\n'
            'artifact_expand_kernel = cv2.getStructuringElement(\n'
            '    cv2.MORPH_RECT,\n'
            '    (7, 7)\n'
            ')\n'
            '\n'
            'google_artifact_mask = cv2.dilate(\n'
            '    google_artifact_mask.astype(np.uint8),\n'
            '    artifact_expand_kernel,\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'artifact_ignore_mask = google_artifact_mask.copy()\n'
            '\n'
            '# ============================================================\n'
            '# v1.5.4 BORDER SEMANTIC ANOMALY / FRINGE DETECTOR\n'
            '#\n'
            '# Target failure mode:\n'
            '#\n'
            '# A narrow lower-left / lower-right Google smear fringe can be\n'
            '# missed by image-only artifact detection and survive as:\n'
            '#   upper_building_facade\n'
            '#   ground_floor_solid_facade\n'
            '#   other_unknown\n'
            '#   pole_fixture\n'
            '#\n'
            '# We do NOT blindly mask border pixels.\n'
            '#\n'
            '# A component is accepted only when:\n'
            '# - it lies in the lower border zone;\n'
            '# - it has one of the suspicious semantic labels above;\n'
            '# - it touches bottom or side edge;\n'
            '# - nearby STREET-SURFACE context is sufficiently roadway-dominant.\n'
            '# ============================================================\n'
            '\n'
            'border_semantic_artifact_mask = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'border_semantic_rows = []\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Search zone:\n'
            '# lower 38% of image, only extreme left/right 9% of width.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'border_zone = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'border_y0 = int(\n'
            '    H * 0.62\n'
            ')\n'
            '\n'
            'border_w = max(\n'
            '    48,\n'
            '    int(round(W * 0.09))\n'
            ')\n'
            '\n'
            'border_zone[\n'
            '    border_y0:H,\n'
            '    0:border_w\n'
            '] = True\n'
            '\n'
            'border_zone[\n'
            '    border_y0:H,\n'
            '    max(0, W - border_w):W\n'
            '] = True\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Semantic labels that are suspicious specifically at a\n'
            '# bottom street-view border.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'border_suspicious_ids = [\n'
            '    LABEL2ID[name]\n'
            '    for name in [\n'
            '        "upper_building_facade",\n'
            '        "ground_floor_solid_facade",\n'
            '        "other_unknown",\n'
            '        "pole_fixture",\n'
            '    ]\n'
            '    if name in LABEL2ID\n'
            ']\n'
            '\n'
            'border_street_ids = [\n'
            '    LABEL2ID[name]\n'
            '    for name in [\n'
            '        "roadway",\n'
            '        "sidewalk",\n'
            '        "bike_lane",\n'
            '        "curb_edge",\n'
            '    ]\n'
            '    if name in LABEL2ID\n'
            ']\n'
            '\n'
            'ROAD_ID_BORDER = LABEL2ID[\n'
            '    "roadway"\n'
            ']\n'
            '\n'
            'SIDEWALK_ID_BORDER = LABEL2ID[\n'
            '    "sidewalk"\n'
            ']\n'
            '\n'
            'CURB_ID_BORDER = LABEL2ID[\n'
            '    "curb_edge"\n'
            ']\n'
            '\n'
            '\n'
            'border_semantic_candidate = (\n'
            '    border_zone\n'
            '    &\n'
            '    np.isin(\n'
            '        final_working,\n'
            '        border_suspicious_ids\n'
            '    )\n'
            '    &\n'
            '    ~artifact_ignore_mask\n'
            ')\n'
            '\n'
            '\n'
            '# Join tiny gaps inside a vertical smear fringe.\n'
            'border_semantic_candidate = cv2.morphologyEx(\n'
            '    border_semantic_candidate.astype(np.uint8),\n'
            '    cv2.MORPH_CLOSE,\n'
            '    cv2.getStructuringElement(\n'
            '        cv2.MORPH_RECT,\n'
            '        (5, 17)\n'
            '    ),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            '\n'
            'n_bs, bs_lab, bs_stats, _ = cv2.connectedComponentsWithStats(\n'
            '    border_semantic_candidate.astype(np.uint8),\n'
            '    connectivity=8\n'
            ')\n'
            '\n'
            '\n'
            'for cid in range(\n'
            '    1,\n'
            '    n_bs\n'
            '):\n'
            '\n'
            '    comp = (\n'
            '        bs_lab == cid\n'
            '    )\n'
            '\n'
            '    area = int(\n'
            '        comp.sum()\n'
            '    )\n'
            '\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(\n'
            '        bs_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_LEFT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    y = int(\n'
            '        bs_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_TOP\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    w = int(\n'
            '        bs_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_WIDTH\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    h = int(\n'
            '        bs_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_HEIGHT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    touches_bottom = (\n'
            '        y + h\n'
            '        >=\n'
            '        H - 2\n'
            '    )\n'
            '\n'
            '    touches_left = (\n'
            '        x <= 2\n'
            '    )\n'
            '\n'
            '    touches_right = (\n'
            '        x + w\n'
            '        >=\n'
            '        W - 2\n'
            '    )\n'
            '\n'
            '    touches_border = (\n'
            '        touches_bottom\n'
            '        or\n'
            '        touches_left\n'
            '        or\n'
            '        touches_right\n'
            '    )\n'
            '\n'
            '    if not touches_border:\n'
            '        continue\n'
            '\n'
            '    area_ratio = (\n'
            '        area\n'
            '        /\n'
            '        (H * W)\n'
            '    )\n'
            '\n'
            '    width_ratio = (\n'
            '        w\n'
            '        /\n'
            '        W\n'
            '    )\n'
            '\n'
            '    height_ratio = (\n'
            '        h\n'
            '        /\n'
            '        H\n'
            '    )\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Local context ring around suspicious component.\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    margin_x = max(\n'
            '        35,\n'
            '        int(round(W * 0.035))\n'
            '    )\n'
            '\n'
            '    margin_y = max(\n'
            '        25,\n'
            '        int(round(H * 0.035))\n'
            '    )\n'
            '\n'
            '    x0 = max(\n'
            '        0,\n'
            '        x - margin_x\n'
            '    )\n'
            '\n'
            '    x1 = min(\n'
            '        W,\n'
            '        x + w + margin_x\n'
            '    )\n'
            '\n'
            '    y0 = max(\n'
            '        border_y0,\n'
            '        y - margin_y\n'
            '    )\n'
            '\n'
            '    y1 = min(\n'
            '        H,\n'
            '        y + h + margin_y\n'
            '    )\n'
            '\n'
            '    local_labels = final_working[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_artifact = artifact_ignore_mask[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_component = comp[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_valid_street = (\n'
            '        ~local_artifact\n'
            '        &\n'
            '        ~local_component\n'
            '        &\n'
            '        np.isin(\n'
            '            local_labels,\n'
            '            border_street_ids\n'
            '        )\n'
            '    )\n'
            '\n'
            '    street_labels = local_labels[\n'
            '        local_valid_street\n'
            '    ]\n'
            '\n'
            '    road_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            ROAD_ID_BORDER\n'
            '        )\n'
            '    )\n'
            '\n'
            '    sidewalk_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            SIDEWALK_ID_BORDER\n'
            '        )\n'
            '    )\n'
            '\n'
            '    curb_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            CURB_ID_BORDER\n'
            '        )\n'
            '    )\n'
            '\n'
            '    street_votes = int(\n'
            '        len(\n'
            '            street_labels\n'
            '        )\n'
            '    )\n'
            '\n'
            '    road_share = (\n'
            '        road_votes\n'
            '        /\n'
            '        street_votes\n'
            '        if street_votes > 0\n'
            '        else 0.0\n'
            '    )\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Conservative acceptance:\n'
            '    #\n'
            '    # - small/narrow border anomaly\n'
            '    # - tall enough to look like a smear fringe\n'
            '    # - enough local street evidence\n'
            '    # - roadway dominates context\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    narrow_enough = (\n'
            '        width_ratio <= 0.085\n'
            '    )\n'
            '\n'
            '    tall_enough = (\n'
            '        height_ratio >= 0.025\n'
            '    )\n'
            '\n'
            '    not_huge = (\n'
            '        area_ratio <= 0.030\n'
            '    )\n'
            '\n'
            '    roadway_context = (\n'
            '        street_votes >= 30\n'
            '        and\n'
            '        road_share >= 0.52\n'
            '        and\n'
            '        road_votes\n'
            '        >=\n'
            '        (\n'
            '            sidewalk_votes\n'
            '            +\n'
            '            curb_votes\n'
            '        )\n'
            '    )\n'
            '\n'
            '    accept = (\n'
            '        touches_border\n'
            '        and\n'
            '        narrow_enough\n'
            '        and\n'
            '        tall_enough\n'
            '        and\n'
            '        not_huge\n'
            '        and\n'
            '        roadway_context\n'
            '    )\n'
            '\n'
            '    if accept:\n'
            '\n'
            '        # -----------------------------------------------\n'
            '        # Complete the suspicious fringe into a compact\n'
            '        # rectangular bottom footprint.\n'
            '        # -----------------------------------------------\n'
            '\n'
            '        pad_x = max(\n'
            '            6,\n'
            '            int(round(W * 0.004))\n'
            '        )\n'
            '\n'
            '        pad_top = max(\n'
            '            8,\n'
            '            int(round(H * 0.012))\n'
            '        )\n'
            '\n'
            '        fx0 = max(\n'
            '            0,\n'
            '            x - pad_x\n'
            '        )\n'
            '\n'
            '        fx1 = min(\n'
            '            W,\n'
            '            x + w + pad_x\n'
            '        )\n'
            '\n'
            '        fy0 = max(\n'
            '            border_y0,\n'
            '            y - pad_top\n'
            '        )\n'
            '\n'
            '        fy1 = H\n'
            '\n'
            '        border_semantic_artifact_mask[\n'
            '            fy0:fy1,\n'
            '            fx0:fx1\n'
            '        ] = True\n'
            '\n'
            '    border_semantic_rows.append({\n'
            '        "component": cid,\n'
            '        "x": x,\n'
            '        "y": y,\n'
            '        "width": w,\n'
            '        "height": h,\n'
            '        "pixels": area,\n'
            '        "width_ratio": width_ratio,\n'
            '        "height_ratio": height_ratio,\n'
            '        "road_votes": road_votes,\n'
            '        "sidewalk_votes": sidewalk_votes,\n'
            '        "curb_votes": curb_votes,\n'
            '        "street_votes": street_votes,\n'
            '        "road_share": road_share,\n'
            '        "accepted_as_artifact_fringe": accept,\n'
            '    })\n'
            '\n'
            '\n'
            '# Union semantic fringe with image-based artifact mask.\n'
            'artifact_ignore_mask |= (\n'
            '    border_semantic_artifact_mask\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Border semantic artifact fringe pixels:",\n'
            '    int(\n'
            '        border_semantic_artifact_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'if border_semantic_rows:\n'
            '\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            border_semantic_rows\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '\n'
            '# Persist ONLY the successful border-fringe capture into the\n'
            '# canonical Google artifact mask. No later footprint expansion.\n'
            'google_artifact_mask = artifact_ignore_mask.copy()\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 3. Install PERMANENT IGNORE\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'artifact_pre_ignore_labels = final_working.copy()\n'
            'final_working[artifact_ignore_mask] = IGNORE_ID\n'
            '\n'
            'print("Detected Google/UI artifact components:", len(accepted_artifact_components))\n'
            'print("STRICT IGNORE pixels:", int(artifact_ignore_mask.sum()))\n'
            'print("STRICT IGNORE image %:", float(artifact_ignore_mask.mean() * 100.0))\n'
            '\n'
            'if accepted_artifact_components:\n'
            '    display(pd.DataFrame(accepted_artifact_components))\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 4. Visual QA\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'artifact_preview = image_np.copy()\n'
            'artifact_preview[artifact_ignore_mask] = (255, 0, 255)\n'
            '\n'
            '# Cyan = the v1.5.4 border semantic fringe capture only.\n'
            'artifact_preview[\n'
            '    border_semantic_artifact_mask\n'
            '] = (\n'
            '    0,\n'
            '    255,\n'
            '    255\n'
            ')\n'
            '\n'
            '# [batch] intermediate matplotlib QA figure skipped\n'
            '\n'
            '\n'
            'print("✓ Google/UI artifact is permanently IGNORE=255")\n'
            'print("✓ These pixels must never contribute to pole / road / sidewalk / any taxonomy class")\n',
 'cell_25': '# ============================================================\n'
            '# COMMIT PRE-REFINEMENT RESULT — v1.4\n'
            '#\n'
            '# Compatibility variable name final_v010 is retained because\n'
            '# later stable cells use it, but its contents are now:\n'
            '#\n'
            '#   v0.9 semantic backbone\n'
            '#   + traffic-cone refinement\n'
            '#   + Google artifact IGNORE\n'
            '#\n'
            '# It does NOT contain the legacy v0.10 tree overwrite.\n'
            '# ============================================================\n'
            '\n'
            'final_v010 = (\n'
            '    final_working.copy()\n'
            ')\n'
            '\n'
            'print(\n'
            '    "✓ v1.4 pre-refinement state committed"\n'
            ')\n',
 'cell_26': '# ============================================================\n'
            '# SCAFFOLD PRECISION FIX — v1.3 GLASS CURTAIN-WALL VETO\n'
            '#\n'
            '# Goal:\n'
            '# - retain real temporary sidewalk shed / scaffold\n'
            '# - reject glass-curtain-wall / glazed-façade false positives\n'
            '# - restore rejected glass-like pixels to glazing, not UNKNOWN\n'
            '# ============================================================\n'
            '\n'
            'final_working = final_v010.copy()\n'
            '\n'
            'SCAFFOLD_ID = LABEL2ID["sidewalk_shed_scaffold"]\n'
            'ROAD_ID = LABEL2ID["roadway"]\n'
            'SIDEWALK_ID = LABEL2ID["sidewalk"]\n'
            'CURB_ID = LABEL2ID["curb_edge"]\n'
            'UPPER_FACADE_ID = LABEL2ID["upper_building_facade"]\n'
            'UPPER_GLASS_ID = LABEL2ID["upper_building_glazing"]\n'
            'GROUND_GLASS_ID = LABEL2ID["ground_floor_glazing"]\n'
            '\n'
            'current_scaffold = final_working == SCAFFOLD_ID\n'
            'print("Scaffold before:", int(current_scaffold.sum()), "pixels")\n'
            '\n'
            'building_support = MAP_BUILDING | ADE_BUILDING_V2\n'
            'building_near = cv2.dilate(\n'
            '    building_support.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (61, 41)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            '# Strong glass evidence from three independent paths.\n'
            'glazing_support = (\n'
            '    (ADE_GLASS_V2 | MAP_WINDOW | precise_window_mask | facade_glass_aggressive)\n'
            '    & building_support\n'
            ')\n'
            '\n'
            '# Stronger DINO scaffold evidence than the original fusion threshold.\n'
            'strong_scaffold_dino = np.zeros((H, W), dtype=bool)\n'
            'for obj in detail_objects:\n'
            '    if (\n'
            '        obj["class_name"] == "sidewalk_shed_scaffold"\n'
            '        and obj["score"] >= 0.32\n'
            '    ):\n'
            '        strong_scaffold_dino |= obj["mask"]\n'
            '\n'
            'num_labels, labels_scaf, stats_scaf, _ = cv2.connectedComponentsWithStats(\n'
            '    current_scaffold.astype(np.uint8), connectivity=8\n'
            ')\n'
            '\n'
            'accepted_scaffold = np.zeros((H, W), dtype=bool)\n'
            'scaffold_diagnostics = []\n'
            '\n'
            'for cid in range(1, num_labels):\n'
            '    component = labels_scaf == cid\n'
            '    area = int(component.sum())\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(stats_scaf[cid, cv2.CC_STAT_LEFT])\n'
            '    y = int(stats_scaf[cid, cv2.CC_STAT_TOP])\n'
            '    w = int(stats_scaf[cid, cv2.CC_STAT_WIDTH])\n'
            '    h = int(stats_scaf[cid, cv2.CC_STAT_HEIGHT])\n'
            '\n'
            '    area_ratio = area / (H * W)\n'
            '    building_ratio = float(building_near[component].mean())\n'
            '    sidewalk_ratio = float(MAP_SIDEWALK[component].mean())\n'
            '    glazing_ratio = float(glazing_support[component].mean())\n'
            '    strong_dino_ratio = float(strong_scaffold_dino[component].mean())\n'
            '    cy = (y + h / 2) / H\n'
            '    vertical_ratio = h / max(w, 1)\n'
            '\n'
            '    # Typical curtain-wall false positive:\n'
            '    # vertically tall + glass-dominant + little sidewalk evidence.\n'
            '    glass_tower_like = (\n'
            '        glazing_ratio >= 0.22\n'
            '        and vertical_ratio >= 1.10\n'
            '        and sidewalk_ratio < 0.08\n'
            '    )\n'
            '\n'
            '    glass_panel_like = (\n'
            '        glazing_ratio >= 0.42\n'
            '        and strong_dino_ratio < 0.18\n'
            '    )\n'
            '\n'
            '    scaffold_evidence = (\n'
            '        strong_dino_ratio >= 0.10\n'
            '        or sidewalk_ratio >= 0.04\n'
            '    )\n'
            '\n'
            '    accept = (\n'
            '        building_ratio >= 0.22\n'
            '        and area_ratio <= 0.08\n'
            '        and 0.20 <= cy <= 0.88\n'
            '        and scaffold_evidence\n'
            '        and not glass_tower_like\n'
            '        and not glass_panel_like\n'
            '    )\n'
            '\n'
            '    if accept:\n'
            '        accepted_scaffold[component] = True\n'
            '\n'
            '    scaffold_diagnostics.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "image_pct": area_ratio * 100,\n'
            '        "building_support": building_ratio,\n'
            '        "sidewalk_support": sidewalk_ratio,\n'
            '        "glazing_support": glazing_ratio,\n'
            '        "strong_scaffold_dino": strong_dino_ratio,\n'
            '        "vertical_ratio": vertical_ratio,\n'
            '        "center_y": cy,\n'
            '        "glass_tower_veto": glass_tower_like,\n'
            '        "glass_panel_veto": glass_panel_like,\n'
            '        "accepted": accept,\n'
            '    })\n'
            '\n'
            'rejected_scaffold = current_scaffold & ~accepted_scaffold\n'
            'print("Accepted scaffold:", int(accepted_scaffold.sum()))\n'
            'print("Rejected scaffold:", int(rejected_scaffold.sum()))\n'
            'if scaffold_diagnostics:\n'
            '    display(pd.DataFrame(scaffold_diagnostics))\n'
            '\n'
            '# Reset rejected scaffold first.\n'
            'final_working[rejected_scaffold] = UNKNOWN_ID\n'
            '\n'
            '# Restore reliable street surfaces.\n'
            'final_working[rejected_scaffold & MAP_ROAD] = ROAD_ID\n'
            'final_working[rejected_scaffold & MAP_SIDEWALK] = SIDEWALK_ID\n'
            'final_working[rejected_scaffold & MAP_CURB] = CURB_ID\n'
            '\n'
            '# Restore building surface. Glazing gets priority over opaque façade.\n'
            'rejected_building = rejected_scaffold & building_support\n'
            'final_working[rejected_building] = UPPER_FACADE_ID\n'
            '\n'
            'rejected_glass = rejected_scaffold & glazing_support\n'
            '\n'
            'ground_glass_recovery = rejected_glass & interface_neighborhood\n'
            'upper_glass_recovery = rejected_glass & ~interface_neighborhood\n'
            '\n'
            'final_working[ground_glass_recovery] = GROUND_GLASS_ID\n'
            'final_working[upper_glass_recovery] = UPPER_GLASS_ID\n'
            '\n'
            '# Keep only scaffold components that survive the glass veto.\n'
            'final_working[accepted_scaffold] = SCAFFOLD_ID\n'
            '\n'
            'print("Recovered rejected scaffold as upper glazing:", int(upper_glass_recovery.sum()))\n'
            'print("Recovered rejected scaffold as ground glazing:", int(ground_glass_recovery.sum()))\n'
            'print("✓ v1.3 scaffold glass-curtain-wall safeguard applied")\n',
 'cell_27': '# ============================================================\n'
            '# FENCE / RAILING RECOVERY — v1.3\n'
            '#\n'
            '# Purpose:\n'
            '# Recover park perimeter fences / railings that remain\n'
            '# other_unknown, without performing generic unknown filling.\n'
            '# ============================================================\n'
            '\n'
            'FENCE_ID = LABEL2ID["fence_railing"]\n'
            '\n'
            '# 1. Semantic-model evidence\n'
            'fence_semantic = MAP_FENCE | ADE_FENCE_V3\n'
            '\n'
            '# 2. DINO + SAM fence evidence\n'
            'fence_dino = np.zeros((H, W), dtype=bool)\n'
            'for obj in detail_objects:\n'
            '    if (\n'
            '        obj["class_name"] == "fence_railing"\n'
            '        and obj["score"] >= 0.18\n'
            '    ):\n'
            '        fence_dino |= obj["mask"]\n'
            '\n'
            '# 3. Park/street-edge context: fence often runs next to sidewalk.\n'
            'sidewalk_near_fence = cv2.dilate(\n'
            '    MAP_SIDEWALK.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (101, 61)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'fence_candidate = (\n'
            '    fence_semantic\n'
            '    |\n'
            '    (fence_dino & sidewalk_near_fence)\n'
            ')\n'
            '\n'
            '# Fence cannot be road, sky, glass façade, or Google UI artifact.\n'
            'fence_candidate &= ~MAP_ROAD\n'
            'fence_candidate &= ~MAP_SKY\n'
            'fence_candidate &= ~ADE_GLASS_V2\n'
            'fence_candidate &= ~MAP_WINDOW\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    fence_candidate &= ~artifact_ignore_mask\n'
            '\n'
            '# Preserve already reliable foreground/object classes.\n'
            'protected_fence_classes = np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        LABEL2ID["person"],\n'
            '        LABEL2ID["vehicle"],\n'
            '        LABEL2ID["traffic_cone_barrel"],\n'
            '        LABEL2ID["traffic_sign_signal"],\n'
            '        LABEL2ID["sidewalk_shed_scaffold"],\n'
            '        IGNORE_ID,\n'
            '    ]\n'
            ')\n'
            'fence_candidate &= ~protected_fence_classes\n'
            '\n'
            '# Deliberately conservative: v1.3 recovery only converts pixels\n'
            '# that are still unknown (plus any already-fence pixels).\n'
            'fence_recovery = (\n'
            '    fence_candidate\n'
            '    & np.isin(final_working, [UNKNOWN_ID, FENCE_ID])\n'
            ')\n'
            '\n'
            '# Remove tiny isolated speckle while retaining thin openwork fence lines.\n'
            'num_f, lab_f, stats_f, _ = cv2.connectedComponentsWithStats(\n'
            '    fence_recovery.astype(np.uint8), connectivity=8\n'
            ')\n'
            'fence_precise = np.zeros((H, W), dtype=bool)\n'
            'fence_rows = []\n'
            '\n'
            'for cid in range(1, num_f):\n'
            '    comp = lab_f == cid\n'
            '    area = int(comp.sum())\n'
            '    x = int(stats_f[cid, cv2.CC_STAT_LEFT])\n'
            '    y = int(stats_f[cid, cv2.CC_STAT_TOP])\n'
            '    w = int(stats_f[cid, cv2.CC_STAT_WIDTH])\n'
            '    h = int(stats_f[cid, cv2.CC_STAT_HEIGHT])\n'
            '    cy = (y + h/2) / H\n'
            '    area_ratio = area / (H*W)\n'
            '    semantic_ratio = float(fence_semantic[comp].mean()) if area else 0.0\n'
            '    dino_ratio = float(fence_dino[comp].mean()) if area else 0.0\n'
            '\n'
            '    accept = (\n'
            '        area >= 8\n'
            '        and area_ratio <= 0.035\n'
            '        and 0.18 <= cy <= 0.90\n'
            '        and (semantic_ratio >= 0.05 or dino_ratio >= 0.05)\n'
            '    )\n'
            '\n'
            '    if accept:\n'
            '        fence_precise[comp] = True\n'
            '\n'
            '    fence_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "width": w,\n'
            '        "height": h,\n'
            '        "center_y": cy,\n'
            '        "semantic_support": semantic_ratio,\n'
            '        "dino_support": dino_ratio,\n'
            '        "accepted": accept,\n'
            '    })\n'
            '\n'
            'unknown_before_fence = int(np.sum(final_working == UNKNOWN_ID))\n'
            'final_working[fence_precise] = FENCE_ID\n'
            'unknown_after_fence = int(np.sum(final_working == UNKNOWN_ID))\n'
            '\n'
            'print("Fence semantic pixels:", int(fence_semantic.sum()))\n'
            'print("Fence DINO/SAM pixels:", int(fence_dino.sum()))\n'
            'print("Recovered fence pixels:", int(fence_precise.sum()))\n'
            'print("Unknown reduction from fence recovery:", unknown_before_fence - unknown_after_fence)\n'
            'if fence_rows:\n'
            '    display(pd.DataFrame(fence_rows))\n'
            'print("✓ v1.3 fence / railing recovery applied")\n',
 'cell_28': '# ============================================================\n'
            '# v1.5 LOCAL RECOVERY\n'
            '#\n'
            '# A. entrance awning / canopy\n'
            '# B. solid boundary wall → wall_ledge\n'
            '# C. precise woody trunk / bare branch → tree\n'
            '#\n'
            '# No broad foliage write is permitted.\n'
            '# ============================================================\n'
            '\n'
            'AWNING_ID = LABEL2ID["awning_canopy"]\n'
            'WALL_ID = LABEL2ID["wall_ledge"]\n'
            'TREE_ID = LABEL2ID["tree"]\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Shared semantic helpers\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'ADE_WALL_V15 = ade_mask_fuzzy(\n'
            '    ["wall"],\n'
            '    ["building"]\n'
            ')\n'
            '\n'
            'MAP_WALL_V15 = mapillary_mask(\n'
            '    ["wall"]\n'
            ')\n'
            '\n'
            'MAP_POLE_V15 = mapillary_mask(\n'
            '    ["pole"]\n'
            ')\n'
            '\n'
            'MAP_VEGETATION_V15 = mapillary_mask(\n'
            '    ["vegetation", "tree", "plant"]\n'
            ')\n'
            '\n'
            'image_rgb_v15 = np.array(image)\n'
            'gray_v15 = cv2.cvtColor(\n'
            '    image_rgb_v15,\n'
            '    cv2.COLOR_RGB2GRAY\n'
            ')\n'
            '\n'
            '# ============================================================\n'
            '# v1.5.6 SCAFFOLD PROTECTED ZONE\n'
            '#\n'
            '# Purpose:\n'
            '# Prevent a real sidewalk shed / scaffold from being fragmented\n'
            '# into awning_canopy + wall_ledge by later local recovery.\n'
            '#\n'
            '# This is intentionally LOCAL:\n'
            '#\n'
            '# accepted_scaffold\n'
            '#   +\n'
            '# strong_scaffold_dino only where it remains close to an\n'
            '# already accepted scaffold component.\n'
            '#\n'
            '# It does NOT expand scaffold across the whole frontage.\n'
            '# ============================================================\n'
            '\n'
            'SCAFFOLD_LOCAL_ID = LABEL2ID[\n'
            '    "sidewalk_shed_scaffold"\n'
            ']\n'
            '\n'
            'if (\n'
            '    "accepted_scaffold" in globals()\n'
            '    and\n'
            '    "strong_scaffold_dino" in globals()\n'
            '):\n'
            '\n'
            '    scaffold_anchor_near = cv2.dilate(\n'
            '        accepted_scaffold.astype(np.uint8),\n'
            '        cv2.getStructuringElement(\n'
            '            cv2.MORPH_RECT,\n'
            '            (35, 25)\n'
            '        ),\n'
            '        iterations=1\n'
            '    ) > 0\n'
            '\n'
            '    scaffold_protected_zone = (\n'
            '        accepted_scaffold\n'
            '        |\n'
            '        (\n'
            '            strong_scaffold_dino\n'
            '            &\n'
            '            scaffold_anchor_near\n'
            '        )\n'
            '    )\n'
            '\n'
            'else:\n'
            '\n'
            '    scaffold_protected_zone = np.zeros(\n'
            '        (H, W),\n'
            '        dtype=bool\n'
            '    )\n'
            '\n'
            '\n'
            '# Protect Google artifact / dynamic objects from this guard.\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    scaffold_protected_zone &= ~artifact_ignore_mask\n'
            '\n'
            'scaffold_protected_zone &= ~np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        LABEL2ID["person"],\n'
            '        LABEL2ID["vehicle"],\n'
            '        LABEL2ID["traffic_cone_barrel"],\n'
            '        LABEL2ID["traffic_sign_signal"],\n'
            '        IGNORE_ID,\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            '# Reclaim any pre-existing awning / wall label that sits inside\n'
            '# evidence-backed scaffold structure.\n'
            'preexisting_scaffold_conflict = (\n'
            '    scaffold_protected_zone\n'
            '    &\n'
            '    np.isin(\n'
            '        final_working,\n'
            '        [\n'
            '            AWNING_ID,\n'
            '            WALL_ID,\n'
            '        ]\n'
            '    )\n'
            ')\n'
            '\n'
            'final_working[\n'
            '    preexisting_scaffold_conflict\n'
            '] = SCAFFOLD_LOCAL_ID\n'
            '\n'
            '\n'
            'print("========================================")\n'
            'print("SCAFFOLD PROTECTED ZONE — v1.5.6")\n'
            'print("========================================")\n'
            'print(\n'
            '    "Protected scaffold-context pixels:",\n'
            '    int(scaffold_protected_zone.sum())\n'
            ')\n'
            'print(\n'
            '    "Pre-existing awning/wall pixels reclaimed:",\n'
            '    int(preexisting_scaffold_conflict.sum())\n'
            ')\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# A. ENTRANCE AWNING RECOVERY\n'
            '# ============================================================\n'
            '\n'
            'awning_dino = np.zeros((H, W), dtype=bool)\n'
            'awning_high = np.zeros((H, W), dtype=bool)\n'
            '\n'
            'for obj in detail_objects:\n'
            '    if obj["class_name"] != "awning_canopy":\n'
            '        continue\n'
            '\n'
            '    if obj["score"] >= 0.18:\n'
            '        awning_dino |= obj["mask"]\n'
            '\n'
            '    if obj["score"] >= 0.24:\n'
            '        awning_high |= obj["mask"]\n'
            '\n'
            'building_near_awning = cv2.dilate(\n'
            '    building_context.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (51, 41)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'interface_near_awning = cv2.dilate(\n'
            '    interface_neighborhood.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (31, 25)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'awning_candidate = (\n'
            '    awning_dino\n'
            '    &\n'
            '    building_near_awning\n'
            '    &\n'
            '    interface_near_awning\n'
            ')\n'
            '\n'
            '# Do not overwrite street surfaces or research foreground objects.\n'
            'awning_candidate &= ~MAP_ROAD\n'
            'awning_candidate &= ~MAP_SIDEWALK\n'
            'awning_candidate &= ~MAP_SKY\n'
            '\n'
            '# v1.5.6: scaffold evidence has precedence over awning recovery.\n'
            'awning_candidate &= ~scaffold_protected_zone\n'
            '\n'
            'awning_candidate &= ~np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        LABEL2ID["person"],\n'
            '        LABEL2ID["vehicle"],\n'
            '        LABEL2ID["traffic_cone_barrel"],\n'
            '        LABEL2ID["traffic_sign_signal"],\n'
            '        LABEL2ID["sidewalk_shed_scaffold"],\n'
            '        IGNORE_ID,\n'
            '    ]\n'
            ')\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    awning_candidate &= ~artifact_ignore_mask\n'
            '\n'
            'num_a, lab_a, stats_a, _ = cv2.connectedComponentsWithStats(\n'
            '    awning_candidate.astype(np.uint8),\n'
            '    connectivity=8\n'
            ')\n'
            '\n'
            'awning_precise = np.zeros((H, W), dtype=bool)\n'
            'awning_rows = []\n'
            '\n'
            'for cid in range(1, num_a):\n'
            '    comp = lab_a == cid\n'
            '    area = int(comp.sum())\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(stats_a[cid, cv2.CC_STAT_LEFT])\n'
            '    y = int(stats_a[cid, cv2.CC_STAT_TOP])\n'
            '    w = int(stats_a[cid, cv2.CC_STAT_WIDTH])\n'
            '    h = int(stats_a[cid, cv2.CC_STAT_HEIGHT])\n'
            '\n'
            '    area_ratio = area / (H * W)\n'
            '    aspect = w / max(h, 1)\n'
            '    cy = (y + h / 2) / H\n'
            '    building_ratio = float(building_near_awning[comp].mean())\n'
            '    interface_ratio = float(interface_near_awning[comp].mean())\n'
            '    high_ratio = float(awning_high[comp].mean())\n'
            '\n'
            '    keep = (\n'
            '        area >= 20\n'
            '        and area_ratio <= 0.030\n'
            '        and 0.22 <= cy <= 0.82\n'
            '        and aspect >= 0.90\n'
            '        and building_ratio >= 0.45\n'
            '        and interface_ratio >= 0.15\n'
            '        and high_ratio >= 0.02\n'
            '    )\n'
            '\n'
            '    if keep:\n'
            '        awning_precise[comp] = True\n'
            '\n'
            '    awning_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "aspect_w_h": aspect,\n'
            '        "center_y": cy,\n'
            '        "building_support": building_ratio,\n'
            '        "interface_support": interface_ratio,\n'
            '        "high_conf_support": high_ratio,\n'
            '        "accepted": keep,\n'
            '    })\n'
            '\n'
            'final_working[awning_precise] = AWNING_ID\n'
            '\n'
            'print("========================================")\n'
            'print("AWNING RECOVERY")\n'
            'print("========================================")\n'
            'print("Recovered awning pixels:", int(awning_precise.sum()))\n'
            'if awning_rows:\n'
            '    display(pd.DataFrame(awning_rows))\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# B. SOLID BOUNDARY WALL RECOVERY\n'
            '#\n'
            '# The taxonomy already contains wall_ledge for low / raised\n'
            '# solid walls.  Recover only pixels still marked other_unknown.\n'
            '# ============================================================\n'
            '\n'
            'wall_dino = np.zeros((H, W), dtype=bool)\n'
            'for obj in detail_objects:\n'
            '    if (\n'
            '        obj["class_name"] == "wall_ledge"\n'
            '        and obj["score"] >= 0.18\n'
            '    ):\n'
            '        wall_dino |= obj["mask"]\n'
            '\n'
            'wall_semantic = (\n'
            '    ADE_WALL_V15\n'
            '    |\n'
            '    MAP_WALL_V15\n'
            ')\n'
            '\n'
            'wall_sidewalk_near = cv2.dilate(\n'
            '    MAP_SIDEWALK.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (111, 71)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'wall_candidate = (\n'
            '    (wall_semantic | wall_dino)\n'
            '    &\n'
            '    wall_sidewalk_near\n'
            '    &\n'
            '    (final_working == UNKNOWN_ID)\n'
            ')\n'
            '\n'
            'wall_candidate &= ~MAP_ROAD\n'
            'wall_candidate &= ~MAP_SKY\n'
            'wall_candidate &= ~MAP_WINDOW\n'
            'wall_candidate &= ~ADE_GLASS_V2\n'
            'wall_candidate &= ~(final_working == TREE_ID)\n'
            '\n'
            '# v1.5.6: evidence-backed scaffold zone has precedence over\n'
            '# wall_ledge recovery.\n'
            'wall_candidate &= ~scaffold_protected_zone\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    wall_candidate &= ~artifact_ignore_mask\n'
            '\n'
            'num_w, lab_w, stats_w, _ = cv2.connectedComponentsWithStats(\n'
            '    wall_candidate.astype(np.uint8),\n'
            '    connectivity=8\n'
            ')\n'
            '\n'
            'wall_precise = np.zeros((H, W), dtype=bool)\n'
            'wall_rows = []\n'
            '\n'
            'for cid in range(1, num_w):\n'
            '    comp = lab_w == cid\n'
            '    area = int(comp.sum())\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(stats_w[cid, cv2.CC_STAT_LEFT])\n'
            '    y = int(stats_w[cid, cv2.CC_STAT_TOP])\n'
            '    w = int(stats_w[cid, cv2.CC_STAT_WIDTH])\n'
            '    h = int(stats_w[cid, cv2.CC_STAT_HEIGHT])\n'
            '\n'
            '    area_ratio = area / (H * W)\n'
            '    cy = (y + h / 2) / H\n'
            '    semantic_ratio = float(wall_semantic[comp].mean())\n'
            '    dino_ratio = float(wall_dino[comp].mean())\n'
            '\n'
            '    keep = (\n'
            '        area >= 20\n'
            '        and area_ratio <= 0.090\n'
            '        and 0.18 <= cy <= 0.92\n'
            '        and (\n'
            '            semantic_ratio >= 0.05\n'
            '            or dino_ratio >= 0.04\n'
            '        )\n'
            '    )\n'
            '\n'
            '    if keep:\n'
            '        wall_precise[comp] = True\n'
            '\n'
            '    wall_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "center_y": cy,\n'
            '        "semantic_support": semantic_ratio,\n'
            '        "dino_support": dino_ratio,\n'
            '        "accepted": keep,\n'
            '    })\n'
            '\n'
            'final_working[wall_precise] = WALL_ID\n'
            '\n'
            'print()\n'
            'print("========================================")\n'
            'print("BOUNDARY WALL RECOVERY")\n'
            'print("========================================")\n'
            'print("Recovered wall_ledge pixels:", int(wall_precise.sum()))\n'
            'if wall_rows:\n'
            '    display(pd.DataFrame(wall_rows))\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# C. PRECISE WOODY TREE RECOVERY\n'
            '#\n'
            '# v1.4 prohibited all DINO/SAM tree writes because broad SAM\n'
            '# foliage contaminated glazing.\n'
            '#\n'
            '# v1.5 restores ONLY trunk / bare-branch structure.\n'
            '#\n'
            '# Requirements:\n'
            '# - inside a woody DINO/SAM proposal\n'
            '# - dark and/or edge-like image structure\n'
            '# - near an existing semantic tree anchor OR high-confidence\n'
            '#   woody proposal with strong elongated/sparse geometry\n'
            '# - never Google UI / road / sidewalk / dynamic object\n'
            '#\n'
            '# Broad foliage remains diagnostic only.\n'
            '# ============================================================\n'
            '\n'
            'raw_woody_v15 = np.zeros((H, W), dtype=bool)\n'
            'high_woody_v15 = np.zeros((H, W), dtype=bool)\n'
            '\n'
            'for obj in tree_objects:\n'
            '    if obj["component"] != "woody":\n'
            '        continue\n'
            '\n'
            '    if obj["score"] >= 0.16:\n'
            '        raw_woody_v15 |= obj["mask"]\n'
            '\n'
            '    if obj["score"] >= 0.22:\n'
            '        high_woody_v15 |= obj["mask"]\n'
            '\n'
            'tree_semantic_anchor = (\n'
            '    backbone_tree_mask\n'
            '    |\n'
            '    ADE_TREE_V2\n'
            '    |\n'
            '    MAP_VEGETATION_V15\n'
            ')\n'
            '\n'
            'tree_anchor_near = cv2.dilate(\n'
            '    tree_semantic_anchor.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (101, 101)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'tree_anchor_contact = cv2.dilate(\n'
            '    tree_semantic_anchor.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'edge_v15 = cv2.Canny(\n'
            '    gray_v15,\n'
            '    45,\n'
            '    135\n'
            ') > 0\n'
            '\n'
            'edge_support_v15 = cv2.dilate(\n'
            '    edge_v15.astype(np.uint8),\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'dark_woody_v15 = gray_v15 < 165\n'
            'very_dark_woody_v15 = gray_v15 < 105\n'
            '\n'
            'woody_pixel_structure = (\n'
            '    dark_woody_v15\n'
            '    &\n'
            '    (\n'
            '        edge_support_v15\n'
            '        |\n'
            '        very_dark_woody_v15\n'
            '    )\n'
            ')\n'
            '\n'
            'woody_candidate = (\n'
            '    raw_woody_v15\n'
            '    &\n'
            '    woody_pixel_structure\n'
            '    &\n'
            '    tree_anchor_near\n'
            ')\n'
            '\n'
            '# Allowed occlusion surfaces: a real branch/trunk can be seen\n'
            '# in front of sky or building/glazing.\n'
            'woody_allowed_source = np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        TREE_ID,\n'
            '        LABEL2ID["sky"],\n'
            '        LABEL2ID["upper_building_facade"],\n'
            '        LABEL2ID["upper_building_glazing"],\n'
            '        LABEL2ID["ground_floor_solid_facade"],\n'
            '        LABEL2ID["ground_floor_glazing"],\n'
            '        UNKNOWN_ID,\n'
            '    ]\n'
            ')\n'
            '\n'
            'woody_candidate &= woody_allowed_source\n'
            'woody_candidate &= ~MAP_ROAD\n'
            'woody_candidate &= ~MAP_SIDEWALK\n'
            'woody_candidate &= ~MAP_CURB\n'
            'woody_candidate &= ~MAP_POLE_V15\n'
            '\n'
            'woody_candidate &= ~np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '        LABEL2ID["person"],\n'
            '        LABEL2ID["vehicle"],\n'
            '        LABEL2ID["traffic_cone_barrel"],\n'
            '        LABEL2ID["traffic_sign_signal"],\n'
            '        LABEL2ID["signboard"],\n'
            '        LABEL2ID["awning_canopy"],\n'
            '        LABEL2ID["sidewalk_shed_scaffold"],\n'
            '        LABEL2ID["fence_railing"],\n'
            '        LABEL2ID["wall_ledge"],\n'
            '        IGNORE_ID,\n'
            '    ]\n'
            ')\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    woody_candidate &= ~artifact_ignore_mask\n'
            '\n'
            '# Connect small gaps in branch/trunk lines without expanding into\n'
            '# a broad silhouette.\n'
            'woody_connected = cv2.morphologyEx(\n'
            '    woody_candidate.astype(np.uint8),\n'
            '    cv2.MORPH_CLOSE,\n'
            '    cv2.getStructuringElement(cv2.MORPH_RECT, (3, 5)),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            'num_t, lab_t, stats_t, _ = cv2.connectedComponentsWithStats(\n'
            '    woody_connected.astype(np.uint8),\n'
            '    connectivity=8\n'
            ')\n'
            '\n'
            'woody_recovery_precise = np.zeros((H, W), dtype=bool)\n'
            'woody_rows = []\n'
            '\n'
            'for cid in range(1, num_t):\n'
            '    comp = lab_t == cid\n'
            '    area = int(comp.sum())\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(stats_t[cid, cv2.CC_STAT_LEFT])\n'
            '    y = int(stats_t[cid, cv2.CC_STAT_TOP])\n'
            '    w = int(stats_t[cid, cv2.CC_STAT_WIDTH])\n'
            '    h = int(stats_t[cid, cv2.CC_STAT_HEIGHT])\n'
            '\n'
            '    area_ratio = area / (H * W)\n'
            '    fill_ratio = area / max(w * h, 1)\n'
            '    elongation = max(\n'
            '        h / max(w, 1),\n'
            '        w / max(h, 1)\n'
            '    )\n'
            '    vertical_span = h / H\n'
            '    anchor_contact_ratio = float(tree_anchor_contact[comp].mean())\n'
            '    high_conf_ratio = float(high_woody_v15[comp].mean())\n'
            '\n'
            '    sparse_branch_network = fill_ratio <= 0.42\n'
            '    elongated_structure = elongation >= 1.45\n'
            '\n'
            '    independent_high_conf_trunk = (\n'
            '        high_conf_ratio >= 0.20\n'
            '        and vertical_span >= 0.07\n'
            '        and elongation >= 1.35\n'
            '    )\n'
            '\n'
            '    keep = (\n'
            '        area >= 8\n'
            '        and area_ratio <= 0.014\n'
            '        and (\n'
            '            sparse_branch_network\n'
            '            or elongated_structure\n'
            '        )\n'
            '        and (\n'
            '            anchor_contact_ratio >= 0.01\n'
            '            or independent_high_conf_trunk\n'
            '        )\n'
            '    )\n'
            '\n'
            '    if keep:\n'
            '        woody_recovery_precise[comp] = True\n'
            '\n'
            '    woody_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "width": w,\n'
            '        "height": h,\n'
            '        "fill_ratio": fill_ratio,\n'
            '        "elongation": elongation,\n'
            '        "vertical_span": vertical_span,\n'
            '        "tree_anchor_contact": anchor_contact_ratio,\n'
            '        "high_conf_woody": high_conf_ratio,\n'
            '        "accepted": keep,\n'
            '    })\n'
            '\n'
            '# Install only precise woody structure.\n'
            'tree_before_woody_v15 = int(np.sum(final_working == TREE_ID))\n'
            'final_working[woody_recovery_precise] = TREE_ID\n'
            'tree_after_woody_v15 = int(np.sum(final_working == TREE_ID))\n'
            '\n'
            'print()\n'
            'print("========================================")\n'
            'print("WOODY TREE RECOVERY")\n'
            'print("========================================")\n'
            'print("Tree pixels before:", tree_before_woody_v15)\n'
            'print("Recovered trunk / branch pixels:", int(woody_recovery_precise.sum()))\n'
            'print("Tree pixels after:", tree_after_woody_v15)\n'
            'if woody_rows:\n'
            '    display(pd.DataFrame(woody_rows))\n'
            '\n'
            '# ============================================================\n'
            '# v1.5.6 FINAL SCAFFOLD PRECEDENCE RE-ASSERTION\n'
            '#\n'
            '# Defensive final guard:\n'
            '# later local recovery may not leave awning_canopy / wall_ledge\n'
            '# inside the evidence-backed scaffold zone.\n'
            '# ============================================================\n'
            '\n'
            'late_scaffold_conflict = (\n'
            '    scaffold_protected_zone\n'
            '    &\n'
            '    np.isin(\n'
            '        final_working,\n'
            '        [\n'
            '            AWNING_ID,\n'
            '            WALL_ID,\n'
            '        ]\n'
            '    )\n'
            ')\n'
            '\n'
            'final_working[\n'
            '    late_scaffold_conflict\n'
            '] = SCAFFOLD_LOCAL_ID\n'
            '\n'
            '\n'
            'remaining_scaffold_awning_wall_conflict = (\n'
            '    scaffold_protected_zone\n'
            '    &\n'
            '    np.isin(\n'
            '        final_working,\n'
            '        [\n'
            '            AWNING_ID,\n'
            '            WALL_ID,\n'
            '        ]\n'
            '    )\n'
            ')\n'
            '\n'
            'assert not np.any(\n'
            '    remaining_scaffold_awning_wall_conflict\n'
            '), (\n'
            '    "SCAFFOLD PRECEDENCE FAILURE: awning_canopy or wall_ledge "\n'
            '    "remains inside evidence-backed scaffold context."\n'
            ')\n'
            '\n'
            '\n'
            'print()\n'
            'print("========================================")\n'
            'print("SCAFFOLD PRECEDENCE QA — v1.5.6")\n'
            'print("========================================")\n'
            'print(\n'
            '    "Late awning/wall conflicts reclaimed:",\n'
            '    int(late_scaffold_conflict.sum())\n'
            ')\n'
            'print(\n'
            '    "Remaining scaffold-context awning/wall conflicts:",\n'
            '    int(remaining_scaffold_awning_wall_conflict.sum())\n'
            ')\n'
            '\n'
            'print()\n'
            'print("✓ v1.5.6 local recovery complete")\n',
 'cell_29': '# ============================================================\n'
            '# TREE / GLAZING BACKBONE LOCK — v1.5\n'
            '#\n'
            '# No tree reconstruction is performed.\n'
            '#\n'
            '# This cell only enforces regression invariants:\n'
            '#\n'
            '# 1. Pixels that were glazing in the semantic backbone may not\n'
            '#    become tree because of a later tree refinement.\n'
            '#\n'
            '# 2. Existing v0.9 tree pixels are retained unless a later\n'
            '#    foreground-specific class legitimately occupies them.\n'
            '#\n'
            '# There is intentionally NO:\n'
            '#\n'
            '#   final_working[raw_foliage] = TREE_ID\n'
            '#\n'
            '# and NO:\n'
            '#\n'
            '#   final_working[raw_woody] = TREE_ID\n'
            '# ============================================================\n'
            '\n'
            'tree_on_backbone_glazing = (\n'
            '    backbone_glazing_mask\n'
            '    &\n'
            '    (\n'
            '        final_working\n'
            '        ==\n'
            '        TREE_ID\n'
            '    )\n'
            ')\n'
            '\n'
            '# v1.5 permits only the explicitly approved woody branch/trunk\n'
            '# recovery to cross a protected glazing surface.\n'
            'illegal_tree_on_glazing = (\n'
            '    tree_on_backbone_glazing\n'
            '    &\n'
            '    ~woody_recovery_precise\n'
            ')\n'
            '\n'
            'if np.any(\n'
            '    illegal_tree_on_glazing\n'
            '):\n'
            '    final_working[\n'
            '        illegal_tree_on_glazing\n'
            '    ] = semantic_backbone[\n'
            '        illegal_tree_on_glazing\n'
            '    ]\n'
            '\n'
            '\n'
            'print(\n'
            '    "Approved woody tree pixels crossing protected glazing:",\n'
            '    int(\n'
            '        np.sum(\n'
            '            backbone_glazing_mask\n'
            '            &\n'
            '            (\n'
            '                final_working\n'
            '                ==\n'
            '                TREE_ID\n'
            '            )\n'
            '        )\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'assert not np.any(\n'
            '    backbone_glazing_mask\n'
            '    &\n'
            '    (\n'
            '        final_working\n'
            '        ==\n'
            '        TREE_ID\n'
            '    )\n'
            '    &\n'
            '    ~woody_recovery_precise\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ v1.5 tree/glazing regression guard passed"\n'
            ')\n',
 'cell_30': '# ============================================================\n'
            '# POLE + TRAFFIC SIGNAL RECOVERY\n'
            '#\n'
            '# pole_fixture:\n'
            '# - streetlight pole\n'
            '# - utility pole\n'
            '# - traffic-signal support pole / mast\n'
            '#\n'
            '# traffic_sign_signal:\n'
            '# - traffic light head\n'
            '# - traffic sign\n'
            '# ============================================================\n'
            '\n'
            'POLE_ID = LABEL2ID[\n'
            '    "pole_fixture"\n'
            ']\n'
            '\n'
            'TRAFFIC_SIGNAL_ID = LABEL2ID[\n'
            '    "traffic_sign_signal"\n'
            ']\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# MAPILLARY SUPPORT\n'
            '# ============================================================\n'
            '\n'
            'MAP_POLE = mapillary_mask(\n'
            '    [\n'
            '        "pole"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'MAP_TRAFFIC_LIGHT = mapillary_mask(\n'
            '    [\n'
            '        "traffic light"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'MAP_TRAFFIC_SIGN = mapillary_mask(\n'
            '    [\n'
            '        "traffic sign"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'print(\n'
            '    "Mapillary pole:",\n'
            '    int(\n'
            '        MAP_POLE.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Mapillary traffic light:",\n'
            '    int(\n'
            '        MAP_TRAFFIC_LIGHT.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Mapillary traffic sign:",\n'
            '    int(\n'
            '        MAP_TRAFFIC_SIGN.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# ADE SUPPORT\n'
            '# ============================================================\n'
            '\n'
            'ADE_POLE_CURRENT = ade_mask_fuzzy(\n'
            '    [\n'
            '        "pole",\n'
            '        "streetlight",\n'
            '        "street light"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'ADE_TRAFFIC_LIGHT_CURRENT = ade_mask_fuzzy(\n'
            '    [\n'
            '        "traffic light"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'ADE_TRAFFIC_SIGN_CURRENT = ade_mask_fuzzy(\n'
            '    [\n'
            '        "traffic sign"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Combined proposal\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'pole_candidate = (\n'
            '\n'
            '    MAP_POLE\n'
            '\n'
            '    |\n'
            '\n'
            '    ADE_POLE_CURRENT\n'
            ')\n'
            '\n'
            '\n'
            'traffic_signal_candidate = (\n'
            '\n'
            '    MAP_TRAFFIC_LIGHT\n'
            '\n'
            '    |\n'
            '\n'
            '    MAP_TRAFFIC_SIGN\n'
            '\n'
            '    |\n'
            '\n'
            '    ADE_TRAFFIC_LIGHT_CURRENT\n'
            '\n'
            '    |\n'
            '\n'
            '    ADE_TRAFFIC_SIGN_CURRENT\n'
            ')\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# POLE GEOMETRY FILTER\n'
            '#\n'
            '# Pole should generally be a vertically persistent component.\n'
            '# ============================================================\n'
            '\n'
            'num_poles, pole_labels, pole_stats, _ = (\n'
            '    cv2.connectedComponentsWithStats(\n'
            '        pole_candidate.astype(np.uint8),\n'
            '        connectivity=8\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'pole_precise = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            '\n'
            'pole_rows = []\n'
            '\n'
            '\n'
            'for cid in range(\n'
            '    1,\n'
            '    num_poles\n'
            '):\n'
            '\n'
            '    component = (\n'
            '        pole_labels\n'
            '        ==\n'
            '        cid\n'
            '    )\n'
            '\n'
            '\n'
            '    area = int(\n'
            '        component.sum()\n'
            '    )\n'
            '\n'
            '\n'
            '    x = pole_stats[\n'
            '        cid,\n'
            '        cv2.CC_STAT_LEFT\n'
            '    ]\n'
            '\n'
            '    y = pole_stats[\n'
            '        cid,\n'
            '        cv2.CC_STAT_TOP\n'
            '    ]\n'
            '\n'
            '    w = pole_stats[\n'
            '        cid,\n'
            '        cv2.CC_STAT_WIDTH\n'
            '    ]\n'
            '\n'
            '    h = pole_stats[\n'
            '        cid,\n'
            '        cv2.CC_STAT_HEIGHT\n'
            '    ]\n'
            '\n'
            '\n'
            '    area_ratio = (\n'
            '        area\n'
            '        /\n'
            '        (H * W)\n'
            '    )\n'
            '\n'
            '\n'
            '    vertical_ratio = (\n'
            '        h\n'
            '        /\n'
            '        max(\n'
            '            w,\n'
            '            1\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '    # Allow:\n'
            '    # - long vertical poles\n'
            '    # - smaller distant poles\n'
            '\n'
            '    accept = (\n'
            '\n'
            '        area_ratio\n'
            '        <=\n'
            '        0.018\n'
            '\n'
            '        and\n'
            '\n'
            '        h\n'
            '        >=\n'
            '        H * 0.025\n'
            '\n'
            '        and\n'
            '\n'
            '        vertical_ratio\n'
            '        >=\n'
            '        1.35\n'
            '    )\n'
            '\n'
            '\n'
            '    if accept:\n'
            '\n'
            '        pole_precise[\n'
            '            component\n'
            '        ] = True\n'
            '\n'
            '\n'
            '    pole_rows.append({\n'
            '\n'
            '        "component":\n'
            '            cid,\n'
            '\n'
            '        "pixels":\n'
            '            area,\n'
            '\n'
            '        "width":\n'
            '            w,\n'
            '\n'
            '        "height":\n'
            '            h,\n'
            '\n'
            '        "vertical_ratio":\n'
            '            vertical_ratio,\n'
            '\n'
            '        "accepted":\n'
            '            accept\n'
            '    })\n'
            '\n'
            '\n'
            'if pole_rows:\n'
            '\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            pole_rows\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Never call road / sidewalk pixels pole unless semantic model\n'
            '# explicitly returned the pole there.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'pole_precise &= (\n'
            '    ~MAP_SKY\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Protect dynamic / research objects\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'do_not_overwrite = np.isin(\n'
            '    final_working,\n'
            '    [\n'
            '\n'
            '        LABEL2ID[\n'
            '            "person"\n'
            '        ],\n'
            '\n'
            '        LABEL2ID[\n'
            '            "vehicle"\n'
            '        ],\n'
            '\n'
            '        LABEL2ID[\n'
            '            "traffic_cone_barrel"\n'
            '        ],\n'
            '\n'
            '        IGNORE_ID\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'pole_precise &= (\n'
            '    ~do_not_overwrite\n'
            ')\n'
            '\n'
            '\n'
            'traffic_signal_candidate &= (\n'
            '    ~do_not_overwrite\n'
            ')\n'
            '\n'
            '# Absolute protection: a detected Street View screenshot artifact\n'
            '# must never be reclassified as a pole / sign / signal.\n'
            'if "google_artifact_mask" in globals():\n'
            '    pole_precise &= ~google_artifact_mask\n'
            '    traffic_signal_candidate &= ~google_artifact_mask\n'
            '\n'
            '# v1.5: a woody component that passed the strict trunk/branch\n'
            '# recovery gate must not be stolen back by the pole recovery.\n'
            'if "woody_recovery_precise" in globals():\n'
            '    pole_precise &= ~woody_recovery_precise\n'
            '    traffic_signal_candidate &= ~woody_recovery_precise\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Install pole first\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'final_working[\n'
            '    pole_precise\n'
            '] = POLE_ID\n'
            '\n'
            '\n'
            '# Signal/sign head has priority over its pole\n'
            '\n'
            'final_working[\n'
            '    traffic_signal_candidate\n'
            '] = TRAFFIC_SIGNAL_ID\n'
            '\n'
            '\n'
            'print(\n'
            '    "Final pole pixels:",\n'
            '    int(\n'
            '        pole_precise.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Final traffic sign/signal pixels:",\n'
            '    int(\n'
            '        traffic_signal_candidate.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "✓ Pole / traffic signal recovery applied"\n'
            ')\n',
 'cell_31': '# ============================================================\n'
            '# v1.5 BACKBONE REGRESSION AUDIT\n'
            '#\n'
            '# The purpose is not to force later labels to equal v0.9.\n'
            '# Foreground classes such as pole / traffic signal may\n'
            '# legitimately occlude a background class.\n'
            '#\n'
            '# The audit specifically checks the known regressions:\n'
            '# - glazing → tree\n'
            '# - broad sidewalk → stoop\n'
            '# - Google artifact → semantic object\n'
            '# ============================================================\n'
            '\n'
            '# 1. Glazing must never become tree.\n'
            'illegal_glazing_to_tree_pixels = int(\n'
            '    np.sum(\n'
            '        backbone_glazing_mask\n'
            '        &\n'
            '        (\n'
            '            final_working\n'
            '            ==\n'
            '            LABEL2ID[\n'
            '                "tree"\n'
            '            ]\n'
            '        )\n'
            '        &\n'
            '        ~woody_recovery_precise\n'
            '    )\n'
            ')\n'
            '\n'
            'approved_woody_on_glazing_pixels = int(\n'
            '    np.sum(\n'
            '        backbone_glazing_mask\n'
            '        &\n'
            '        (\n'
            '            final_working\n'
            '            ==\n'
            '            LABEL2ID[\n'
            '                "tree"\n'
            '            ]\n'
            '        )\n'
            '        &\n'
            '        woody_recovery_precise\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            '# 2. Any surviving stoop must belong to a component that already\n'
            '#    passed the v1.4 component-level stoop precision gate.\n'
            '#\n'
            '# IMPORTANT:\n'
            '# v1.4 validates stoop by connected component, not by requiring\n'
            '# every pixel to sit inside entrance_neighborhood.\n'
            '#\n'
            '# Therefore the regression audit must compare against\n'
            '# valid_stoop_v14, which is the authoritative approved-component\n'
            '# mask created by the stoop precision stage.\n'
            'final_stoop_mask = (\n'
            '    final_working\n'
            '    ==\n'
            '    LABEL2ID[\n'
            '        "stoop_stair"\n'
            '    ]\n'
            ')\n'
            '\n'
            '\n'
            'unsupported_final_stoop = (\n'
            '    final_stoop_mask\n'
            '    &\n'
            '    ~valid_stoop_v14\n'
            ')\n'
            '\n'
            '\n'
            '# 3. Google artifact remains IGNORE.\n'
            'artifact_semantic_contamination = 0\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '\n'
            '    artifact_semantic_contamination = int(\n'
            '        np.sum(\n'
            '            artifact_ignore_mask\n'
            '            &\n'
            '            (\n'
            '                final_working\n'
            '                !=\n'
            '                IGNORE_ID\n'
            '            )\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            'print(\n'
            '    "========================================"\n'
            ')\n'
            '\n'
            'print(\n'
            '    "v1.5 BACKBONE REGRESSION AUDIT"\n'
            ')\n'
            '\n'
            'print(\n'
            '    "========================================"\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Illegal glazing → tree:",\n'
            '    illegal_glazing_to_tree_pixels\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Approved woody crossings on glazing:",\n'
            '    approved_woody_on_glazing_pixels\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Final stoop pixels outside approved components:",\n'
            '    int(\n'
            '        unsupported_final_stoop.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Google artifact semantic contamination:",\n'
            '    artifact_semantic_contamination\n'
            ')\n'
            '\n'
            '\n'
            'assert illegal_glazing_to_tree_pixels == 0\n'
            '\n'
            'assert not np.any(\n'
            '    unsupported_final_stoop\n'
            '), (\n'
            '    "Stoop regression detected: final output contains stoop pixels "\n'
            '    "outside components approved by the v1.4 stoop precision gate."\n'
            ')\n'
            '\n'
            'assert artifact_semantic_contamination == 0\n'
            '\n'
            '\n'
            'print(\n'
            '    "✓ v1.5 regression audit passed"\n'
            ')\n',
 'cell_32': '# ============================================================\n'
            '# STREET VIEW ARTIFACT FINAL LOCK — v1.5\n'
            '#\n'
            '# IMPORTANT:\n'
            '# Scientific data:\n'
            '#   artifact pixels stay IGNORE=255 permanently.\n'
            '#\n'
            '# Display only:\n'
            '#   create a separate contextual visualization fill so the\n'
            '#   ignored screenshot block does not appear as a gray/black\n'
            '#   object in the figure.\n'
            '#\n'
            '# The display fill is NEVER copied back into final_working.\n'
            '# ============================================================\n'
            '\n'
            'from scipy.ndimage import distance_transform_edt\n'
            '\n'
            'if "google_artifact_mask" not in globals():\n'
            '    google_artifact_mask = np.zeros((H, W), dtype=bool)\n'
            '\n'
            'artifact_ignore_mask = google_artifact_mask.copy()\n'
            'artifact_detected_pixels = int(artifact_ignore_mask.sum())\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 1. HARD FINAL LOCK\n'
            '#\n'
            '# Even if an earlier/later refinement accidentally touched the\n'
            '# area, this line wins immediately before final commit.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'final_working[artifact_ignore_mask] = IGNORE_ID\n'
            '\n'
            '# Compatibility audit variables.\n'
            'artifact_restored_mask = np.zeros((H, W), dtype=bool)\n'
            'artifact_unresolved_mask = artifact_ignore_mask.copy()\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 2. DISPLAY-ONLY contextual fill\n'
            '#\n'
            '# Use only broad scene-surface / vegetation classes as donors.\n'
            '# Explicitly EXCLUDE narrow object classes that could recreate\n'
            '# the screenshot shape:\n'
            '#   pole, sign/signal, person, vehicle, cone, signboard, etc.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'DISPLAY_CONTEXT_CLASS_NAMES = [\n'
            '    "roadway",\n'
            '    "sidewalk",\n'
            '    "bike_lane",\n'
            '    "curb_edge",\n'
            '    "upper_building_facade",\n'
            '    "ground_floor_solid_facade",\n'
            '    "ground_floor_glazing",\n'
            '    "upper_building_glazing",\n'
            '    "door_entrance",\n'
            '    "awning_canopy",\n'
            '    "sidewalk_shed_scaffold",\n'
            '    "tree",\n'
            '    "shrub_hedge",\n'
            '    "ground_vegetation",\n'
            '    "vertical_green_wall",\n'
            '    "sky",\n'
            ']\n'
            '\n'
            'display_context_ids = [\n'
            '    LABEL2ID[name]\n'
            '    for name in DISPLAY_CONTEXT_CLASS_NAMES\n'
            '    if name in LABEL2ID\n'
            ']\n'
            '\n'
            'display_context_labels = final_working.copy()\n'
            '\n'
            'display_donor_mask = (\n'
            '    ~artifact_ignore_mask\n'
            '    &\n'
            '    np.isin(final_working, display_context_ids)\n'
            '    &\n'
            '    (final_working != IGNORE_ID)\n'
            ')\n'
            '\n'
            'if artifact_detected_pixels > 0 and display_donor_mask.any():\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # PASS 1 — row-aware street-surface restoration.\n'
            '    #\n'
            '    # Google screenshot residuals in these SVI exports are\n'
            '    # vertical columns.  Filling from the same image row allows\n'
            '    # the upper part of a residual to inherit sidewalk while the\n'
            '    # lower part inherits roadway, instead of painting the whole\n'
            '    # block as one class.\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    street_surface_ids = [\n'
            '        LABEL2ID[name]\n'
            '        for name in [\n'
            '            "roadway",\n'
            '            "sidewalk",\n'
            '            "bike_lane",\n'
            '            "curb_edge",\n'
            '        ]\n'
            '        if name in LABEL2ID\n'
            '    ]\n'
            '\n'
            '    display_filled = np.zeros((H, W), dtype=bool)\n'
            '\n'
            '    artifact_rows = np.where(\n'
            '        artifact_ignore_mask.any(axis=1)\n'
            '    )[0]\n'
            '\n'
            '    for yy in artifact_rows:\n'
            '        target_x = np.where(\n'
            '            artifact_ignore_mask[yy]\n'
            '        )[0]\n'
            '\n'
            '        donor_x = np.where(\n'
            '            (~artifact_ignore_mask[yy])\n'
            '            &\n'
            '            np.isin(\n'
            '                final_working[yy],\n'
            '                street_surface_ids\n'
            '            )\n'
            '        )[0]\n'
            '\n'
            '        if len(target_x) == 0 or len(donor_x) == 0:\n'
            '            continue\n'
            '\n'
            '        # Vectorized nearest horizontal donor.\n'
            '        pos = np.searchsorted(\n'
            '            donor_x,\n'
            '            target_x\n'
            '        )\n'
            '\n'
            '        left_pos = np.clip(\n'
            '            pos - 1,\n'
            '            0,\n'
            '            len(donor_x) - 1\n'
            '        )\n'
            '\n'
            '        right_pos = np.clip(\n'
            '            pos,\n'
            '            0,\n'
            '            len(donor_x) - 1\n'
            '        )\n'
            '\n'
            '        left_x = donor_x[left_pos]\n'
            '        right_x = donor_x[right_pos]\n'
            '\n'
            '        choose_right = (\n'
            '            np.abs(right_x - target_x)\n'
            '            <\n'
            '            np.abs(target_x - left_x)\n'
            '        )\n'
            '\n'
            '        nearest_x_row = np.where(\n'
            '            choose_right,\n'
            '            right_x,\n'
            '            left_x\n'
            '        )\n'
            '\n'
            '        display_context_labels[\n'
            '            yy,\n'
            '            target_x\n'
            '        ] = final_working[\n'
            '            yy,\n'
            '            nearest_x_row\n'
            '        ]\n'
            '\n'
            '        display_filled[\n'
            '            yy,\n'
            '            target_x\n'
            '        ] = True\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # PASS 2 — fallback for artifact pixels where no road /\n'
            '    # sidewalk / curb donor exists in exactly the same row.\n'
            '    #\n'
            '    # Use a strong horizontal bias and only non-UNKNOWN broad\n'
            '    # context classes.\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    unresolved_display = (\n'
            '        artifact_ignore_mask\n'
            '        &\n'
            '        ~display_filled\n'
            '    )\n'
            '\n'
            '    if np.any(unresolved_display):\n'
            '\n'
            '        search_space = ~display_donor_mask\n'
            '\n'
            '        _, nearest_indices = distance_transform_edt(\n'
            '            search_space,\n'
            '            sampling=(6.0, 1.0),\n'
            '            return_distances=True,\n'
            '            return_indices=True\n'
            '        )\n'
            '\n'
            '        nearest_y = nearest_indices[0]\n'
            '        nearest_x = nearest_indices[1]\n'
            '\n'
            '        contextual_fill = final_working[\n'
            '            nearest_y,\n'
            '            nearest_x\n'
            '        ]\n'
            '\n'
            '        display_context_labels[\n'
            '            unresolved_display\n'
            '        ] = contextual_fill[\n'
            '            unresolved_display\n'
            '        ]\n'
            '\n'
            'else:\n'
            '    display_context_labels[artifact_ignore_mask] = UNKNOWN_ID\n'
            '\n'
            '# Absolute display safety:\n'
            '# even if future edits alter donor logic, Google/UI pixels may\n'
            '# never display as pole_fixture or traffic_sign_signal.\n'
            'for forbidden_name in [\n'
            '    "pole_fixture",\n'
            '    "traffic_sign_signal",\n'
            '    "person",\n'
            '    "vehicle",\n'
            '    "traffic_cone_barrel",\n'
            '    "signboard",\n'
            ']:\n'
            '    if forbidden_name in LABEL2ID:\n'
            '        bad_display = (\n'
            '            artifact_ignore_mask\n'
            '            &\n'
            '            (display_context_labels == LABEL2ID[forbidden_name])\n'
            '        )\n'
            '        display_context_labels[bad_display] = UNKNOWN_ID\n'
            '\n'
            '# ============================================================\n'
            '# v1.5.6 BOTTOM ARTIFACT ROADWAY-CONTEXT DISPLAY LOCK\n'
            '#\n'
            '# IMPORTANT:\n'
            '# This operates ONLY on pixels already confirmed by\n'
            '# artifact_ignore_mask.\n'
            '#\n'
            '# It does NOT:\n'
            '# - expand the artifact mask;\n'
            '# - modify normal sidewalk;\n'
            '# - modify normal roadway;\n'
            '# - modify scientific labels.\n'
            '#\n'
            '# For each bottom-touching artifact component, inspect nearby\n'
            '# valid street context. If roadway is dominant, display only\n'
            '# that existing artifact component as roadway.\n'
            '# ============================================================\n'
            '\n'
            'BOTTOM_ARTIFACT_ROAD_ID = LABEL2ID[\n'
            '    "roadway"\n'
            ']\n'
            '\n'
            'BOTTOM_ARTIFACT_SIDEWALK_ID = LABEL2ID[\n'
            '    "sidewalk"\n'
            ']\n'
            '\n'
            'BOTTOM_ARTIFACT_CURB_ID = LABEL2ID[\n'
            '    "curb_edge"\n'
            ']\n'
            '\n'
            'BOTTOM_ARTIFACT_BIKE_ID = LABEL2ID[\n'
            '    "bike_lane"\n'
            ']\n'
            '\n'
            'bottom_artifact_road_lock_mask = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'bottom_artifact_context_rows = []\n'
            '\n'
            '\n'
            'n_bottom_art, bottom_lab, bottom_stats, _ = (\n'
            '    cv2.connectedComponentsWithStats(\n'
            '        artifact_ignore_mask.astype(np.uint8),\n'
            '        connectivity=8\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'for cid in range(\n'
            '    1,\n'
            '    n_bottom_art\n'
            '):\n'
            '\n'
            '    component = (\n'
            '        bottom_lab == cid\n'
            '    )\n'
            '\n'
            '    area = int(\n'
            '        component.sum()\n'
            '    )\n'
            '\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(\n'
            '        bottom_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_LEFT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    y = int(\n'
            '        bottom_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_TOP\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    w = int(\n'
            '        bottom_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_WIDTH\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    h = int(\n'
            '        bottom_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_HEIGHT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    touches_bottom = (\n'
            '        y + h\n'
            '        >=\n'
            '        H - 2\n'
            '    )\n'
            '\n'
            '    center_y = (\n'
            '        y + h / 2\n'
            '    ) / H\n'
            '\n'
            '    # Only the true bottom screenshot-smear use case.\n'
            '    bottom_like = (\n'
            '        touches_bottom\n'
            '        and\n'
            '        center_y >= 0.72\n'
            '    )\n'
            '\n'
            '    if not bottom_like:\n'
            '        continue\n'
            '\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Context ring around the EXISTING artifact component.\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    margin_x = max(\n'
            '        36,\n'
            '        int(round(W * 0.030))\n'
            '    )\n'
            '\n'
            '    margin_y = max(\n'
            '        20,\n'
            '        int(round(H * 0.025))\n'
            '    )\n'
            '\n'
            '    x0 = max(\n'
            '        0,\n'
            '        x - margin_x\n'
            '    )\n'
            '\n'
            '    x1 = min(\n'
            '        W,\n'
            '        x + w + margin_x\n'
            '    )\n'
            '\n'
            '    y0 = max(\n'
            '        0,\n'
            '        y - margin_y\n'
            '    )\n'
            '\n'
            '    y1 = min(\n'
            '        H,\n'
            '        y + h\n'
            '    )\n'
            '\n'
            '\n'
            '    local_labels = final_working[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_artifact = artifact_ignore_mask[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_component = component[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '\n'
            '    # Internal bike_lane is counted as roadway evidence because\n'
            '    # v1.5.5+ merges bike_lane into roadway in final output.\n'
            '    local_street_valid = (\n'
            '        ~local_artifact\n'
            '        &\n'
            '        ~local_component\n'
            '        &\n'
            '        np.isin(\n'
            '            local_labels,\n'
            '            [\n'
            '                BOTTOM_ARTIFACT_ROAD_ID,\n'
            '                BOTTOM_ARTIFACT_BIKE_ID,\n'
            '                BOTTOM_ARTIFACT_SIDEWALK_ID,\n'
            '                BOTTOM_ARTIFACT_CURB_ID,\n'
            '            ]\n'
            '        )\n'
            '    )\n'
            '\n'
            '    street_labels = local_labels[\n'
            '        local_street_valid\n'
            '    ]\n'
            '\n'
            '\n'
            '    roadway_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            BOTTOM_ARTIFACT_ROAD_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    bike_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            BOTTOM_ARTIFACT_BIKE_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    sidewalk_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            BOTTOM_ARTIFACT_SIDEWALK_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    curb_votes = int(\n'
            '        np.sum(\n'
            '            street_labels\n'
            '            ==\n'
            '            BOTTOM_ARTIFACT_CURB_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    street_votes = int(\n'
            '        len(\n'
            '            street_labels\n'
            '        )\n'
            '    )\n'
            '\n'
            '    road_like_votes = (\n'
            '        roadway_votes\n'
            '        +\n'
            '        bike_votes\n'
            '    )\n'
            '\n'
            '    road_like_share = (\n'
            '        road_like_votes / street_votes\n'
            '        if street_votes > 0\n'
            '        else 0.0\n'
            '    )\n'
            '\n'
            '\n'
            '    roadway_dominant = (\n'
            '        street_votes >= 24\n'
            '        and\n'
            '        road_like_share >= 0.55\n'
            '        and\n'
            '        road_like_votes\n'
            '        >=\n'
            '        (\n'
            '            sidewalk_votes\n'
            '            +\n'
            '            curb_votes\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '    if roadway_dominant:\n'
            '\n'
            '        # DISPLAY ONLY.\n'
            '        # Scientific final_working remains IGNORE=255 here.\n'
            '        display_context_labels[\n'
            '            component\n'
            '        ] = BOTTOM_ARTIFACT_ROAD_ID\n'
            '\n'
            '        bottom_artifact_road_lock_mask |= (\n'
            '            component\n'
            '        )\n'
            '\n'
            '\n'
            '    bottom_artifact_context_rows.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "center_y": center_y,\n'
            '        "roadway_votes": roadway_votes,\n'
            '        "bike_votes_as_road": bike_votes,\n'
            '        "sidewalk_votes": sidewalk_votes,\n'
            '        "curb_votes": curb_votes,\n'
            '        "street_votes": street_votes,\n'
            '        "road_like_share": road_like_share,\n'
            '        "roadway_locked": roadway_dominant,\n'
            '    })\n'
            '\n'
            '\n'
            'print()\n'
            'print("========================================")\n'
            'print("BOTTOM ARTIFACT ROADWAY DISPLAY — v1.5.6")\n'
            'print("========================================")\n'
            'print(\n'
            '    "Roadway-locked existing artifact pixels:",\n'
            '    int(\n'
            '        bottom_artifact_road_lock_mask.sum()\n'
            '    )\n'
            ')\n'
            '\n'
            'if bottom_artifact_context_rows:\n'
            '\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            bottom_artifact_context_rows\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# v1.5.4A BORDER-FRINGE DISPLAY LOCK\n'
            '#\n'
            '# This is the successful v1.5.4 behavior:\n'
            '# the accepted gray semantic fringe has already passed\n'
            '# roadway-dominant context validation.\n'
            '#\n'
            '# Scientific map remains IGNORE=255.\n'
            '# Display map alone shows the fringe as roadway.\n'
            '#\n'
            '# IMPORTANT:\n'
            '# Only border_semantic_artifact_mask is modified here.\n'
            '# No large bottom footprint or sidewalk block is touched.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'if "border_semantic_artifact_mask" in globals():\n'
            '\n'
            '    display_context_labels[\n'
            '        border_semantic_artifact_mask\n'
            '    ] = LABEL2ID[\n'
            '        "roadway"\n'
            '    ]\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# v1.5.6 BOTTOM ARTIFACT DISPLAY QA\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'assert not np.any(\n'
            '    bottom_artifact_road_lock_mask\n'
            '    &\n'
            '    (\n'
            '        display_context_labels\n'
            '        !=\n'
            '        BOTTOM_ARTIFACT_ROAD_ID\n'
            '    )\n'
            '), (\n'
            '    "BOTTOM ARTIFACT ROAD LOCK FAILURE: a roadway-dominant "\n'
            '    "confirmed Google artifact is not displayed as roadway."\n'
            ')\n'
            '\n'
            '# Scientific data is still IGNORE for those same pixels.\n'
            'assert np.all(\n'
            '    final_working[\n'
            '        bottom_artifact_road_lock_mask\n'
            '    ]\n'
            '    ==\n'
            '    IGNORE_ID\n'
            '), (\n'
            '    "BOTTOM ARTIFACT SCIENTIFIC FAILURE: display-only roadway "\n'
            '    "lock leaked into scientific labels."\n'
            ')\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# 3. HARD QA ASSERTIONS\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'assert np.all(\n'
            '    final_working[artifact_ignore_mask] == IGNORE_ID\n'
            '), "STRICT IGNORE FAILURE: artifact pixels received semantic labels."\n'
            '\n'
            'if "border_semantic_artifact_mask" in globals():\n'
            '\n'
            '    assert np.all(\n'
            '        final_working[\n'
            '            border_semantic_artifact_mask\n'
            '        ]\n'
            '        ==\n'
            '        IGNORE_ID\n'
            '    ), (\n'
            '        "BORDER FRINGE SCIENTIFIC FAILURE: "\n'
            '        "captured gray fringe is not IGNORE=255."\n'
            '    )\n'
            '\n'
            '    assert not np.any(\n'
            '        border_semantic_artifact_mask\n'
            '        &\n'
            '        (\n'
            '            display_context_labels\n'
            '            !=\n'
            '            LABEL2ID[\n'
            '                "roadway"\n'
            '            ]\n'
            '        )\n'
            '    ), (\n'
            '        "BORDER FRINGE DISPLAY FAILURE: "\n'
            '        "captured gray fringe is not displayed as roadway."\n'
            '    )\n'
            '\n'
            'if "pole_fixture" in LABEL2ID:\n'
            '    assert not np.any(\n'
            '        artifact_ignore_mask\n'
            '        &\n'
            '        (final_working == LABEL2ID["pole_fixture"])\n'
            '    ), "STRICT IGNORE FAILURE: Google artifact became pole_fixture."\n'
            '\n'
            'if "traffic_sign_signal" in LABEL2ID:\n'
            '    assert not np.any(\n'
            '        artifact_ignore_mask\n'
            '        &\n'
            '        (final_working == LABEL2ID["traffic_sign_signal"])\n'
            '    ), "STRICT IGNORE FAILURE: Google artifact became traffic_sign_signal."\n'
            '\n'
            'print("========================================")\n'
            'print("STREET VIEW ARTIFACT FINAL LOCK")\n'
            'print("========================================")\n'
            'print("Detected artifact pixels:", artifact_detected_pixels)\n'
            'print("Scientific semantic assignments inside artifact:", 0)\n'
            'print("Permanent IGNORE pixels:", int(artifact_ignore_mask.sum()))\n'
            '\n'
            'if "border_semantic_artifact_mask" in globals():\n'
            '\n'
            '    print(\n'
            '        "v1.5.4 border-fringe pixels:",\n'
            '        int(\n'
            '            border_semantic_artifact_mask.sum()\n'
            '        )\n'
            '    )\n'
            '\n'
            '    print(\n'
            '        "Border-fringe non-road display pixels:",\n'
            '        int(\n'
            '            np.sum(\n'
            '                border_semantic_artifact_mask\n'
            '                &\n'
            '                (\n'
            '                    display_context_labels\n'
            '                    !=\n'
            '                    LABEL2ID[\n'
            '                        "roadway"\n'
            '                    ]\n'
            '                )\n'
            '            )\n'
            '        )\n'
            '    )\n'
            'print(\n'
            '    "Bottom artifact road-lock non-road display pixels:",\n'
            '    int(\n'
            '        np.sum(\n'
            '            bottom_artifact_road_lock_mask\n'
            '            &\n'
            '            (\n'
            '                display_context_labels\n'
            '                !=\n'
            '                BOTTOM_ARTIFACT_ROAD_ID\n'
            '            )\n'
            '        )\n'
            '    )\n'
            ')\n'
            '\n'
            'print("✓ Artifact cannot contribute to pole_fixture")\n'
            'print("✓ Artifact cannot contribute to traffic_sign_signal")\n'
            'print("✓ Artifact is excluded from all class percentages")\n'
            'print("✓ Row-aware roadway/sidewalk contextual fill exists ONLY for visualization")\n',
 'cell_33': '# ============================================================\n'
            '# COMMIT + STRICT IGNORE VISUAL CHECK — v1.5\n'
            '# ============================================================\n'
            '\n'
            'final_v010 = final_working.copy()\n'
            '\n'
            '# Re-assert permanent scientific IGNORE at the commit boundary.\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    final_v010[artifact_ignore_mask] = IGNORE_ID\n'
            '\n'
            'rgb_v010_scientific = render_taxonomy(final_v010)\n'
            '\n'
            '# Scientific IGNORE shown dark gray in this QA panel only.\n'
            'rgb_v010_scientific[\n'
            '    final_v010 == IGNORE_ID\n'
            '] = (35, 35, 35)\n'
            '\n'
            'original_np = np.array(image)\n'
            '\n'
            '# Display-clean semantic map is separate from scientific labels.\n'
            'display_label_map = (\n'
            '    display_context_labels.copy()\n'
            '    if "display_context_labels" in globals()\n'
            '    else final_v010.copy()\n'
            ')\n'
            '\n'
            'display_label_map[\n'
            '    display_label_map == IGNORE_ID\n'
            '] = UNKNOWN_ID\n'
            '\n'
            'rgb_v010 = render_taxonomy(display_label_map)\n'
            '\n'
            'overlay_v010 = original_np.copy()\n'
            '\n'
            'normal_classified = (\n'
            '    (final_v010 != UNKNOWN_ID)\n'
            '    &\n'
            '    (final_v010 != IGNORE_ID)\n'
            ')\n'
            '\n'
            'overlay_v010[normal_classified] = (\n'
            '    original_np[normal_classified] * 0.42\n'
            '    +\n'
            '    rgb_v010[normal_classified] * 0.58\n'
            ').astype(np.uint8)\n'
            '\n'
            '# Artifact area: use display-only contextual semantic color,\n'
            '# not the original screenshot pixels.\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    overlay_v010[artifact_ignore_mask] = rgb_v010[artifact_ignore_mask]\n'
            '\n'
            '# Hard post-commit assertion.\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)\n'
            '\n'
            '# [batch] intermediate matplotlib QA figure skipped\n'
            '\n'
            '\n'
            'print("✓ Scientific map keeps Google/UI residual as IGNORE=255")\n'
            'print("✓ Display map fills the hole without changing scientific statistics")\n',
 'cell_34': '# ============================================================\n'
            '# ROADWAY PRECEDENCE CLEANUP — v1.5.7\n'
            '#\n'
            '# Target:\n'
            '# sidewalk-colored fragments located INSIDE the roadway.\n'
            '#\n'
            '# Key principle:\n'
            '# Roadway may reclaim a sidewalk component only when multiple\n'
            '# independent spatial / semantic checks agree.\n'
            '#\n'
            '# This is component-level, not a global sidewalk rewrite.\n'
            '# ============================================================\n'
            '\n'
            'ROAD_CLEAN_ID = LABEL2ID["roadway"]\n'
            'SIDEWALK_CLEAN_ID = LABEL2ID["sidewalk"]\n'
            '\n'
            'current_sidewalk_v157 = (\n'
            '    final_working == SIDEWALK_CLEAN_ID\n'
            ')\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Road / sidewalk evidence\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'road_semantic_support_v157 = (\n'
            '    MAP_ROAD\n'
            '    |\n'
            '    ADE_ROAD_V2\n'
            ')\n'
            '\n'
            'strong_sidewalk_support_v157 = (\n'
            '    MAP_SIDEWALK\n'
            '    &\n'
            '    ADE_SIDEWALK_V2\n'
            ')\n'
            '\n'
            '# Keep genuine sidewalk near building / interface.\n'
            'building_interface_near_v157 = cv2.dilate(\n'
            '    building_context.astype(np.uint8),\n'
            '    cv2.getStructuringElement(\n'
            '        cv2.MORPH_RECT,\n'
            '        (35, 25)\n'
            '    ),\n'
            '    iterations=1\n'
            ') > 0\n'
            '\n'
            '# Street foreground zone.\n'
            'lower_foreground_v157 = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'lower_foreground_v157[\n'
            '    int(H * 0.50):H,\n'
            '    :\n'
            '] = True\n'
            '\n'
            '# Candidate sidewalk that might actually be roadway.\n'
            'sidewalk_road_candidate_v157 = (\n'
            '    current_sidewalk_v157\n'
            '    &\n'
            '    lower_foreground_v157\n'
            '    &\n'
            '    road_semantic_support_v157\n'
            ')\n'
            '\n'
            '# Do not touch strong true-sidewalk evidence.\n'
            'sidewalk_road_candidate_v157 &= ~strong_sidewalk_support_v157\n'
            '\n'
            '# Do not touch building-edge sidewalk.\n'
            'sidewalk_road_candidate_v157 &= ~building_interface_near_v157\n'
            '\n'
            '# Never touch artifact scientific pixels.\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    sidewalk_road_candidate_v157 &= ~artifact_ignore_mask\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Connected-component validation\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'n_sw, sw_lab, sw_stats, _ = cv2.connectedComponentsWithStats(\n'
            '    sidewalk_road_candidate_v157.astype(np.uint8),\n'
            '    connectivity=8\n'
            ')\n'
            '\n'
            'false_sidewalk_to_road_v157 = np.zeros(\n'
            '    (H, W),\n'
            '    dtype=bool\n'
            ')\n'
            '\n'
            'road_cleanup_rows_v157 = []\n'
            '\n'
            'for cid in range(1, n_sw):\n'
            '\n'
            '    comp = (\n'
            '        sw_lab == cid\n'
            '    )\n'
            '\n'
            '    area = int(\n'
            '        comp.sum()\n'
            '    )\n'
            '\n'
            '    if area == 0:\n'
            '        continue\n'
            '\n'
            '    x = int(\n'
            '        sw_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_LEFT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    y = int(\n'
            '        sw_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_TOP\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    w = int(\n'
            '        sw_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_WIDTH\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    h = int(\n'
            '        sw_stats[\n'
            '            cid,\n'
            '            cv2.CC_STAT_HEIGHT\n'
            '        ]\n'
            '    )\n'
            '\n'
            '    area_ratio = (\n'
            '        area / (H * W)\n'
            '    )\n'
            '\n'
            '    width_ratio = (\n'
            '        w / W\n'
            '    )\n'
            '\n'
            '    height_ratio = (\n'
            '        h / H\n'
            '    )\n'
            '\n'
            '    center_y = (\n'
            '        y + h / 2\n'
            '    ) / H\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Local ring around the component.\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    margin_x = max(\n'
            '        24,\n'
            '        int(round(W * 0.022))\n'
            '    )\n'
            '\n'
            '    margin_y = max(\n'
            '        18,\n'
            '        int(round(H * 0.022))\n'
            '    )\n'
            '\n'
            '    x0 = max(\n'
            '        0,\n'
            '        x - margin_x\n'
            '    )\n'
            '\n'
            '    x1 = min(\n'
            '        W,\n'
            '        x + w + margin_x\n'
            '    )\n'
            '\n'
            '    y0 = max(\n'
            '        0,\n'
            '        y - margin_y\n'
            '    )\n'
            '\n'
            '    y1 = min(\n'
            '        H,\n'
            '        y + h + margin_y\n'
            '    )\n'
            '\n'
            '    local_labels = final_working[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    local_comp = comp[\n'
            '        y0:y1,\n'
            '        x0:x1\n'
            '    ]\n'
            '\n'
            '    valid_local = (\n'
            '        ~local_comp\n'
            '        &\n'
            '        (\n'
            '            local_labels != IGNORE_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    local_valid_labels = local_labels[\n'
            '        valid_local\n'
            '    ]\n'
            '\n'
            '    road_votes = int(\n'
            '        np.sum(\n'
            '            local_valid_labels\n'
            '            ==\n'
            '            ROAD_CLEAN_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    sidewalk_votes = int(\n'
            '        np.sum(\n'
            '            local_valid_labels\n'
            '            ==\n'
            '            SIDEWALK_CLEAN_ID\n'
            '        )\n'
            '    )\n'
            '\n'
            '    bike_votes = int(\n'
            '        np.sum(\n'
            '            local_valid_labels\n'
            '            ==\n'
            '            LABEL2ID["bike_lane"]\n'
            '        )\n'
            '    )\n'
            '\n'
            '    road_like_votes = (\n'
            '        road_votes\n'
            '        +\n'
            '        bike_votes\n'
            '    )\n'
            '\n'
            '    street_votes = (\n'
            '        road_like_votes\n'
            '        +\n'
            '        sidewalk_votes\n'
            '    )\n'
            '\n'
            '    road_like_share = (\n'
            '        road_like_votes / street_votes\n'
            '        if street_votes > 0\n'
            '        else 0.0\n'
            '    )\n'
            '\n'
            '    road_support_ratio = float(\n'
            '        road_semantic_support_v157[\n'
            '            comp\n'
            '        ].mean()\n'
            '    )\n'
            '\n'
            '    map_sidewalk_ratio = float(\n'
            '        MAP_SIDEWALK[\n'
            '            comp\n'
            '        ].mean()\n'
            '    )\n'
            '\n'
            '    ade_sidewalk_ratio = float(\n'
            '        ADE_SIDEWALK_V2[\n'
            '            comp\n'
            '        ].mean()\n'
            '    )\n'
            '\n'
            '    # --------------------------------------------------------\n'
            '    # Two accepted false-sidewalk patterns:\n'
            '    #\n'
            '    # A. Small isolated road fragment\n'
            '    # B. Thin / elongated road-edge strip\n'
            '    # --------------------------------------------------------\n'
            '\n'
            '    small_isolated = (\n'
            '        area_ratio <= 0.012\n'
            '        and\n'
            '        road_like_share >= 0.62\n'
            '        and\n'
            '        road_support_ratio >= 0.45\n'
            '    )\n'
            '\n'
            '    thin_road_strip = (\n'
            '        height_ratio <= 0.10\n'
            '        and\n'
            '        (\n'
            '            width_ratio >= 0.08\n'
            '            or\n'
            '            w >= 3 * max(h, 1)\n'
            '        )\n'
            '        and\n'
            '        road_like_share >= 0.58\n'
            '        and\n'
            '        road_support_ratio >= 0.45\n'
            '    )\n'
            '\n'
            '    weak_true_sidewalk = (\n'
            '        map_sidewalk_ratio < 0.40\n'
            '        or\n'
            '        ade_sidewalk_ratio < 0.40\n'
            '    )\n'
            '\n'
            '    foreground_position_ok = (\n'
            '        center_y >= 0.55\n'
            '    )\n'
            '\n'
            '    accept_as_road = (\n'
            '        foreground_position_ok\n'
            '        and\n'
            '        weak_true_sidewalk\n'
            '        and\n'
            '        (\n'
            '            small_isolated\n'
            '            or\n'
            '            thin_road_strip\n'
            '        )\n'
            '    )\n'
            '\n'
            '    if accept_as_road:\n'
            '        false_sidewalk_to_road_v157[\n'
            '            comp\n'
            '        ] = True\n'
            '\n'
            '    road_cleanup_rows_v157.append({\n'
            '        "component": cid,\n'
            '        "pixels": area,\n'
            '        "width": w,\n'
            '        "height": h,\n'
            '        "area_ratio": area_ratio,\n'
            '        "width_ratio": width_ratio,\n'
            '        "height_ratio": height_ratio,\n'
            '        "center_y": center_y,\n'
            '        "road_like_share": road_like_share,\n'
            '        "road_semantic_support": road_support_ratio,\n'
            '        "map_sidewalk_support": map_sidewalk_ratio,\n'
            '        "ade_sidewalk_support": ade_sidewalk_ratio,\n'
            '        "small_isolated": small_isolated,\n'
            '        "thin_road_strip": thin_road_strip,\n'
            '        "accepted_as_roadway": accept_as_road,\n'
            '    })\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# Install correction\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'sidewalk_before_v157 = int(\n'
            '    np.sum(\n'
            '        final_working\n'
            '        ==\n'
            '        SIDEWALK_CLEAN_ID\n'
            '    )\n'
            ')\n'
            '\n'
            'road_before_v157 = int(\n'
            '    np.sum(\n'
            '        final_working\n'
            '        ==\n'
            '        ROAD_CLEAN_ID\n'
            '    )\n'
            ')\n'
            '\n'
            'final_working[\n'
            '    false_sidewalk_to_road_v157\n'
            '] = ROAD_CLEAN_ID\n'
            '\n'
            '# Keep display map synchronized if already created.\n'
            'if "display_context_labels" in globals():\n'
            '\n'
            '    display_context_labels[\n'
            '        false_sidewalk_to_road_v157\n'
            '    ] = ROAD_CLEAN_ID\n'
            '\n'
            '\n'
            'sidewalk_after_v157 = int(\n'
            '    np.sum(\n'
            '        final_working\n'
            '        ==\n'
            '        SIDEWALK_CLEAN_ID\n'
            '    )\n'
            ')\n'
            '\n'
            'road_after_v157 = int(\n'
            '    np.sum(\n'
            '        final_working\n'
            '        ==\n'
            '        ROAD_CLEAN_ID\n'
            '    )\n'
            ')\n'
            '\n'
            '\n'
            'print("========================================")\n'
            'print("ROADWAY PRECEDENCE CLEANUP — v1.5.7")\n'
            'print("========================================")\n'
            'print(\n'
            '    "False sidewalk pixels → roadway:",\n'
            '    int(\n'
            '        false_sidewalk_to_road_v157.sum()\n'
            '    )\n'
            ')\n'
            'print(\n'
            '    "Sidewalk pixels:",\n'
            '    sidewalk_before_v157,\n'
            '    "→",\n'
            '    sidewalk_after_v157\n'
            ')\n'
            'print(\n'
            '    "Roadway pixels:",\n'
            '    road_before_v157,\n'
            '    "→",\n'
            '    road_after_v157\n'
            ')\n'
            '\n'
            'if road_cleanup_rows_v157:\n'
            '    display(\n'
            '        pd.DataFrame(\n'
            '            road_cleanup_rows_v157\n'
            '        )\n'
            '    )\n'
            '\n'
            '\n'
            '# ------------------------------------------------------------\n'
            '# QA:\n'
            '# corrected pixels must no longer remain sidewalk.\n'
            '# ------------------------------------------------------------\n'
            '\n'
            'assert not np.any(\n'
            '    false_sidewalk_to_road_v157\n'
            '    &\n'
            '    (\n'
            '        final_working\n'
            '        ==\n'
            '        SIDEWALK_CLEAN_ID\n'
            '    )\n'
            '), (\n'
            '    "ROADWAY PRECEDENCE FAILURE: accepted false sidewalk "\n'
            '    "component still remains sidewalk."\n'
            ')\n'
            '\n'
            'print(\n'
            '    "✓ v1.5.7 roadway precedence cleanup applied"\n'
            ')\n'
            '\n'
            '\n'
            '# ============================================================\n'
            '# SHARE-READY AUTHORITATIVE STATE COMMIT — v1.5.7.1\n'
            '#\n'
            '# v1.5.7 performs roadway cleanup after the earlier final_v010\n'
            '# snapshot.  Re-commit here so scientific QA / stats / exports\n'
            '# cannot drift from the final visualization state.\n'
            '# ============================================================\n'
            '\n'
            'final_v010 = final_working.copy()\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    final_v010[\n'
            '        artifact_ignore_mask\n'
            '    ] = IGNORE_ID\n'
            '\n'
            '# Keep display state aligned with the corrected semantic state\n'
            '# while preserving display-only artifact context fill.\n'
            'if "display_context_labels" in globals():\n'
            '    non_artifact_pixels_v1571 = np.ones(\n'
            '        (H, W),\n'
            '        dtype=bool\n'
            '    )\n'
            '\n'
            '    if "artifact_ignore_mask" in globals():\n'
            '        non_artifact_pixels_v1571 &= ~artifact_ignore_mask\n'
            '\n'
            '    display_context_labels[\n'
            '        non_artifact_pixels_v1571\n'
            '    ] = final_working[\n'
            '        non_artifact_pixels_v1571\n'
            '    ]\n'
            '\n'
            'print(\n'
            '    "✓ Authoritative scientific state re-committed after roadway cleanup"\n'
            ')\n',
 'cell_35': '# ============================================================\n'
            '# FINAL TAXONOMY MERGE — v1.5.5\n'
            '# bike_lane → roadway\n'
            '#\n'
            '# IMPORTANT:\n'
            '# The internal bike_lane ID remains declared so earlier model /\n'
            '# fusion code does not break.\n'
            '#\n'
            '# But from THIS POINT FORWARD:\n'
            '# - scientific final labels\n'
            '# - class percentages\n'
            '# - exported label PNG\n'
            '# - Combined Semantic Mask\n'
            '# - final legend\n'
            '#\n'
            '# contain NO independent bike_lane class.\n'
            '# ============================================================\n'
            '\n'
            'ROADWAY_ID = LABEL2ID["roadway"]\n'
            'BIKE_LANE_ID = LABEL2ID["bike_lane"]\n'
            '\n'
            'bike_lane_pixels_before_merge = int(\n'
            '    np.sum(\n'
            '        final_working == BIKE_LANE_ID\n'
            '    )\n'
            ')\n'
            '\n'
            'final_working[\n'
            '    final_working == BIKE_LANE_ID\n'
            '] = ROADWAY_ID\n'
            '\n'
            '# Keep compatibility states aligned if they exist.\n'
            'if "final_v010" in globals():\n'
            '    final_v010[\n'
            '        final_v010 == BIKE_LANE_ID\n'
            '    ] = ROADWAY_ID\n'
            '\n'
            'if "display_context_labels" in globals():\n'
            '    display_context_labels[\n'
            '        display_context_labels == BIKE_LANE_ID\n'
            '    ] = ROADWAY_ID\n'
            '\n'
            'print("========================================")\n'
            'print("FINAL TAXONOMY MERGE — v1.5.5")\n'
            'print("========================================")\n'
            'print("bike_lane pixels merged into roadway:", bike_lane_pixels_before_merge)\n'
            'print(\n'
            '    "bike_lane pixels remaining in final_working:",\n'
            '    int(np.sum(final_working == BIKE_LANE_ID))\n'
            ')\n'
            '\n'
            'assert not np.any(\n'
            '    final_working == BIKE_LANE_ID\n'
            '), (\n'
            '    "BIKE-LANE MERGE FAILURE: bike_lane is still present "\n'
            '    "in the final scientific label map."\n'
            ')\n'
            '\n'
            'print("✓ bike_lane is no longer an independent final output class")\n'
            '\n'
            '# v1.5.7.1: final_v010 is the authoritative scientific export\n'
            '# state and must equal final_working everywhere except that\n'
            '# artifact pixels are explicitly IGNORE=255 in both.\n'
            'assert np.array_equal(\n'
            '    final_v010,\n'
            '    final_working\n'
            '), (\n'
            '    "FINAL STATE DRIFT: final_v010 and final_working differ "\n'
            '    "after final taxonomy merge."\n'
            ')\n'
            '\n'
            'print(\n'
            '    "✓ final_v010 == final_working at final taxonomy boundary"\n'
            ')\n',
 'cell_36': '# ============================================================\n'
            '# CURRENT QA — v1.5.7.1 SHARE-READY AUDITED\n'
            '# ============================================================\n'
            '\n'
            'valid = final_v010 != IGNORE_ID\n'
            'valid_count = max(int(valid.sum()), 1)\n'
            '\n'
            'def current_pct(class_name):\n'
            '    cid = LABEL2ID[class_name]\n'
            '    return np.sum(valid & (final_v010 == cid)) / valid_count * 100\n'
            '\n'
            'qa_classes = [\n'
            '    "roadway",\n'
            '    "sidewalk",\n'
            '    "stoop_stair",\n'
            '    "awning_canopy",\n'
            '    "wall_ledge",\n'
            '    "sidewalk_shed_scaffold",\n'
            '    "fence_railing",\n'
            '    "traffic_cone_barrel",\n'
            '    "pole_fixture",\n'
            '    "traffic_sign_signal",\n'
            '    "tree",\n'
            '    "upper_building_facade",\n'
            '    "upper_building_glazing",\n'
            '    "ground_floor_glazing",\n'
            '    "other_unknown",\n'
            ']\n'
            '\n'
            'qa_df = pd.DataFrame([\n'
            '    {"class_name": name, "percentage": current_pct(name)}\n'
            '    for name in qa_classes\n'
            '])\n'
            'display(qa_df)\n'
            '\n'
            '# -----------------------------\n'
            '# Scientific / schema invariants\n'
            '# -----------------------------\n'
            'declared_ids = set(CLASS_NAMES.keys())\n'
            'observed_ids = set(np.unique(final_v010[valid]).astype(int).tolist())\n'
            '\n'
            'assert observed_ids.issubset(declared_ids), (\n'
            '    f"Undeclared label IDs found: {sorted(observed_ids - declared_ids)}"\n'
            ')\n'
            'assert NUM_CLASSES == TAXONOMY["num_classes"] == len(CVAT_LABELS) == 30\n'
            'assert LABEL2ID["traffic_cone_barrel"] == 29\n'
            'assert "tree_canopy" not in LABEL2ID and "tree_trunk" not in LABEL2ID\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)\n'
            '\n'
            'total_pct = sum(\n'
            '    np.sum(valid & (final_v010 == cid)) / valid_count * 100\n'
            '    for cid in CLASS_NAMES\n'
            ')\n'
            '\n'
            'print("Tree:", current_pct("tree"), "%")\n'
            'print("Sidewalk:", current_pct("sidewalk"), "%")\n'
            'print("Stoop / stair:", current_pct("stoop_stair"), "%")\n'
            'print("Awning / canopy:", current_pct("awning_canopy"), "%")\n'
            'print("Wall / ledge:", current_pct("wall_ledge"), "%")\n'
            'print("Upper façade:", current_pct("upper_building_facade"), "%")\n'
            'print("Upper glazing:", current_pct("upper_building_glazing"), "%")\n'
            'print("Scaffold:", current_pct("sidewalk_shed_scaffold"), "%")\n'
            'print("Fence / railing:", current_pct("fence_railing"), "%")\n'
            'print("Ground glazing:", current_pct("ground_floor_glazing"), "%")\n'
            'print("Other unknown:", current_pct("other_unknown"), "%")\n'
            'print("Declared-class percentage sum:", total_pct, "%")\n'
            '\n'
            'assert abs(total_pct - 100.0) < 1e-6\n'
            'print("✓ SHARE QA PASSED: schema, labels, IGNORE policy, and percentages are internally consistent")\n'
            '\n'
            '\n'
            '# v1.5.5 final-class merge QA\n'
            'assert current_pct("bike_lane") == 0.0, (\n'
            '    "bike_lane must be merged into roadway before final statistics."\n'
            ')\n'
            '\n'
            'print(\n'
            '    "Bike lane exposed in final output:",\n'
            '    "NO"\n'
            ')\n'
            '\n'
            '\n'
            '# Share-ready state consistency\n'
            'assert np.array_equal(\n'
            '    final_v010,\n'
            '    final_working\n'
            '), "SHARE QA FAILURE: final_v010 != final_working"\n'
            '\n'
            'print("✓ SHARE STATE QA PASSED: scientific/export state matches final inference state")\n',
 'cell_37': '# ============================================================\n'
            '# STRICT-IGNORE DISPLAY — v1.5\n'
            '#\n'
            '# SCIENTIFIC:\n'
            '#   final_v010 keeps Google/UI artifact as IGNORE=255.\n'
            '#\n'
            '# DISPLAY:\n'
            '#   display_label_map uses contextual broad-surface fill only.\n'
            '#   It is never used for statistics.\n'
            '# ============================================================\n'
            '\n'
            'display_label_map = (\n'
            '    display_context_labels.copy()\n'
            '    if "display_context_labels" in globals()\n'
            '    else final_v010.copy()\n'
            ')\n'
            '\n'
            'display_label_map[\n'
            '    display_label_map == IGNORE_ID\n'
            '] = UNKNOWN_ID\n'
            '\n'
            'rgb_display_clean = render_taxonomy(display_label_map)\n'
            '\n'
            'original_np = np.array(image)\n'
            'overlay_display_clean = original_np.copy()\n'
            '\n'
            'normal_classified = (\n'
            '    (final_v010 != UNKNOWN_ID)\n'
            '    &\n'
            '    (final_v010 != IGNORE_ID)\n'
            ')\n'
            '\n'
            'overlay_display_clean[normal_classified] = (\n'
            '    original_np[normal_classified] * 0.42\n'
            '    +\n'
            '    rgb_display_clean[normal_classified] * 0.58\n'
            ').astype(np.uint8)\n'
            '\n'
            'if "artifact_ignore_mask" in globals():\n'
            '    overlay_display_clean[artifact_ignore_mask] = (\n'
            '        rgb_display_clean[artifact_ignore_mask]\n'
            '    )\n'
            '\n'
            '# [batch] intermediate matplotlib QA figure skipped\n'
            '\n'
            '\n'
            'print("✓ Google/UI pixels are excluded from scientific labels")\n'
            'print("✓ Display fill cannot create pole/sign/person/vehicle/cone artifacts")\n',
 'cell_38': '# ============================================================\n'
            '# USE STRICT-IGNORE DISPLAY VERSION FOR VISUAL EXPORT\n'
            '# ============================================================\n'
            '\n'
            'rgb_v010 = rgb_display_clean.copy()\n'
            'overlay_v010 = overlay_display_clean.copy()\n'
            '\n'
            'print("✓ RGB / Overlay export uses display-only context fill")\n'
            'print("✓ Scientific label export still contains IGNORE=255")\n'}


## 6. Batch runner

In [ ]:

# ============================================================
# PER-IMAGE RUNNER + INCREMENTAL CSV
# ============================================================
import csv

VISIBLE_CLASS_NAMES = [
    CLASS_NAMES[cid]
    for cid in sorted(CLASS_NAMES)
    if CLASS_NAMES[cid] != 'bike_lane'
]

CSV_FIELDS = (
    ['image_name', 'test_id', 'relative_path', 'width', 'height', 'status',
     'elapsed_sec', 'valid_pixels', 'ignore_pct_of_full_image',
     'unknown_pct_excluding_ignore', 'coverage_pct_excluding_ignore']
    + [f'{name}_pct' for name in VISIBLE_CLASS_NAMES]
    + ['error_message']
)

def _write_row(path, row):
    exists = path.exists() and path.stat().st_size > 0
    with open(path, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow({k: row.get(k, '') for k in CSV_FIELDS})
        f.flush()

def _completed_ids():
    if not RESUME or not MASTER_CSV.exists():
        return set()
    try:
        df = pd.read_csv(MASTER_CSV)
        if 'status' in df.columns:
            return set(df.loc[df['status'].eq('ok'), 'test_id'].astype(str))
        return set(df['test_id'].astype(str))
    except Exception:
        return set()

def _make_stats_row(state, image_path, elapsed):
    final_map = state['final_v010']
    valid = final_map != IGNORE_ID
    valid_count = max(int(valid.sum()), 1)
    row = {
        'image_name': image_path.name,
        'test_id': state['TEST_ID'],
        'relative_path': str(image_path.relative_to(INPUT_DIR)) if image_path.is_relative_to(INPUT_DIR) else str(image_path),
        'width': state['W'],
        'height': state['H'],
        'status': 'ok',
        'elapsed_sec': round(float(elapsed), 3),
        'valid_pixels': valid_count,
        'ignore_pct_of_full_image': float(np.mean(final_map == IGNORE_ID) * 100.0),
    }
    unknown_px = int(np.sum(valid & (final_map == LABEL2ID['other_unknown'])))
    unknown_pct = 100.0 * unknown_px / valid_count
    row['unknown_pct_excluding_ignore'] = unknown_pct
    row['coverage_pct_excluding_ignore'] = 100.0 - unknown_pct
    for cname in VISIBLE_CLASS_NAMES:
        cid = LABEL2ID[cname]
        px = int(np.sum(valid & (final_map == cid)))
        row[f'{cname}_pct'] = 100.0 * px / valid_count
    row['error_message'] = ''
    return row

def _export_required_outputs(state, image_path):
    test_id = state['TEST_ID']
    final_map = state['final_v010']
    display_labels = state.get('display_context_labels', final_map)

    if SAVE_LABEL_MAP:
        Image.fromarray(final_map.astype(np.uint8)).save(
            BATCH_OUTPUT_ROOT / 'label_maps' / f'{test_id}_LABEL_MAP_STABLE.png',
            compress_level=PNG_COMPRESS_LEVEL,
        )

    if SAVE_SUMMARY_IMAGE:
        save_fast_summary(
            state['image'],
            display_labels,
            final_map,
            BATCH_OUTPUT_ROOT / 'summary_images' / f'{test_id}_SUMMARY.png',
        )

    if SAVE_EXTRA_AUDIT_OUTPUTS:
        audit_dir = BATCH_OUTPUT_ROOT / 'audit_outputs' / test_id
        audit_dir.mkdir(parents=True, exist_ok=True)
        state['image'].save(audit_dir / f'{test_id}_ORIGINAL.png')
        # Display-clean RGB and overlay if present in the frozen pipeline.
        if 'rgb_v010' in state:
            Image.fromarray(state['rgb_v010'].astype(np.uint8)).save(
                audit_dir / f'{test_id}_RGB_CLEAN.png', compress_level=PNG_COMPRESS_LEVEL)
        if 'overlay_v010' in state:
            Image.fromarray(state['overlay_v010'].astype(np.uint8)).save(
                audit_dir / f'{test_id}_OVERLAY_CLEAN.png', compress_level=PNG_COMPRESS_LEVEL)
        if 'artifact_ignore_mask' in state:
            Image.fromarray((state['artifact_ignore_mask'].astype(np.uint8) * 255)).save(
                audit_dir / f'{test_id}_STREETVIEW_ARTIFACT_IGNORE_MASK.png', compress_level=PNG_COMPRESS_LEVEL)


def process_one_image(image_path: Path):
    t_start = time.perf_counter()
    image = Image.open(image_path).convert('RGB')
    W, H = image.size
    test_id = safe_test_id(image_path, INPUT_DIR)

    # Fresh per-image namespace prevents state leakage between 10,000 images.
    state = dict(globals())
    state.update({
        'image': image,
        'W': W,
        'H': H,
        'TEST_ID': test_id,
        'filename': image_path.name,
        # suppress DataFrame/display spam in production
        'display': (display if VERBOSE_PER_IMAGE else (lambda *args, **kwargs: None)),
    })

    def run_step(key, model_name=None):
        if model_name:
            activate_model(model_name)
        exec(PIPELINE_SOURCE[key], state, state)

    output_sink = None if VERBOSE_PER_IMAGE else io.StringIO()
    redirect = contextlib.nullcontext() if VERBOSE_PER_IMAGE else contextlib.redirect_stdout(output_sink)

    with redirect:
        # Same inference/refinement order as the source notebook.
        run_step('ADE', 'ade')
        run_step('MAPILLARY', 'mapillary')
        run_step('DINO', 'dino')
        run_step('SAM', 'sam')
        run_step('cell_18')
        run_step('cell_20')
        run_step('cell_21')
        run_step('TRAFFIC_DETECT', 'dino')
        run_step('TRAFFIC_SAM', 'sam')
        for key in [
            'cell_23','cell_24','cell_25','cell_26','cell_27','cell_28','cell_29',
            'cell_30','cell_31','cell_32','cell_33','cell_34','cell_35','cell_36',
            'cell_37','cell_38'
        ]:
            run_step(key)

    # Final scientific invariants retained from the original export boundary.
    final_map = state['final_v010']
    assert np.array_equal(final_map, state['final_working'])
    assert not np.any(final_map == LABEL2ID['bike_lane'])
    if 'artifact_ignore_mask' in state:
        assert np.all(final_map[state['artifact_ignore_mask']] == IGNORE_ID)

    elapsed = time.perf_counter() - t_start
    _export_required_outputs(state, image_path)
    row = _make_stats_row(state, image_path, elapsed)

    # FAST V2: release references, but do NOT force a full GC / CUDA cache flush
    # after every image. Periodic cleanup is handled by the outer batch loop.
    image.close()
    del state

    return row

print('✓ Per-image runner ready')


## 7. First benchmark run

For this file, **do not edit anything** for the first run. Choose a GPU runtime and use **Runtime → Run all**. The notebook is already pointed at `My Drive/StreetViewSegmentation/input_images`, uses a fresh FAST V2 result folder, and forces the benchmark images to run.


## FAST V2 benchmark

The final cell reports the measured average seconds per processed image and a projected runtime for 10,000 images. With only 3 test images, the estimate is preliminary but still useful for the next tuning step.


In [ ]:

# ============================================================
# RUN THE BATCH
# ============================================================
image_paths = get_image_paths()
print('Images found:', len(image_paths))
if not image_paths:
    raise RuntimeError(
        f'No images found in {INPUT_DIR}. Put images there or edit INPUT_DIR above.'
    )

completed = _completed_ids()
print('Already completed:', len(completed))

ok_count = 0
error_count = 0
skipped_count = 0
batch_start = time.perf_counter()

for image_path in tqdm(image_paths, desc='Segmentation'):
    test_id = safe_test_id(image_path, INPUT_DIR)
    if RESUME and test_id in completed:
        skipped_count += 1
        continue
    try:
        row = process_one_image(image_path)
        _write_row(MASTER_CSV, row)
        completed.add(test_id)
        ok_count += 1

        # Periodic memory cleanup instead of every image.
        if CLEANUP_EVERY_N and ok_count % int(CLEANUP_EVERY_N) == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    except Exception as e:
        error_count += 1
        err = {
            'image_name': image_path.name,
            'test_id': test_id,
            'relative_path': str(image_path),
            'status': 'error',
            'error_message': f'{type(e).__name__}: {e}',
        }
        _write_row(MASTER_CSV, err)
        with open(ERROR_LOG, 'a', encoding='utf-8') as f:
            f.write(f'\n===== {test_id} | {image_path} =====\n')
            f.write(traceback.format_exc())
        print(f'ERROR {test_id}: {type(e).__name__}: {e}')
        if STOP_ON_ERROR:
            raise

elapsed = time.perf_counter() - batch_start
print('========================================')
print('BATCH COMPLETE')
print('========================================')
print('OK      :', ok_count)
print('Skipped :', skipped_count)
print('Errors  :', error_count)
print('Hours   :', round(elapsed / 3600, 3))
if ok_count > 0:
    avg_sec = elapsed / ok_count
    projected_hours_10k = avg_sec * 10000 / 3600.0
    print('Avg sec/image:', round(avg_sec, 2))
    print('Projected 10,000 images:', round(projected_hours_10k, 1), 'hours')
    if avg_sec <= 17.28:
        print('TARGET: within 48 hours ✓')
    else:
        print('TARGET: over 48 hours — needs another speed pass')
print('CSV     :', MASTER_CSV)
print('Summaries:', BATCH_OUTPUT_ROOT / 'summary_images')
print('Label maps:', BATCH_OUTPUT_ROOT / 'label_maps')


## 8. Optional equivalence check

In [ ]:

# ============================================================
# OPTIONAL EXACT LABEL-MAP COMPARISON AGAINST AN OLD SINGLE-IMAGE RESULT
# ============================================================
# Set this to a label map produced by the original single-image notebook.
REFERENCE_LABEL_MAP = None
# Example:
# REFERENCE_LABEL_MAP = Path('/content/old_result/IMG_001_LABEL_MAP_STABLE.png')

if REFERENCE_LABEL_MAP:
    reference = np.array(Image.open(REFERENCE_LABEL_MAP), dtype=np.uint8)
    test_id = Path(REFERENCE_LABEL_MAP).name.replace('_LABEL_MAP_STABLE.png', '')
    candidate_path = BATCH_OUTPUT_ROOT / 'label_maps' / f'{test_id}_LABEL_MAP_STABLE.png'
    candidate = np.array(Image.open(candidate_path), dtype=np.uint8)
    print('Shape equal:', reference.shape == candidate.shape)
    print('Pixel-for-pixel equal:', np.array_equal(reference, candidate))
    if reference.shape == candidate.shape:
        print('Different pixels:', int(np.sum(reference != candidate)))
else:
    print('Optional validation skipped. Set REFERENCE_LABEL_MAP if you want an exact old-vs-batch comparison.')
